In [ ]:
# ---- Configuration: edit config/config.yaml (see README) ----
# Genetic groups (Control, NoMut, MutBpos, MutCpos) and the DrugB time course: PCA/k-means, correlations with clinical score
# Input files are read from, and outputs written to, <genetic_groups_workdir> (see README for the file list).
import os, sys
from pathlib import Path
_root = Path(os.environ.get("PROJECT_ROOT") or next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "utils" / "config.py").exists()))
os.environ["PROJECT_ROOT"] = str(_root)
sys.path.insert(0, str(_root))
from utils.config import cfg, cfg_path
os.chdir(cfg_path("genetic_groups_workdir", create=True))
EXCLUDED_SAMPLE_1 = cfg["exclude_samples"]["sample_1"]   # sample dropped from the analysis


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.spatial.distance import euclidean, squareform
from scipy.cluster import hierarchy
import textwrap
from scipy.spatial.distance import euclidean
from scipy.cluster.hierarchy import linkage, dendrogram  # Add this import statement
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from matplotlib.patches import Patch

In [ ]:
Data = pd.read_csv("Data_final_analysis.csv", index_col=0)
Data = Data.drop(EXCLUDED_SAMPLE_1, axis=1)   # see config: exclude_samples
Data.head(7000000)

In [ ]:
row_names = Data.index.tolist()
row_names

In [ ]:
proteins_to_keep = [
    "ACHE","AGRN","ANXA5","APOE","APOE4","AÎ²38","AÎ²40","AÎ²42","BACE1","BD-MAPT",
    "BD-pTau-181","BD-pTau-217","BD-pTau-231","CALB2","CCL11","CCL13","CCL17",
    "CCL2","CCL22","CCL26","CCL3","CCL4","CD40LG","CD63","CHI3L1","CHIT1",
    "CNTN2","CRH","CRP","CSF2","CST3","CX3CL1","CXCL1","CXCL10","CXCL8","DDC",
    "ENO2","FABP3","FCN2","FGF2","FLT1","FOLR1","GDF15","GDI1","GFAP","GOT1",
    "HBA1","HTT","ICAM1","IFNG","IGF1R","IGFBP7","IL10","IL12p70","IL13","IL15",
    "IL16","IL18","IL33","IL5","IL6","IL6R","IL7","IL9","KDR","KLK6","MAPT",
    "MDH1","MSLN","NEFH","NEFL","NGF","NPTX1","NPTX2","NPTXR","NPY","NRGN",
    "PARK7","PDGFRB","PDLIM5","PGF","PGK1","POSTN","PRDX6","PSEN1","pTau-181",
    "pTau-217","pTau-231","PTN","RUVBL2","S100A12","S100B","SAA1","SFRP1",
    "SFTPD","SLIT2","SMOC1","SNAP25","SNCA","SNCB","SOD1","SQSTM1","TAFA5",
    "TARDBP","TEK","TIMP3","TNF","TREM1","TREM2","UBB","UCHL1","VCAM1","VEGFA",
    "VEGFD","VGF","VSNL1","YWHAG"
]

In [ ]:
matched = [name for name in row_names if name in proteins_to_keep]
len(matched)

In [ ]:
# assuming your DataFrame is called df and the column is 'Target'
Data = Data[Data.index.isin(proteins_to_keep)]
Data

In [ ]:
import re
import pandas as pd

def remove_prefix(col_name):
    pattern = r'.*?(Control|NoMut|MutBpos|MutCpos|MutB)'
    match = re.search(pattern, col_name)
    if match:
        return col_name[match.start(1):]
    else:
        return col_name

# Apply the function to the DataFrame columns
Data.columns = [remove_prefix(col) for col in Data.columns]
Data

In [ ]:
list(Data.columns)

In [ ]:
# Assuming your DataFrame is called Data_csf
counts = {
    'Control': sum('Control' in col for col in Data.columns),
    'NoMut': sum('NoMut' in col for col in Data.columns),
    'MutBpos': sum('MutBpos' in col for col in Data.columns),
    'MutCpos': sum('MutCpos' in col for col in Data.columns),
    'MutB-Baseline': sum('MutB-Baseline' in col for col in Data.columns)
}
print(counts)

In [ ]:
# Remove columns with 'DrugA-12months' or 'DrugB-4month' in their names
filtered_df = Data.loc[:, ~Data.columns.str.contains('DrugA|DrugB|MutC')]
filtered_df

In [ ]:
list(filtered_df.columns)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Complete color assignment for all columns in filtered_df
samples = {
    'Control-1': 'DimGray',
    'Control-30': 'DimGray',
    'Control-11': 'DimGray',
    'NoMut Case-3': 'green',
    'MutBpos-2': '#800080',
    'Control-27': 'DimGray',
    'NoMut Case-29': 'green',
    'NoMut Case-17': 'green',
    'Control-10': 'DimGray',
    'NoMut Case-8': 'green',
    'NoMut Case-2': 'green',
    'NoMut Case-16': 'green',
    'Control-9': 'DimGray',
    'MutBpos-21': '#800080',
    'Control-22': 'DimGray',
    'NoMut Case-18': 'green',
    'NoMut Case-33': 'green',
    'NoMut Case-24': 'green',
    'Control-25': 'DimGray',
    'NoMut Case-32': 'green',
    'NoMut Case-10': 'green',
    'MutBpos-15': '#800080',
    'Control-13': 'DimGray',
    'MutBpos-16': '#800080',
    'Control-5': 'DimGray',
    'Control-19': 'DimGray',
    'MutBpos-4': '#800080',
    'NoMut Case-30': 'green',
    'Control-7': 'DimGray',
    'Control-26': 'DimGray',
    'MutBpos-13': '#800080',
    'NoMut Case-22': 'green',
    'NoMut Case-13': 'green',
    'NoMut Case-20': 'green',
    'Control-17': 'DimGray',
    'NoMut Case-23': 'green',
    'Control-16': 'DimGray',
    'Control-24': 'DimGray',
    'NoMut Case-4': 'green',
    'NoMut Case-11': 'green',
    'Control-20': 'DimGray',
    'MutBpos-12': '#800080',
    'NoMut Case-7': 'green',
    'Control-31': 'DimGray',
    'Control-4': 'DimGray',
    'Control-28': 'DimGray',
    'NoMut Case-26': 'green',
    'NoMut Case-28': 'green',
    'Control-34': 'DimGray',
    'NoMut Case-34': 'green',
    'NoMut Case-14': 'green',
    'MutBpos-17': '#800080',
    'MutBpos-10': '#800080',
    'MutBpos-18': '#800080',
    'Control-3': 'DimGray',
    'MutBpos-14': '#800080',
    'Control-6': 'DimGray',
    'MutBpos-11': '#800080',
    'Control-23': 'DimGray',
    'Control-33': 'DimGray',
    'MutBpos-3': '#800080',
    'Control-2': 'DimGray',
    'Control-32': 'DimGray',
    'MutBpos-1': '#800080',
    'MutBpos-6': '#800080',
    'Control-15': 'DimGray',
    'NoMut Case-5': 'green',
    'Control-29': 'DimGray',
    'Control-12': 'DimGray',
    'NoMut Case-31': 'green',
    'Control-35': 'DimGray',
    'MutBpos-5': '#800080',
    'MutBpos-8': '#800080',
    'Control-14': 'DimGray',
    'Control-21': 'DimGray',
    'Control-18': 'DimGray',
    'NoMut Case-12': 'green',
    'NoMut Case-9': 'green',
    'NoMut Case-35': 'green',
    'NoMut Case-1': 'green',
    'MutBpos-20': '#800080',
    'NoMut Case-6': 'green',
    'Control-37': 'DimGray',
    'NoMut Case-25': 'green',
    'NoMut Case-27': 'green',
    'Control-36': 'DimGray',
    'NoMut Case-15': 'green',
    'Control-38': 'DimGray',
    'NoMut Case-21': 'green',
    'NoMut Case-19': 'green',
    'MutBpos-19': '#800080',
    'MutBpos-9': '#800080',
    'MutBpos-7': '#800080',
    'Control-8': 'DimGray',
    'MutB-Baseline-1': '#800080',
    'MutB-Baseline-2': '#800080',
    'MutB-Baseline-3': '#800080',
    'MutB-Baseline-4': '#800080',
    'MutB-Baseline-5': '#800080',
    'MutB-Baseline-6': '#800080',
    'MutB-Baseline-7': '#800080'
}

# Assign colors to all columns of your dataframe using the updated mapping
column_colors = [samples[col] for col in filtered_df.columns]

plt.figure(figsize=(50, 50))

plot4 = sns.clustermap(
    filtered_df, z_score=0, cmap="vlag", figsize=(20, 40),
    vmin=-1, vmax=1, cbar_kws={"label": "z score"},
    col_colors=[column_colors],
    cbar_pos=(1, 0.5, 0.05, 0.15), xticklabels=True
)

# Set x tick labels color to white
plot4.ax_heatmap.tick_params(axis='x', colors='white')

# Increase font size of color bar tick labels and label
plot4.cax.set_yticklabels(plot4.cax.get_yticklabels(), fontsize=15)
plot4.cax.set_ylabel('z score', fontsize=15)

# Example: Define legend labels for specific columns of samples
legend_labels = {
    "Control-1": "Control",
    "NoMut Case-3": "NoMut",
    "MutBpos-21": "MutBpos"
}

# Create legend handles from defined legend_labels
handles1 = [plt.Rectangle((0, 0), 1.5, 1.5, color=samples[col]) for col in legend_labels.keys()]
legend1 = plot4.ax_col_dendrogram.legend(
    handles=handles1,
    labels=legend_labels.values(),
    loc='upper left',
    bbox_to_anchor=(1.05, -0.25),
    fontsize='xx-large',
    frameon=False
)
plot4.ax_col_dendrogram.add_artist(legend1)

plt.savefig('NoMut_MutB_Control_all_prots_clustermap.png', bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(filtered_df)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Create an instance of PCA with more components
pca_all = PCA(n_components=len(filtered_df.columns))

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_all.fit(data_transposed)

# Calculate the explained variance ratio (as a percentage) for each principal component
explained_variance_ratio_percentage = pca_all.explained_variance_ratio_ * 100

# Create a figure with one subplot
fig, ax = plt.subplots(figsize=(50, 20))

# Plot the bar plot for explained variance of each principal component
ax.bar(np.arange(1, len(filtered_df.columns) + 1), explained_variance_ratio_percentage, color='skyblue', alpha=0.8)

# Customize the plot
ax.set_xlabel('Principal Component')
ax.set_ylabel('Percentage of Variance Explained')
ax.set_title('Percentage of Variance Explained by Principal Components')
ax.set_xticks(np.arange(1, len(filtered_df.columns) + 1))
#ax.grid(True)

plt.tight_layout()
plt.savefig('MutB NoMut Eigen_values_Bar_grah.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2
import re

Data_csf = filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]
noMut_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# MODIFICATION: Combine Case subtypes for centroid calculation
case_cols = mutB_cols + noMut_cols

# Custom labels for annotation
custom_labels = {}
for col in Data_csf.columns:
    # Extract the sample number after the LAST hyphen
    match = re.search(r'-(\\d+)$', col)
    col_number = match.group(1) if match else None
    if col in control_cols and col_number:
        custom_labels[col] = f'HC{col_number}'
    elif col in mutB_cols and col_number:
        custom_labels[col] = f'MutB_{col_number}'
    elif col in noMut_cols and col_number:
        custom_labels[col] = f'SALS_{col_number}'
    elif col_number:
        custom_labels[col] = f'O_{col_number}'
    else:
        custom_labels[col] = col  # fallback to raw name if number not found


column_colors = {
    col: 'DimGray' if col in control_cols else
         '#800080' if col in mutB_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_csf.columns
}

# MODIFICATION: Updated legend to show both Case subtypes separately but one combined group
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# MODIFICATION: Define groups for centroid/ellipse plotting - combined Case
selected_columns_list = [case_cols, control_cols]

# MODIFICATION: Centroid color for combined Case (using a blended or new color)
centroid_colors = {
    case_cols[0]: '#B8860B',  # Dark goldenrod for combined Case
    control_cols[0]: 'DimGray'
}

# MODIFICATION: Outline colors for ellipses
outline_colors = ['#B8860B', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'  # Diamond for MutBpos
    elif col in noMut_cols:
        marker = '^'  # Triangle for NoMut
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5)
   # plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=0,
          #   weight='bold', color='black', ha='center', va='center')

# Legend - still showing individual Case subtypes
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# MODIFICATION: Centroids and confidence ellipses - combined Case
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2, linestyle='dashed')
    plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Combined Case Centroid and Confidence Ellipse')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('PC1 VS PC2 Case Centroid.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2
import re

Data_csf = filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]
noMut_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# MODIFICATION: Combine Case subtypes for centroid calculation
case_cols = mutB_cols + noMut_cols

# Custom labels for annotation
custom_labels = {}
for col in Data_csf.columns:
    # Extract the sample number after the LAST hyphen
    match = re.search(r'-(\\d+)$', col)
    col_number = match.group(1) if match else None
    if col in control_cols and col_number:
        custom_labels[col] = f'HC{col_number}'
    elif col in mutB_cols and col_number:
        custom_labels[col] = f'MutB_{col_number}'
    elif col in noMut_cols and col_number:
        custom_labels[col] = f'SALS_{col_number}'
    elif col_number:
        custom_labels[col] = f'O_{col_number}'
    else:
        custom_labels[col] = col  # fallback to raw name if number not found


column_colors = {
    col: 'DimGray' if col in control_cols else
         '#800080' if col in mutB_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_csf.columns
}

# MODIFICATION: Updated legend to show both Case subtypes separately but one combined group
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# MODIFICATION: Define groups for centroid/ellipse plotting - combined Case
selected_columns_list = [case_cols, control_cols]

# MODIFICATION: Centroid color for combined Case (using a blended or new color)
centroid_colors = {
    case_cols[0]: '#B8860B',  # Dark goldenrod for combined Case
    control_cols[0]: 'DimGray'
}

# MODIFICATION: Outline colors for ellipses
outline_colors = ['#B8860B', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'  # Diamond for MutBpos
    elif col in noMut_cols:
        marker = '^'  # Triangle for NoMut
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5)
    #plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=0,
            # weight='bold', color='black', ha='center', va='center')

# Legend - still showing individual Case subtypes
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# MODIFICATION: Centroids and confidence ellipses - combined Case
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2, linestyle='dashed')
    plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Combined Case Centroid and Confidence Ellipse')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('PC1 VS PC2 Case Centroid wo points.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2
import re

Data_csf = filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]
noMut_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# MODIFICATION: Combine Case subtypes for centroid calculation
case_cols = mutB_cols + noMut_cols

# Custom labels for annotation
custom_labels = {}
for col in Data_csf.columns:
    # Extract the sample number after the LAST hyphen
    match = re.search(r'-(\\d+)$', col)
    col_number = match.group(1) if match else None
    if col in control_cols and col_number:
        custom_labels[col] = f'HC{col_number}'
    elif col in mutB_cols and col_number:
        custom_labels[col] = f'MutB_{col_number}'
    elif col in noMut_cols and col_number:
        custom_labels[col] = f'SALS_{col_number}'
    elif col_number:
        custom_labels[col] = f'O_{col_number}'
    else:
        custom_labels[col] = col  # fallback to raw name if number not found


column_colors = {
    col: 'DimGray' if col in control_cols else
         '#800080' if col in mutB_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_csf.columns
}

# MODIFICATION: Updated legend to show both Case subtypes separately but one combined group
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# MODIFICATION: Define groups for centroid/ellipse plotting - combined Case
selected_columns_list = [case_cols, control_cols]

# MODIFICATION: Centroid color for combined Case (using a blended or new color)
centroid_colors = {
    case_cols[0]: '#B8860B',  # Dark goldenrod for combined Case
    control_cols[0]: 'DimGray'
}

# MODIFICATION: Outline colors for ellipses
outline_colors = ['#B8860B', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'  # Diamond for MutBpos
    elif col in noMut_cols:
        marker = '^'  # Triangle for NoMut
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5)
   # plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=0,
          #   weight='bold', color='black', ha='center', va='center')

# Legend - still showing individual Case subtypes
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# MODIFICATION: Centroids and confidence ellipses - combined Case
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    
    # Extract only PC1 and PC3 (columns 0 and 2)
    group_data_2d = group_data[:, [0, 2]]  # <-- KEY FIX
    
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data_2d)
    centroid = kmeans.cluster_centers_[0]
    
    # Now centroid is 2D, so unpacking works correctly
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data_2d.T)  # <-- Also fix covariance calculation
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2, linestyle='dashed')
    plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Combined Case Centroid and Confidence Ellipse')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('PC1 VS PC3 Case Centroid.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2
import re

Data_csf = filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]
noMut_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# MODIFICATION: Combine Case subtypes for centroid calculation
case_cols = mutB_cols + noMut_cols

# Custom labels for annotation
custom_labels = {}
for col in Data_csf.columns:
    # Extract the sample number after the LAST hyphen
    match = re.search(r'-(\\d+)$', col)
    col_number = match.group(1) if match else None
    if col in control_cols and col_number:
        custom_labels[col] = f'HC{col_number}'
    elif col in mutB_cols and col_number:
        custom_labels[col] = f'MutB_{col_number}'
    elif col in noMut_cols and col_number:
        custom_labels[col] = f'SALS_{col_number}'
    elif col_number:
        custom_labels[col] = f'O_{col_number}'
    else:
        custom_labels[col] = col  # fallback to raw name if number not found


column_colors = {
    col: 'DimGray' if col in control_cols else
         '#800080' if col in mutB_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_csf.columns
}

# MODIFICATION: Updated legend to show both Case subtypes separately but one combined group
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# MODIFICATION: Define groups for centroid/ellipse plotting - combined Case
selected_columns_list = [case_cols, control_cols]

# MODIFICATION: Centroid color for combined Case (using a blended or new color)
centroid_colors = {
    case_cols[0]: '#B8860B',  # Dark goldenrod for combined Case
    control_cols[0]: 'DimGray'
}

# MODIFICATION: Outline colors for ellipses
outline_colors = ['#B8860B', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'  # Diamond for MutBpos
    elif col in noMut_cols:
        marker = '^'  # Triangle for NoMut
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5)
   # plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=0,
          #   weight='bold', color='black', ha='center', va='center')

# Legend - still showing individual Case subtypes
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# MODIFICATION: Centroids and confidence ellipses - combined Case
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    
    # Extract only PC1 and PC3 (columns 0 and 2)
    group_data_2d = group_data[:, [0, 2]]  # <-- KEY FIX
    
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data_2d)
    centroid = kmeans.cluster_centers_[0]
    
    # Now centroid is 2D, so unpacking works correctly
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data_2d.T)  # <-- Also fix covariance calculation
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2, linestyle='dashed')
    plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Combined Case Centroid and Confidence Ellipse')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('PC1 VS PC3 Case Centroid wo points.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2
import re

Data_csf = filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]
noMut_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# MODIFICATION: Combine Case subtypes for centroid calculation
case_cols = mutB_cols + noMut_cols

# Custom labels for annotation
custom_labels = {}
for col in Data_csf.columns:
    # Extract the sample number after the LAST hyphen
    match = re.search(r'-(\\d+)$', col)
    col_number = match.group(1) if match else None
    if col in control_cols and col_number:
        custom_labels[col] = f'HC{col_number}'
    elif col in mutB_cols and col_number:
        custom_labels[col] = f'MutB_{col_number}'
    elif col in noMut_cols and col_number:
        custom_labels[col] = f'SALS_{col_number}'
    elif col_number:
        custom_labels[col] = f'O_{col_number}'
    else:
        custom_labels[col] = col  # fallback to raw name if number not found


column_colors = {
    col: 'DimGray' if col in control_cols else
         '#800080' if col in mutB_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_csf.columns
}

# MODIFICATION: Updated legend to show both Case subtypes separately but one combined group
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# MODIFICATION: Define groups for centroid/ellipse plotting - combined Case
selected_columns_list = [case_cols, control_cols]

# MODIFICATION: Centroid color for combined Case (using a blended or new color)
centroid_colors = {
    case_cols[0]: '#B8860B',  # Dark goldenrod for combined Case
    control_cols[0]: 'DimGray'
}

# MODIFICATION: Outline colors for ellipses
outline_colors = ['#B8860B', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'  # Diamond for MutBpos
    elif col in noMut_cols:
        marker = '^'  # Triangle for NoMut
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 1], pca_result[i, 2], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5)
   # plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=0,
          #   weight='bold', color='black', ha='center', va='center')

# Legend - still showing individual Case subtypes
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 1], pca_result[i, 2], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# MODIFICATION: Centroids and confidence ellipses - combined Case
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    
    # Extract only PC2 and PC3 (columns 1 and 2)
    group_data_2d = group_data[:, [1, 2]]  # <-- KEY FIX
    
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data_2d)
    centroid = kmeans.cluster_centers_[0]
    
    # Now centroid is 2D, so unpacking works correctly
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data_2d.T)  # <-- Also fix covariance calculation
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2, linestyle='dashed')
    plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Combined Case Centroid and Confidence Ellipse')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('PC2 VS PC3 Case Centroid.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2
import re

Data_csf = filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]
noMut_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# MODIFICATION: Combine Case subtypes for centroid calculation
case_cols = mutB_cols + noMut_cols

# Custom labels for annotation
custom_labels = {}
for col in Data_csf.columns:
    # Extract the sample number after the LAST hyphen
    match = re.search(r'-(\\d+)$', col)
    col_number = match.group(1) if match else None
    if col in control_cols and col_number:
        custom_labels[col] = f'HC{col_number}'
    elif col in mutB_cols and col_number:
        custom_labels[col] = f'MutB_{col_number}'
    elif col in noMut_cols and col_number:
        custom_labels[col] = f'SALS_{col_number}'
    elif col_number:
        custom_labels[col] = f'O_{col_number}'
    else:
        custom_labels[col] = col  # fallback to raw name if number not found


column_colors = {
    col: 'DimGray' if col in control_cols else
         '#800080' if col in mutB_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_csf.columns
}

# MODIFICATION: Updated legend to show both Case subtypes separately but one combined group
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# MODIFICATION: Define groups for centroid/ellipse plotting - combined Case
selected_columns_list = [case_cols, control_cols]

# MODIFICATION: Centroid color for combined Case (using a blended or new color)
centroid_colors = {
    case_cols[0]: '#B8860B',  # Dark goldenrod for combined Case
    control_cols[0]: 'DimGray'
}

# MODIFICATION: Outline colors for ellipses
outline_colors = ['#B8860B', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'  # Diamond for MutBpos
    elif col in noMut_cols:
        marker = '^'  # Triangle for NoMut
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 1], pca_result[i, 2], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5)
   # plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=0,
          #   weight='bold', color='black', ha='center', va='center')

# Legend - still showing individual Case subtypes
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 1], pca_result[i, 2], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# MODIFICATION: Centroids and confidence ellipses - combined Case
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    
    # Extract only PC2 and PC3 (columns 1 and 2)
    group_data_2d = group_data[:, [1, 2]]  # <-- KEY FIX
    
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data_2d)
    centroid = kmeans.cluster_centers_[0]
    
    # Now centroid is 2D, so unpacking works correctly
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data_2d.T)  # <-- Also fix covariance calculation
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2, linestyle='dashed')
    plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Combined Case Centroid and Confidence Ellipse')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('PC2 VS PC3 Case Centroid wo points.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2
import re

Data_csf = filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]
noMut_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# Custom labels for annotation
custom_labels = {}
for col in Data_csf.columns:
    # Extract the sample number after the LAST hyphen
    match = re.search(r'-(\d+)$', col)
    col_number = match.group(1) if match else None
    if col in control_cols and col_number:
        custom_labels[col] = f'HC{col_number}'
    elif col in mutB_cols and col_number:
        custom_labels[col] = f'MutB_{col_number}'
    elif col in noMut_cols and col_number:
        custom_labels[col] = f'SALS_{col_number}'
    elif col_number:
        custom_labels[col] = f'O_{col_number}'
    else:
        custom_labels[col] = col  # fallback to raw name if number not found


column_colors = {
    col: 'DimGray' if col in control_cols else
         '#800080' if col in mutB_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_csf.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [mutB_cols, noMut_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    noMut_cols[0]: 'green',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'green', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5)
    plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
             weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2, linestyle='dashed')
    plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Centroids and Confidence Ellipses')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('PC1 VS PC2.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

Data_csf = filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]
noMut_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# Custom labels for annotation
custom_labels = {}
for col in Data_csf.columns:
    # Extract the sample number after the LAST hyphen
    match = re.search(r'-(\d+)$', col)
    col_number = match.group(1) if match else None
    if col in control_cols and col_number:
        custom_labels[col] = f'HC{col_number}'
    elif col in mutB_cols and col_number:
        custom_labels[col] = f'MutB_{col_number}'
    elif col in noMut_cols and col_number:
        custom_labels[col] = f'SALS_{col_number}'
    elif col_number:
        custom_labels[col] = f'O_{col_number}'
    else:
        custom_labels[col] = col  # fallback to raw name if number not found

column_colors = {
    col: 'DimGray' if col in control_cols else
         '#800080' if col in mutB_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_csf.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [mutB_cols, noMut_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    noMut_cols[0]: 'green',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'green', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5)
   # plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
        #     weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
   # ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
   #                   facecolor='none', linewidth=2, linestyle='dashed')
   # plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Centroids and Confidence Ellipses')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('PC1 VS PC2 centroids.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

Data_csf = filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]
noMut_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# Custom labels for annotation
custom_labels = {}
for col in Data_csf.columns:
    # Extract the sample number after the LAST hyphen
    match = re.search(r'-(\d+)$', col)
    col_number = match.group(1) if match else None
    if col in control_cols and col_number:
        custom_labels[col] = f'HC{col_number}'
    elif col in mutB_cols and col_number:
        custom_labels[col] = f'MutB_{col_number}'
    elif col in noMut_cols and col_number:
        custom_labels[col] = f'SALS_{col_number}'
    elif col_number:
        custom_labels[col] = f'O_{col_number}'
    else:
        custom_labels[col] = col  # fallback to raw name if number not found
        
column_colors = {
    col: 'DimGray' if col in control_cols else
         '#800080' if col in mutB_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_csf.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [mutB_cols, noMut_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    noMut_cols[0]: 'green',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'green', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5)
    plt.text(pca_result[i, 0]-0.1, pca_result[i, 2]+0.1, custom_labels[col], fontsize=7,
             weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices][:, [0, 2]]  # PC1 vs PC3
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2, linestyle='dashed')
    plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Centroids and Confidence Ellipses')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('PC1 VS PC3.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

Data_csf = filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]
noMut_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# Custom labels for annotation
custom_labels = {}
for col in Data_csf.columns:
    # Extract the sample number after the LAST hyphen
    match = re.search(r'-(\d+)$', col)
    col_number = match.group(1) if match else None
    if col in control_cols and col_number:
        custom_labels[col] = f'HC{col_number}'
    elif col in mutB_cols and col_number:
        custom_labels[col] = f'MutB_{col_number}'
    elif col in noMut_cols and col_number:
        custom_labels[col] = f'SALS_{col_number}'
    elif col_number:
        custom_labels[col] = f'O_{col_number}'
    else:
        custom_labels[col] = col  # fallback to raw name if number not found

column_colors = {
    col: 'DimGray' if col in control_cols else
         '#800080' if col in mutB_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_csf.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [mutB_cols, noMut_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    noMut_cols[0]: 'green',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'green', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5)
    #plt.text(pca_result[i, 0]-0.1, pca_result[i, 2]+0.1, custom_labels[col], fontsize=7,
       #      weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices][:, [0, 2]]  # PC1 vs PC3
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    #ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
    #                  facecolor='none', linewidth=2, linestyle='dashed')
    #plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Centroids and Confidence Ellipses')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('PC1 VS PC3 centroids.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

Data_csf = filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]
noMut_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# Custom labels for annotation
custom_labels = {}
for col in Data_csf.columns:
    # Extract the sample number after the LAST hyphen
    match = re.search(r'-(\d+)$', col)
    col_number = match.group(1) if match else None
    if col in control_cols and col_number:
        custom_labels[col] = f'HC{col_number}'
    elif col in mutB_cols and col_number:
        custom_labels[col] = f'MutB_{col_number}'
    elif col in noMut_cols and col_number:
        custom_labels[col] = f'SALS_{col_number}'
    elif col_number:
        custom_labels[col] = f'O_{col_number}'
    else:
        custom_labels[col] = col  # fallback to raw name if number not found

column_colors = {
    col: 'DimGray' if col in control_cols else
         '#800080' if col in mutB_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_csf.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [mutB_cols, noMut_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    noMut_cols[0]: 'green',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'green', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 1], pca_result[i, 2], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5)
    plt.text(pca_result[i, 1]-0.1, pca_result[i, 2]+0.1, custom_labels[col], fontsize=7,
             weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices][:, [1, 2]]  # PC2 vs PC3
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2, linestyle='dashed')
    plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Centroids and Confidence Ellipses')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('PC2 VS PC3.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

Data_csf = filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]
noMut_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# Custom labels for annotation
custom_labels = {}
for col in Data_csf.columns:
    # Extract the sample number after the LAST hyphen
    match = re.search(r'-(\d+)$', col)
    col_number = match.group(1) if match else None
    if col in control_cols and col_number:
        custom_labels[col] = f'HC{col_number}'
    elif col in mutB_cols and col_number:
        custom_labels[col] = f'MutB_{col_number}'
    elif col in noMut_cols and col_number:
        custom_labels[col] = f'SALS_{col_number}'
    elif col_number:
        custom_labels[col] = f'O_{col_number}'
    else:
        custom_labels[col] = col  # fallback to raw name if number not found

column_colors = {
    col: 'DimGray' if col in control_cols else
         '#800080' if col in mutB_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_csf.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [mutB_cols, noMut_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    noMut_cols[0]: 'green',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'green', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 1], pca_result[i, 2], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5)
    #plt.text(pca_result[i, 1]-0.1, pca_result[i, 2]+0.1, custom_labels[col], fontsize=7,
     #        weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices][:, [1, 2]]  # PC2 vs PC3
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    #ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
     #                 facecolor='none', linewidth=2, linestyle='dashed')
    #plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Centroids and Confidence Ellipses')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('PC2 VS PC3 centroids.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2
import re

# Assume filtered_df is already defined
Data_csf = filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]
noMut_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# Custom labels for annotation
custom_labels = {}
for col in Data_csf.columns:
    match = re.search(r'-(\d+)$', col)
    col_number = match.group(1) if match else None
    if col in control_cols and col_number:
        custom_labels[col] = f'HC{col_number}'
    elif col in mutB_cols and col_number:
        custom_labels[col] = f'MutB_{col_number}'
    elif col in noMut_cols and col_number:
        custom_labels[col] = f'SALS_{col_number}'
    elif col_number:
        custom_labels[col] = f'O_{col_number}'
    else:
        custom_labels[col] = col  # fallback to raw name if number not found

column_colors = {
    col: 'DimGray' if col in control_cols else
         '#800080' if col in mutB_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_csf.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# Define groups for centroid plotting
selected_columns_list = [mutB_cols, noMut_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    noMut_cols[0]: 'green',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'green', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    ax.scatter(pca_result[i, 0], pca_result[i, 1], pca_result[i, 2], color=color, marker=marker, s=100,
               edgecolor='k', linewidth=0.5)
    ax.text(pca_result[i, 0], pca_result[i, 1], pca_result[i, 2], custom_labels[col], fontsize=7,
            weight='bold', color='black', ha='center', va='center')

# Add centroids
for group_cols in selected_columns_list:
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices, :3]  # PC1, PC2, PC3
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    ax.scatter(*centroid, marker='X', s=300, color=centroid_colors[group_cols[0]],
               edgecolor='k', linewidth=1.5)

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    ax.scatter(pca_result[i, 0], pca_result[i, 1], pca_result[i, 2], color=color, marker=marker, s=100,
               edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
ax.set_zlabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
ax.set_title('3D PCA of Case CSF Samples with Centroids')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('3D_PCA.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2
import re

# Assume filtered_df is already defined
Data_csf = filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]
noMut_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# Custom labels for annotation
custom_labels = {}
for col in Data_csf.columns:
    match = re.search(r'-(\d+)$', col)
    col_number = match.group(1) if match else None
    if col in control_cols and col_number:
        custom_labels[col] = f'HC{col_number}'
    elif col in mutB_cols and col_number:
        custom_labels[col] = f'MutB_{col_number}'
    elif col in noMut_cols and col_number:
        custom_labels[col] = f'SALS_{col_number}'
    elif col_number:
        custom_labels[col] = f'O_{col_number}'
    else:
        custom_labels[col] = col  # fallback to raw name if number not found

column_colors = {
    col: 'DimGray' if col in control_cols else
         '#800080' if col in mutB_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_csf.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# Define groups for centroid plotting
selected_columns_list = [mutB_cols, noMut_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    noMut_cols[0]: 'green',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'green', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    ax.scatter(pca_result[i, 0], pca_result[i, 1], pca_result[i, 2], color=color, marker=marker, s=0,
               edgecolor='k', linewidth=0.5)
   # ax.text(pca_result[i, 0], pca_result[i, 1], pca_result[i, 2], custom_labels[col], fontsize=7,
     #       weight='bold', color='black', ha='center', va='center')

# Add centroids
for group_cols in selected_columns_list:
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices, :3]  # PC1, PC2, PC3
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    ax.scatter(*centroid, marker='X', s=300, color=centroid_colors[group_cols[0]],
               edgecolor='k', linewidth=1.5)

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    ax.scatter(pca_result[i, 0], pca_result[i, 1], pca_result[i, 2], color=color, marker=marker, s=0,
               edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
ax.set_zlabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
ax.set_title('3D PCA of Case CSF Samples with Centroids')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('3D_PCA_Centroids.png')
plt.show()

In [ ]:
# Extract PC loadings (components show how much each gene contributes to each PC)
pc_loadings = pca.components_

# Create DataFrame: genes as index, PCs as columns
pc_loadings_df = pd.DataFrame(pc_loadings.T, columns=['PC1', 'PC2', 'PC3'], index=Data_csf.index)

# Save to CSV
pc_loadings_df.to_csv('pc_loadings.csv')

# Display the loadings
pc_loadings_df

In [ ]:
# Extract only PC1 loadings from the full loadings DataFrame
pc1_sorted_df = pd.DataFrame(pc_loadings_df['PC1'])

# Sort genes by their PC1 loading in descending order
pc1_loading_df_sorted = pc1_sorted_df.sort_values(by='PC1', ascending=False)

# Save the sorted PC1 loadings to CSV
pc1_loading_df_sorted.to_csv('pc1_loading_df_sorted.csv')

# Display the sorted DataFrame
pc1_loading_df_sorted

In [ ]:
import matplotlib.colors as mcolors

# Define custom colormap
custom_cmap = mcolors.TwoSlopeNorm(vmin=pc1_loading_df_sorted['PC1'].min(),
                                   vcenter=0,
                                   vmax=pc1_loading_df_sorted['PC1'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC1 values
colors = plt.cm.RdBu_r(custom_cmap(pc1_loading_df_sorted['PC1']))

# Create the bar plot
bars = ax.bar(pc1_loading_df_sorted.index, pc1_loading_df_sorted['PC1'], color=colors)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC1 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC1 Loadings')
ax.set_title('PC1 Loadings for Proteins')

# Rotate gene names for better readability
ax.set_xticklabels(pc1_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('MutB Sporoadic All PC1 loadings.png')
plt.show()

In [ ]:
# Extract only PC2 loadings from the full loadings DataFrame
pc2_sorted_df = pd.DataFrame(pc_loadings_df['PC2'])

# Sort genes by their PC2 loading in descending order
pc2_loading_df_sorted = pc2_sorted_df.sort_values(by='PC2', ascending=False)

# Save the sorted PC2 loadings to CSV
pc2_loading_df_sorted.to_csv('pc2_loading_df_sorted.csv')

# Display the sorted DataFrame
pc2_loading_df_sorted

In [ ]:
# Define custom colormap for PC2
custom_cmap_pc2 = mcolors.TwoSlopeNorm(vmin=pc2_loading_df_sorted['PC2'].min(),
                                       vcenter=0,
                                       vmax=pc2_loading_df_sorted['PC2'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC2 values
colors = plt.cm.RdBu_r(custom_cmap_pc2(pc2_loading_df_sorted['PC2']))

# Create the bar plot
bars = ax.bar(pc2_loading_df_sorted.index, pc2_loading_df_sorted['PC2'], color=colors)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap_pc2)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC2 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC2 Loadings')
ax.set_title('PC2 Loadings for Proteins')

# Rotate gene names for better readability
ax.set_xticklabels(pc2_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('MutB Sporoadic All PC2 loadings.png')
plt.show()

In [ ]:
# Extract only PC3 loadings from the full loadings DataFrame
pc3_sorted_df = pd.DataFrame(pc_loadings_df['PC3'])

# Sort genes by their PC3 loading in descending order
pc3_loading_df_sorted = pc3_sorted_df.sort_values(by='PC3', ascending=False)

# Save the sorted PC3 loadings to CSV
pc3_loading_df_sorted.to_csv('pc3_loading_df_sorted.csv')

# Display the sorted DataFrame
pc3_loading_df_sorted

In [ ]:
# Define custom colormap for PC3
custom_cmap_pc3 = mcolors.TwoSlopeNorm(vmin=pc3_loading_df_sorted['PC3'].min(),
                                       vcenter=0,
                                       vmax=pc3_loading_df_sorted['PC3'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC3 values
colors = plt.cm.RdBu_r(custom_cmap_pc3(pc3_loading_df_sorted['PC3']))

# Create the bar plot
bars = ax.bar(pc3_loading_df_sorted.index, pc3_loading_df_sorted['PC3'], color=colors)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap_pc3)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC3 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC3 Loadings')
ax.set_title('PC3 Loadings for Proteins')

# Rotate gene names for better readability
ax.set_xticklabels(pc3_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('MutB Sporoadic All PC3 loadings.png')
plt.show()

In [ ]:
# Sort the DataFrame by 'PC1' values in descending order
pc1_loading_df_sorted = pc1_loading_df_sorted.sort_values(by='PC1', ascending=False)

# Extract the top 20 and bottom 20 genes
top_20_genes_PC1 = pc1_loading_df_sorted.head(20)
bottom_20_genes_PC1 = pc1_loading_df_sorted.tail(20)

# Combine both into one DataFrame for convenience
top_bottom_20_genes_PC1 = pd.concat([top_20_genes_PC1, bottom_20_genes_PC1])

# Save the results to a CSV file
top_bottom_20_genes_PC1.to_csv('top_bottom_20_genes_PC1.csv')
top_bottom_20_genes_PC1

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Define custom colormap for PC1 with vcenter set to zero
custom_cmap = mcolors.TwoSlopeNorm(vmin=top_bottom_20_genes_PC1['PC1'].min(),
                                   vcenter=0,
                                   vmax=top_bottom_20_genes_PC1['PC1'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(10, 10))

# Define colors for the bars based on the PC1 values
colors = plt.cm.RdBu_r(custom_cmap(top_bottom_20_genes_PC1['PC1']))

# Create the horizontal bar plot using the index (gene names) as labels
bars = ax.barh(top_bottom_20_genes_PC1.index, top_bottom_20_genes_PC1['PC1'], color=colors)

# Invert the y-axis to have the top gene at the top
ax.invert_yaxis()

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, orientation='vertical')
cbar.set_label('PC1 Loadings')

# Set labels and title
ax.set_xlabel('PC1 Loadings')
ax.set_ylabel('Proteins')
ax.set_title('PC1 Loadings')

# Show the plot
plt.tight_layout()
plt.savefig('Top Bottom 20 PC1 loadings.png')
plt.show()

In [ ]:
# Sort the DataFrame by 'PC2' values in descending order
pc2_loading_df_sorted = pc2_loading_df_sorted.sort_values(by='PC2', ascending=False)

# Extract the top 20 and bottom 20 genes
top_20_genes_PC2 = pc2_loading_df_sorted.head(20)
bottom_20_genes_PC2 = pc2_loading_df_sorted.tail(20)

# Combine top and bottom genes
top_bottom_20_genes_PC2 = pd.concat([top_20_genes_PC2, bottom_20_genes_PC2])

# Save to CSV
top_bottom_20_genes_PC2.to_csv('top_bottom_20_genes_PC2.csv')

# Display
top_bottom_20_genes_PC2

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Define custom colormap for PC2 with center at zero
custom_cmap_pc2 = mcolors.TwoSlopeNorm(vmin=top_bottom_20_genes_PC2['PC2'].min(),
                                       vcenter=0,
                                       vmax=top_bottom_20_genes_PC2['PC2'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(10, 10))

# Define bar colors based on PC2 values
colors = plt.cm.RdBu_r(custom_cmap_pc2(top_bottom_20_genes_PC2['PC2']))

# Create horizontal bar plot using index as gene names
bars = ax.barh(top_bottom_20_genes_PC2.index, top_bottom_20_genes_PC2['PC2'], color=colors)

# Invert y-axis
ax.invert_yaxis()

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap_pc2)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, orientation='vertical')
cbar.set_label('PC2 Loadings')

# Set labels and title
ax.set_xlabel('PC2 Loadings')
ax.set_ylabel('Proteins')
ax.set_title('PC2 Loadings')

# Save and show plot
plt.tight_layout()
plt.savefig('Top Bottom 20 PC2 loadings.png')
plt.show()

In [ ]:
# Sort the DataFrame by 'PC3' values in descending order
pc3_loading_df_sorted = pc3_loading_df_sorted.sort_values(by='PC3', ascending=False)

# Extract the top 20 and bottom 20 genes
top_20_genes_PC3 = pc3_loading_df_sorted.head(20)
bottom_20_genes_PC3 = pc3_loading_df_sorted.tail(20)

# Combine top and bottom genes
top_bottom_20_genes_PC3 = pd.concat([top_20_genes_PC3, bottom_20_genes_PC3])

# Save to CSV
top_bottom_20_genes_PC3.to_csv('top_bottom_20_genes_PC3.csv')

# Display
top_bottom_20_genes_PC3

In [ ]:
# Define custom colormap for PC3 with center at zero
custom_cmap_pc3 = mcolors.TwoSlopeNorm(vmin=top_bottom_20_genes_PC3['PC3'].min(),
                                       vcenter=0,
                                       vmax=top_bottom_20_genes_PC3['PC3'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(10, 10))

# Define bar colors based on PC3 values
colors = plt.cm.RdBu_r(custom_cmap_pc3(top_bottom_20_genes_PC3['PC3']))

# Create horizontal bar plot using index as gene names
bars = ax.barh(top_bottom_20_genes_PC3.index, top_bottom_20_genes_PC3['PC3'], color=colors)

# Invert y-axis
ax.invert_yaxis()

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap_pc3)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, orientation='vertical')
cbar.set_label('PC3 Loadings')

# Set labels and title
ax.set_xlabel('PC3 Loadings')
ax.set_ylabel('Proteins')
ax.set_title('PC3 Loadings')

# Save and show plot
plt.tight_layout()
plt.savefig('Top Bottom 20 PC3 loadings.png')
plt.show()

In [ ]:
# Remove columns with 'DrugA-12months' or 'DrugB-4month' in their names
NoMut_filtered_df = Data.loc[:, ~Data.columns.str.contains('DrugA|DrugB|MutC|MutB')]
NoMut_filtered_df

In [ ]:
list(NoMut_filtered_df.columns)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Complete color assignment for all columns in filtered_df
samples = {
    'Control-1': 'DimGray',
    'Control-30': 'DimGray',
    'Control-11': 'DimGray',
    'NoMut Case-3': 'green',
    'Control-27': 'DimGray',
    'NoMut Case-29': 'green',
    'NoMut Case-17': 'green',
    'Control-10': 'DimGray',
    'NoMut Case-8': 'green',
    'NoMut Case-2': 'green',
    'NoMut Case-16': 'green',
    'Control-9': 'DimGray',
    'Control-22': 'DimGray',
    'NoMut Case-18': 'green',
    'NoMut Case-33': 'green',
    'NoMut Case-24': 'green',
    'Control-25': 'DimGray',
    'NoMut Case-32': 'green',
    'NoMut Case-10': 'green',
    'Control-13': 'DimGray',
    'Control-5': 'DimGray',
    'Control-19': 'DimGray',
    'NoMut Case-30': 'green',
    'Control-7': 'DimGray',
    'Control-26': 'DimGray',
    'NoMut Case-22': 'green',
    'NoMut Case-13': 'green',
    'NoMut Case-20': 'green',
    'Control-17': 'DimGray',
    'NoMut Case-23': 'green',
    'Control-16': 'DimGray',
    'Control-24': 'DimGray',
    'NoMut Case-4': 'green',
    'NoMut Case-11': 'green',
    'Control-20': 'DimGray',
    'NoMut Case-7': 'green',
    'Control-31': 'DimGray',
    'Control-4': 'DimGray',
    'Control-28': 'DimGray',
    'NoMut Case-26': 'green',
    'NoMut Case-28': 'green',
    'Control-34': 'DimGray',
    'NoMut Case-34': 'green',
    'NoMut Case-14': 'green',
    'Control-3': 'DimGray',
    'Control-6': 'DimGray',
    'Control-23': 'DimGray',
    'Control-33': 'DimGray',
    'Control-2': 'DimGray',
    'Control-32': 'DimGray',
    'Control-15': 'DimGray',
    'NoMut Case-5': 'green',
    'Control-29': 'DimGray',
    'Control-12': 'DimGray',
    'NoMut Case-31': 'green',
    'Control-35': 'DimGray',
    'Control-14': 'DimGray',
    'Control-21': 'DimGray',
    'Control-18': 'DimGray',
    'NoMut Case-12': 'green',
    'NoMut Case-9': 'green',
    'NoMut Case-35': 'green',
    'NoMut Case-1': 'green',
    'NoMut Case-6': 'green',
    'Control-37': 'DimGray',
    'NoMut Case-25': 'green',
    'NoMut Case-27': 'green',
    'Control-36': 'DimGray',
    'NoMut Case-15': 'green',
    'Control-38': 'DimGray',
    'NoMut Case-21': 'green',
    'NoMut Case-19': 'green',
    'Control-8': 'DimGray'
}

# Assign colors to all columns of your dataframe using the updated mapping
column_colors = [samples[col] for col in NoMut_filtered_df.columns]

plt.figure(figsize=(50, 50))

plot4 = sns.clustermap(
    NoMut_filtered_df, z_score=0, cmap="vlag", figsize=(20, 40),
    vmin=-1, vmax=1, cbar_kws={"label": "z score"},
    col_colors=[column_colors],
    cbar_pos=(1, 0.5, 0.05, 0.15), xticklabels=True
)

# Set x tick labels color to white
plot4.ax_heatmap.tick_params(axis='x', colors='white')

# Increase font size of color bar tick labels and label
plot4.cax.set_yticklabels(plot4.cax.get_yticklabels(), fontsize=15)
plot4.cax.set_ylabel('z score', fontsize=15)

# Example: Define legend labels for specific columns of samples
legend_labels = {
    "Control-1": "Control",
    "NoMut Case-3": "NoMut",
}

# Create legend handles from defined legend_labels
handles1 = [plt.Rectangle((0, 0), 1.5, 1.5, color=samples[col]) for col in legend_labels.keys()]
legend1 = plot4.ax_col_dendrogram.legend(
    handles=handles1,
    labels=legend_labels.values(),
    loc='upper left',
    bbox_to_anchor=(1.05, -0.25),
    fontsize='xx-large',
    frameon=False
)
plot4.ax_col_dendrogram.add_artist(legend1)

plt.savefig('NoMut_Control_all_prots_clustermap.png', bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(NoMut_filtered_df)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Create an instance of PCA with more components
pca_all = PCA(n_components=len(NoMut_filtered_df.columns))

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_all.fit(data_transposed)

# Calculate the explained variance ratio (as a percentage) for each principal component
explained_variance_ratio_percentage = pca_all.explained_variance_ratio_ * 100

# Create a figure with one subplot
fig, ax = plt.subplots(figsize=(50, 20))

# Plot the bar plot for explained variance of each principal component
ax.bar(np.arange(1, len(NoMut_filtered_df.columns) + 1), explained_variance_ratio_percentage, color='skyblue', alpha=0.8)

# Customize the plot
ax.set_xlabel('Principal Component')
ax.set_ylabel('Percentage of Variance Explained')
ax.set_title('Percentage of Variance Explained by Principal Components')
ax.set_xticks(np.arange(1, len(NoMut_filtered_df.columns) + 1))
#ax.grid(True)

plt.tight_layout()
plt.savefig('NoMut_Eigen_values_Bar_grah.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2
import re

Data_noMut = NoMut_filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_noMut.columns if "Control" in col]
noMut_cols = [col for col in Data_noMut.columns if "NoMut Case" in col]

# Custom labels for annotation
custom_labels = {}
for col in Data_noMut.columns:
    # Extract the last number after the hyphen
    match = re.search(r'-(\d+)$', col)
    col_number = match.group(1) if match else ''
    if col in control_cols:
        custom_labels[col] = f'HC{col_number}'
    elif col in noMut_cols:
        custom_labels[col] = f'SALS_{col_number}'
    else:
        custom_labels[col] = f'O_{col_number}'


column_colors = {
    col: 'DimGray' if col in control_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_noMut.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [noMut_cols, control_cols]

centroid_colors = {
    noMut_cols[0]: 'green',
    control_cols[0]: 'DimGray'
}

outline_colors = ['green', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_noMut)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_noMut.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5)
    plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
             weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_noMut.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_noMut.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2, linestyle='dashed')
    plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of Case NoMut Samples with Centroids and Confidence Ellipses')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('NoMut PC1 VS PC2.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

Data_csf = NoMut_filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_csf.columns if "Control" in col]
noMut_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# Custom labels for annotation
custom_labels = {}
for col in Data_noMut.columns:
    # Extract the last number after the hyphen
    match = re.search(r'-(\d+)$', col)
    col_number = match.group(1) if match else ''
    if col in control_cols:
        custom_labels[col] = f'HC{col_number}'
    elif col in noMut_cols:
        custom_labels[col] = f'SALS_{col_number}'
    else:
        custom_labels[col] = f'O_{col_number}'
column_colors = {
    col: 'DimGray' if col in control_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_csf.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [noMut_cols, control_cols]

centroid_colors = {
    noMut_cols[0]: 'green',
    control_cols[0]: 'DimGray'
}

outline_colors = ['green', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5)
    #plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
      #       weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
   # ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
   #                   facecolor='none', linewidth=2, linestyle='dashed')
   # plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Centroids and Confidence Ellipses')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('NoMut PC1 VS PC2 Centroids.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

Data_noMut = NoMut_filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_noMut.columns if "Control" in col]
noMut_cols = [col for col in Data_noMut.columns if "NoMut Case" in col]

# Custom labels for annotation
custom_labels = {}
for col in Data_noMut.columns:
    # Extract the last number after the hyphen
    match = re.search(r'-(\d+)$', col)
    col_number = match.group(1) if match else ''
    if col in control_cols:
        custom_labels[col] = f'HC{col_number}'
    elif col in noMut_cols:
        custom_labels[col] = f'SALS_{col_number}'
    else:
        custom_labels[col] = f'O_{col_number}'

column_colors = {
    col: 'DimGray' if col in control_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_noMut.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [noMut_cols, control_cols]

centroid_colors = {
    noMut_cols[0]: 'green',
    control_cols[0]: 'DimGray'
}

outline_colors = ['green', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_noMut)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_noMut.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5)
    #plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
    #         weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_noMut.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_noMut.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    # ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
    #                   facecolor='none', linewidth=2, linestyle='dashed')
    # plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of Case NoMut Samples with Centroids and Confidence Ellipses')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('NoMut PC1 VS PC2 Centroids.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

Data_noMut = NoMut_filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_noMut.columns if "Control" in col]
noMut_cols = [col for col in Data_noMut.columns if "NoMut Case" in col]

# Custom labels for annotation
custom_labels = {}
for col in Data_noMut.columns:
    # Extract the last number after the hyphen
    match = re.search(r'-(\d+)$', col)
    col_number = match.group(1) if match else ''
    if col in control_cols:
        custom_labels[col] = f'HC{col_number}'
    elif col in noMut_cols:
        custom_labels[col] = f'SALS_{col_number}'
    else:
        custom_labels[col] = f'O_{col_number}'

column_colors = {
    col: 'DimGray' if col in control_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_noMut.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [noMut_cols, control_cols]

centroid_colors = {
    noMut_cols[0]: 'green',
    control_cols[0]: 'DimGray'
}

outline_colors = ['green', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_noMut)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_noMut.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5)
    plt.text(pca_result[i, 0]-0.1, pca_result[i, 2]+0.1, custom_labels[col], fontsize=7,
             weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_noMut.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_noMut.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices][:, [0, 2]]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                     facecolor='none', linewidth=2, linestyle='dashed')
    plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of Case NoMut Samples with Centroids and Confidence Ellipses (PC1 vs PC3)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('NoMut PC1 VS PC3.png')
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

Data_noMut = NoMut_filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_noMut.columns if "Control" in col]
noMut_cols = [col for col in Data_noMut.columns if "NoMut Case" in col]

# Custom labels for annotation
custom_labels = {}
for col in Data_noMut.columns:
    # Extract the last number after the hyphen
    match = re.search(r'-(\d+)$', col)
    col_number = match.group(1) if match else ''
    if col in control_cols:
        custom_labels[col] = f'HC{col_number}'
    elif col in noMut_cols:
        custom_labels[col] = f'SALS_{col_number}'
    else:
        custom_labels[col] = f'O_{col_number}'

column_colors = {
    col: 'DimGray' if col in control_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_noMut.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [noMut_cols, control_cols]

centroid_colors = {
    noMut_cols[0]: 'green',
    control_cols[0]: 'DimGray'
}

outline_colors = ['green', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_noMut)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_noMut.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5)
   # plt.text(pca_result[i, 0]-0.1, pca_result[i, 2]+0.1, custom_labels[col], fontsize=7,
      #       weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_noMut.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_noMut.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices][:, [0, 2]]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    #ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
    #                  facecolor='none', linewidth=2, linestyle='dashed')
    #plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of Case NoMut Samples with Centroids and Confidence Ellipses (PC1 vs PC3)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('NoMut PC1 VS PC3 Centroids.png')
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

Data_noMut = NoMut_filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_noMut.columns if "Control" in col]
noMut_cols = [col for col in Data_noMut.columns if "NoMut Case" in col]

# Custom labels for annotation
custom_labels = {}
for col in Data_noMut.columns:
    # Extract the last number after the hyphen
    match = re.search(r'-(\d+)$', col)
    col_number = match.group(1) if match else ''
    if col in control_cols:
        custom_labels[col] = f'HC{col_number}'
    elif col in noMut_cols:
        custom_labels[col] = f'SALS_{col_number}'
    else:
        custom_labels[col] = f'O_{col_number}'
column_colors = {
    col: 'DimGray' if col in control_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_noMut.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [noMut_cols, control_cols]

centroid_colors = {
    noMut_cols[0]: 'green',
    control_cols[0]: 'DimGray'
}

outline_colors = ['green', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_noMut)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_noMut.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 1], pca_result[i, 2], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5)
    plt.text(pca_result[i, 1]-0.1, pca_result[i, 2]+0.1, custom_labels[col], fontsize=7,
             weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_noMut.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 1], pca_result[i, 2], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_noMut.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices][:, [1, 2]]  # Use PC2 and PC3
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2, linestyle='dashed')
    plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of Case NoMut Samples with Centroids and Confidence Ellipses (PC2 vs PC3)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('NoMut PC2 VS PC3.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

Data_noMut = NoMut_filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_noMut.columns if "Control" in col]
noMut_cols = [col for col in Data_noMut.columns if "NoMut Case" in col]

# Custom labels for annotation
custom_labels = {}
for col in Data_noMut.columns:
    # Extract the last number after the hyphen
    match = re.search(r'-(\d+)$', col)
    col_number = match.group(1) if match else ''
    if col in control_cols:
        custom_labels[col] = f'HC{col_number}'
    elif col in noMut_cols:
        custom_labels[col] = f'SALS_{col_number}'
    else:
        custom_labels[col] = f'O_{col_number}'

column_colors = {
    col: 'DimGray' if col in control_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_noMut.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [noMut_cols, control_cols]

centroid_colors = {
    noMut_cols[0]: 'green',
    control_cols[0]: 'DimGray'
}

outline_colors = ['green', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_noMut)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_noMut.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 1], pca_result[i, 2], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5)
    #plt.text(pca_result[i, 1]-0.1, pca_result[i, 2]+0.1, custom_labels[col], fontsize=7,
      #       weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_noMut.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 1], pca_result[i, 2], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_noMut.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices][:, [1, 2]]  # Use PC2 and PC3
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    # ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
    #                  facecolor='none', linewidth=2, linestyle='dashed')
    #plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of Case NoMut Samples with Centroids and Confidence Ellipses (PC2 vs PC3)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('NoMut PC2 VS PC3 Centroids.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 unused import

Data_noMut = NoMut_filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_noMut.columns if "Control" in col]
noMut_cols = [col for col in Data_noMut.columns if "NoMut Case" in col]

# Custom labels for annotation
custom_labels = {}
for col in Data_noMut.columns:
    # Extract the last number after the hyphen
    match = re.search(r'-(\d+)$', col)
    col_number = match.group(1) if match else ''
    if col in control_cols:
        custom_labels[col] = f'HC{col_number}'
    elif col in noMut_cols:
        custom_labels[col] = f'SALS_{col_number}'
    else:
        custom_labels[col] = f'O_{col_number}'

column_colors = {
    col: 'DimGray' if col in control_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_noMut.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [noMut_cols, control_cols]

centroid_colors = {
    noMut_cols[0]: 'green',
    control_cols[0]: 'DimGray'
}

outline_colors = ['green', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_noMut)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# Set up 3D plot
sns.set(style='white', font_scale=1.1)
fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111, projection='3d')

# Plot each sample
for i, col in enumerate(Data_noMut.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    ax.scatter(pca_result[i, 0], pca_result[i, 1], pca_result[i, 2], color=color,
               marker=marker, s=100, edgecolor='k', linewidth=0.5)
    ax.text(pca_result[i, 0]-0.1, pca_result[i, 1], pca_result[i, 2]+0.1, custom_labels[col],
            fontsize=7, weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_noMut.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    ax.scatter(pca_result[i, 0], pca_result[i, 1], pca_result[i, 2], color=color,
               marker=marker, s=100, edgecolor='k', linewidth=0.5,
               label=selected_legend_columns[col])

# Centroids (no ellipses in 3D by default)
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_noMut.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices][:, [0, 1, 2]]  # Use all 3 PCs
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    ax.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
               edgecolor='k', linewidth=1)

    # (Optional) For ellipses in 3D, would need a mesh or wireframe; not included here.

# Final touches
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
ax.set_zlabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
ax.set_title('3D PCA of Case NoMut Samples with Centroids')
ax.legend(loc='best', title='Groups')
plt.tight_layout()
plt.savefig('NoMut_3D_PCA.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 unused import

Data_noMut = NoMut_filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_noMut.columns if "Control" in col]
noMut_cols = [col for col in Data_noMut.columns if "NoMut Case" in col]

# Custom labels for annotation
custom_labels = {}
for col in Data_noMut.columns:
    # Extract the last number after the hyphen
    match = re.search(r'-(\d+)$', col)
    col_number = match.group(1) if match else ''
    if col in control_cols:
        custom_labels[col] = f'HC{col_number}'
    elif col in noMut_cols:
        custom_labels[col] = f'SALS_{col_number}'
    else:
        custom_labels[col] = f'O_{col_number}'

column_colors = {
    col: 'DimGray' if col in control_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_noMut.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [noMut_cols, control_cols]

centroid_colors = {
    noMut_cols[0]: 'green',
    control_cols[0]: 'DimGray'
}

outline_colors = ['green', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_noMut)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# Set up 3D plot
sns.set(style='white', font_scale=1.1)
fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111, projection='3d')

# Plot each sample
for i, col in enumerate(Data_noMut.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    ax.scatter(pca_result[i, 0], pca_result[i, 1], pca_result[i, 2], color=color,
               marker=marker, s=0, edgecolor='k', linewidth=0.5)
    #ax.text(pca_result[i, 0]-0.1, pca_result[i, 1], pca_result[i, 2]+0.1, custom_labels[col],
        #    fontsize=7, weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_noMut.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    ax.scatter(pca_result[i, 0], pca_result[i, 1], pca_result[i, 2], color=color,
               marker=marker, s=0, edgecolor='k', linewidth=0.5,
               label=selected_legend_columns[col])

# Centroids (no ellipses in 3D by default)
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_noMut.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices][:, [0, 1, 2]]  # Use all 3 PCs
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    ax.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
               edgecolor='k', linewidth=1)

    # (Optional) For ellipses in 3D, would need a mesh or wireframe; not included here.

# Final touches
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
ax.set_zlabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
ax.set_title('3D PCA of Case NoMut Samples with Centroids')
ax.legend(loc='best', title='Groups')
plt.tight_layout()
plt.savefig('NoMut_3D_PCA_Centroids.png')
plt.show()


In [ ]:
# Extract PC loadings (components show how much each gene contributes to each PC)
noMut_pc_loadings = pca.components_

# Create DataFrame: genes as index, PCs as columns
noMut_pc_loadings_df = pd.DataFrame(noMut_pc_loadings.T, columns=['PC1', 'PC2', 'PC3'], index=Data_noMut.index)

# Save to CSV
noMut_pc_loadings_df.to_csv('noMut_pc_loadings.csv')

# Display the loadings
noMut_pc_loadings_df

In [ ]:
# Extract only PC1 loadings from the full loadings DataFrame
noMut_pc1_sorted_df = pd.DataFrame(noMut_pc_loadings_df['PC1'])

# Sort genes by their PC1 loading in descending order
noMut_pc1_loading_df_sorted = noMut_pc1_sorted_df.sort_values(by='PC1', ascending=False)

# Save the sorted PC1 loadings to CSV
noMut_pc1_loading_df_sorted.to_csv('noMut_pc1_loading_df_sorted.csv')

# Display the sorted DataFrame
noMut_pc1_loading_df_sorted

In [ ]:
import matplotlib.colors as mcolors

# Define custom colormap
custom_cmap = mcolors.TwoSlopeNorm(vmin=noMut_pc1_loading_df_sorted['PC1'].min(),
                                   vcenter=0,
                                   vmax=noMut_pc1_loading_df_sorted['PC1'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC1 values
colors = plt.cm.RdBu_r(custom_cmap(noMut_pc1_loading_df_sorted['PC1']))

# Create the bar plot
bars = ax.bar(noMut_pc1_loading_df_sorted.index, noMut_pc1_loading_df_sorted['PC1'], color=colors)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC1 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC1 Loadings')
ax.set_title('PC1 Loadings for Proteins')

# Rotate gene names for better readability
ax.set_xticklabels(noMut_pc1_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('Sporoadic All PC1 loadings.png')
plt.show()

In [ ]:
# Sort the DataFrame by 'PC1' values in descending order
noMut_pc1_loading_df_sorted = noMut_pc1_loading_df_sorted.sort_values(by='PC1', ascending=False)

# Extract the top 20 and bottom 20 genes
noMut_top_20_genes_PC1 = noMut_pc1_loading_df_sorted.head(20)
noMut_bottom_20_genes_PC1 = noMut_pc1_loading_df_sorted.tail(20)

# Combine both into one DataFrame for convenience
noMut_top_bottom_20_genes_PC1 = pd.concat([noMut_top_20_genes_PC1, noMut_bottom_20_genes_PC1])

# Save the results to a CSV file
noMut_top_bottom_20_genes_PC1.to_csv('noMut_top_bottom_20_genes_PC1.csv')
noMut_top_bottom_20_genes_PC1

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Define custom colormap for PC1 with vcenter set to zero
custom_cmap = mcolors.TwoSlopeNorm(vmin=noMut_top_bottom_20_genes_PC1['PC1'].min(),
                                   vcenter=0,
                                   vmax=noMut_top_bottom_20_genes_PC1['PC1'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(10, 10))

# Define colors for the bars based on the PC1 values
colors = plt.cm.RdBu_r(custom_cmap(noMut_top_bottom_20_genes_PC1['PC1']))

# Create the horizontal bar plot using the index (gene names) as labels
bars = ax.barh(noMut_top_bottom_20_genes_PC1.index, noMut_top_bottom_20_genes_PC1['PC1'], color=colors)

# Invert the y-axis to have the top gene at the top
ax.invert_yaxis()

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, orientation='vertical')
cbar.set_label('PC1 Loadings')

# Set labels and title
ax.set_xlabel('PC1 Loadings')
ax.set_ylabel('Proteins')
ax.set_title('PC1 Loadings')

# Show the plot
plt.tight_layout()
plt.savefig('NoMut Top Bottom 20 PC1 loadings.png')
plt.show()

In [ ]:
# Extract only PC2 loadings from the full loadings DataFrame
noMut_pc2_sorted_df = pd.DataFrame(noMut_pc_loadings_df['PC2'])

# Sort genes by their PC2 loading in descending order
noMut_pc2_loading_df_sorted = noMut_pc2_sorted_df.sort_values(by='PC2', ascending=False)

# Save the sorted PC2 loadings to CSV
noMut_pc2_loading_df_sorted.to_csv('noMut_pc2_loading_df_sorted.csv')

# Display the sorted DataFrame
noMut_pc2_loading_df_sorted

In [ ]:
import matplotlib.colors as mcolors

# Define custom colormap
custom_cmap = mcolors.TwoSlopeNorm(vmin=noMut_pc2_loading_df_sorted['PC2'].min(),
                                   vcenter=0,
                                   vmax=noMut_pc2_loading_df_sorted['PC2'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC2 values
colors = plt.cm.RdBu_r(custom_cmap(noMut_pc2_loading_df_sorted['PC2']))

# Create the bar plot
bars = ax.bar(noMut_pc2_loading_df_sorted.index, noMut_pc2_loading_df_sorted['PC2'], color=colors)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC2 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC2 Loadings')
ax.set_title('PC2 Loadings for Proteins')

# Rotate gene names for better readability
ax.set_xticklabels(noMut_pc2_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('NoMut All PC2 loadings.png')
plt.show()

In [ ]:
# Sort the DataFrame by 'PC2' values in descending order
noMut_pc2_loading_df_sorted = noMut_pc2_loading_df_sorted.sort_values(by='PC2', ascending=False)

# Extract the top 20 and bottom 20 genes
noMut_top_20_genes_PC2 = noMut_pc2_loading_df_sorted.head(20)
noMut_bottom_20_genes_PC2 = noMut_pc2_loading_df_sorted.tail(20)

# Combine both into one DataFrame for convenience
noMut_top_bottom_20_genes_PC2 = pd.concat([noMut_top_20_genes_PC2, noMut_bottom_20_genes_PC2])

# Save the results to a CSV file
noMut_top_bottom_20_genes_PC2.to_csv('noMut_top_bottom_20_genes_PC2.csv')
noMut_top_bottom_20_genes_PC2

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Define custom colormap for PC2 with vcenter set to zero
custom_cmap = mcolors.TwoSlopeNorm(vmin=noMut_top_bottom_20_genes_PC2['PC2'].min(),
                                   vcenter=0,
                                   vmax=noMut_top_bottom_20_genes_PC2['PC2'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(10, 10))

# Define colors for the bars based on the PC2 values
colors = plt.cm.RdBu_r(custom_cmap(noMut_top_bottom_20_genes_PC2['PC2']))

# Create the horizontal bar plot using the index (gene names) as labels
bars = ax.barh(noMut_top_bottom_20_genes_PC2.index, noMut_top_bottom_20_genes_PC2['PC2'], color=colors)

# Invert the y-axis to have the top gene at the top
ax.invert_yaxis()

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, orientation='vertical')
cbar.set_label('PC2 Loadings')

# Set labels and title
ax.set_xlabel('PC2 Loadings')
ax.set_ylabel('Proteins')
ax.set_title('PC2 Loadings')

# Show the plot
plt.tight_layout()
plt.savefig('NoMut Top Bottom 20 PC2 loadings.png')
plt.show()

In [ ]:
# Extract only PC3 loadings from the full loadings DataFrame
noMut_pc3_sorted_df = pd.DataFrame(noMut_pc_loadings_df['PC3'])

# Sort genes by their PC3 loading in descending order
noMut_pc3_loading_df_sorted = noMut_pc3_sorted_df.sort_values(by='PC3', ascending=False)

# Save the sorted PC3 loadings to CSV
noMut_pc3_loading_df_sorted.to_csv('noMut_pc3_loading_df_sorted.csv')

# Display the sorted DataFrame
noMut_pc3_loading_df_sorted

In [ ]:
import matplotlib.colors as mcolors

# Define custom colormap
custom_cmap = mcolors.TwoSlopeNorm(vmin=noMut_pc3_loading_df_sorted['PC3'].min(),
                                   vcenter=0,
                                   vmax=noMut_pc3_loading_df_sorted['PC3'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC3 values
colors = plt.cm.RdBu_r(custom_cmap(noMut_pc3_loading_df_sorted['PC3']))

# Create the bar plot
bars = ax.bar(noMut_pc3_loading_df_sorted.index, noMut_pc3_loading_df_sorted['PC3'], color=colors)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC3 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC3 Loadings')
ax.set_title('PC3 Loadings for Proteins')

# Rotate gene names for better readability
ax.set_xticklabels(noMut_pc3_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('NoMut All PC3 loadings.png')
plt.show()

In [ ]:
# Sort the DataFrame by 'PC3' values in descending order
noMut_pc3_loading_df_sorted = noMut_pc3_loading_df_sorted.sort_values(by='PC3', ascending=False)

# Extract the top 20 and bottom 20 genes
noMut_top_20_genes_PC3 = noMut_pc3_loading_df_sorted.head(20)
noMut_bottom_20_genes_PC3 = noMut_pc3_loading_df_sorted.tail(20)

# Combine both into one DataFrame for convenience
noMut_top_bottom_20_genes_PC3 = pd.concat([noMut_top_20_genes_PC3, noMut_bottom_20_genes_PC3])

# Save the results to a CSV file
noMut_top_bottom_20_genes_PC3.to_csv('noMut_top_bottom_20_genes_PC3.csv')
noMut_top_bottom_20_genes_PC3

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Define custom colormap for PC3 with vcenter set to zero
custom_cmap = mcolors.TwoSlopeNorm(vmin=noMut_top_bottom_20_genes_PC3['PC3'].min(),
                                   vcenter=0,
                                   vmax=noMut_top_bottom_20_genes_PC3['PC3'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(10, 10))

# Define colors for the bars based on the PC3 values
colors = plt.cm.RdBu_r(custom_cmap(noMut_top_bottom_20_genes_PC3['PC3']))

# Create the horizontal bar plot using the index (gene names) as labels
bars = ax.barh(noMut_top_bottom_20_genes_PC3.index, noMut_top_bottom_20_genes_PC3['PC3'], color=colors)

# Invert the y-axis to have the top gene at the top
ax.invert_yaxis()

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, orientation='vertical')
cbar.set_label('PC3 Loadings')

# Set labels and title
ax.set_xlabel('PC3 Loadings')
ax.set_ylabel('Proteins')
ax.set_title('PC3 Loadings')

# Show the plot
plt.tight_layout()
plt.savefig('NoMut Top Bottom 20 PC3 loadings.png')
plt.show()

In [ ]:
# Remove columns with 'DrugA-12months' or 'DrugB-4month' in their names
MutBpos_filtered_df = Data.loc[:, ~Data.columns.str.contains('DrugA|DrugB|MutC|NoMut')]
MutBpos_filtered_df

In [ ]:
list(MutBpos_filtered_df.columns)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Complete color assignment for all columns in filtered_df
samples = {
    'Control-1': 'DimGray',
    'Control-30': 'DimGray',
    'Control-11': 'DimGray',
    'MutBpos-2': '#800080',
    'Control-27': 'DimGray',
    'Control-10': 'DimGray',
    'Control-9': 'DimGray',
    'MutBpos-21': '#800080',
    'Control-22': 'DimGray',
    'Control-25': 'DimGray',
    'MutBpos-15': '#800080',
    'Control-13': 'DimGray',
    'MutBpos-16': '#800080',
    'Control-5': 'DimGray',
    'Control-19': 'DimGray',
    'MutBpos-4': '#800080',
    'Control-7': 'DimGray',
    'Control-26': 'DimGray',
    'MutBpos-13': '#800080',
    'Control-17': 'DimGray',
    'Control-16': 'DimGray',
    'Control-24': 'DimGray',
    'Control-20': 'DimGray',
    'MutBpos-12': '#800080',
    'Control-31': 'DimGray',
    'Control-4': 'DimGray',
    'Control-28': 'DimGray',
    'Control-34': 'DimGray',
    'MutBpos-17': '#800080',
    'MutBpos-10': '#800080',
    'MutBpos-18': '#800080',
    'Control-3': 'DimGray',
    'MutBpos-14': '#800080',
    'Control-6': 'DimGray',
    'MutBpos-11': '#800080',
    'Control-23': 'DimGray',
    'Control-33': 'DimGray',
    'MutBpos-3': '#800080',
    'Control-2': 'DimGray',
    'Control-32': 'DimGray',
    'MutBpos-1': '#800080',
    'MutBpos-6': '#800080',
    'Control-15': 'DimGray',
    'Control-29': 'DimGray',
    'Control-12': 'DimGray',
    'Control-35': 'DimGray',
    'MutBpos-5': '#800080',
    'MutBpos-8': '#800080',
    'Control-14': 'DimGray',
    'Control-21': 'DimGray',
    'Control-18': 'DimGray',
    'MutBpos-20': '#800080',
    'Control-37': 'DimGray',
    'Control-36': 'DimGray',
    'Control-38': 'DimGray',
    'MutBpos-19': '#800080',
    'MutBpos-9': '#800080',
    'MutBpos-7': '#800080',
    'Control-8': 'DimGray',
    'MutB-Baseline-1': '#800080',
    'MutB-Baseline-2': '#800080',
    'MutB-Baseline-3': '#800080',
    'MutB-Baseline-4': '#800080',
    'MutB-Baseline-5': '#800080',
    'MutB-Baseline-6': '#800080',
    'MutB-Baseline-7': '#800080',
}


# Assign colors to all columns of your dataframe using the updated mapping
column_colors = [samples[col] for col in MutBpos_filtered_df.columns]

plt.figure(figsize=(50, 50))

plot4 = sns.clustermap(
    MutBpos_filtered_df, z_score=0, cmap="vlag", figsize=(20, 40),
    vmin=-1, vmax=1, cbar_kws={"label": "z score"},
    col_colors=[column_colors],
    cbar_pos=(1, 0.5, 0.05, 0.15), xticklabels=True
)

# Set x tick labels color to white
plot4.ax_heatmap.tick_params(axis='x', colors='white')

# Increase font size of color bar tick labels and label
plot4.cax.set_yticklabels(plot4.cax.get_yticklabels(), fontsize=15)
plot4.cax.set_ylabel('z score', fontsize=15)

# Example: Define legend labels for specific columns of samples
legend_labels = {
    "Control-1": "Control",
    "MutBpos-2": "MutBpos"
}

# Create legend handles from defined legend_labels
handles1 = [plt.Rectangle((0, 0), 1.5, 1.5, color=samples[col]) for col in legend_labels.keys()]
legend1 = plot4.ax_col_dendrogram.legend(
    handles=handles1,
    labels=legend_labels.values(),
    loc='upper left',
    bbox_to_anchor=(1.05, -0.25),
    fontsize='xx-large',
    frameon=False
)
plot4.ax_col_dendrogram.add_artist(legend1)

plt.savefig('MutBpos_Control_all_prots_clustermap.png', bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(MutBpos_filtered_df)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Create an instance of PCA with more components
pca_all = PCA(n_components=len(MutBpos_filtered_df.columns))

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_all.fit(data_transposed)

# Calculate the explained variance ratio (as a percentage) for each principal component
explained_variance_ratio_percentage = pca_all.explained_variance_ratio_ * 100

# Create a figure with one subplot
fig, ax = plt.subplots(figsize=(50, 20))

# Plot the bar plot for explained variance of each principal component
ax.bar(np.arange(1, len(MutBpos_filtered_df.columns) + 1), explained_variance_ratio_percentage, color='skyblue', alpha=0.8)

# Customize the plot
ax.set_xlabel('Principal Component')
ax.set_ylabel('Percentage of Variance Explained')
ax.set_title('Percentage of Variance Explained by Principal Components')
ax.set_xticks(np.arange(1, len(MutBpos_filtered_df.columns) + 1))
#ax.grid(True)

plt.tight_layout()
plt.savefig('MutBpos_Eigen_values_Bar_graph.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2
import re

Data_mutBpos = MutBpos_filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_mutBpos.columns if "Control" in col]
mutB_cols = [col for col in Data_mutBpos.columns if "MutBpos" in col or "MutB-Baseline" in col]

# Custom labels for annotation
custom_labels = {}
for col in Data_mutBpos.columns:
    # Extract the sample number after the last hyphen
    match = re.search(r'-(\d+)$', col)
    col_number = match.group(1) if match else ''
    if col in control_cols:
        custom_labels[col] = f'HC{col_number}'
    elif col in mutB_cols:
        custom_labels[col] = f'MutB_{col_number}'
    else:
        custom_labels[col] = col


column_colors = {
    col: 'DimGray' if col in control_cols 
         else ('#800080' if col in mutB_cols else 'black')
    for col in Data_mutBpos.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [mutB_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_mutBpos)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_mutBpos.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5)
    plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
             weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_mutBpos.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_mutBpos.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2, linestyle='dashed')
    plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Centroids and Confidence Ellipses')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('MutBpos PC1 VS PC2.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

Data_mutBpos = MutBpos_filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_mutBpos.columns if "Control" in col]
mutB_cols = [col for col in Data_mutBpos.columns if "MutBpos" in col or "MutB-Baseline" in col]

# Custom labels for annotation
custom_labels = {}
for col in Data_mutBpos.columns:
    # Extract the sample number after the last hyphen
    match = re.search(r'-(\d+)$', col)
    col_number = match.group(1) if match else ''
    if col in control_cols:
        custom_labels[col] = f'HC{col_number}'
    elif col in mutB_cols:
        custom_labels[col] = f'MutB_{col_number}'
    else:
        custom_labels[col] = col


column_colors = {
    col: 'DimGray' if col in control_cols 
         else ('#800080' if col in mutB_cols else 'black')
    for col in Data_mutBpos.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [mutB_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_mutBpos)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_mutBpos.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5)
    #plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
    #         weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_mutBpos.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_mutBpos.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    #ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
    #                  facecolor='none', linewidth=2, linestyle='dashed')
    #plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Centroids and Confidence Ellipses')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('MutBpos PC1 VS PC2 Centroids.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

Data_mutBpos = MutBpos_filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_mutBpos.columns if "Control" in col]
mutB_cols = [col for col in Data_mutBpos.columns if "MutBpos" in col or "MutB-Baseline" in col]

# Custom labels for annotation
custom_labels = {}
for col in Data_mutBpos.columns:
    # Extract the sample number after the last hyphen
    match = re.search(r'-(\d+)$', col)
    col_number = match.group(1) if match else ''
    if col in control_cols:
        custom_labels[col] = f'HC{col_number}'
    elif col in mutB_cols:
        custom_labels[col] = f'MutB_{col_number}'
    else:
        custom_labels[col] = col

column_colors = {
    col: 'DimGray' if col in control_cols 
         else ('#800080' if col in mutB_cols else 'black')
    for col in Data_mutBpos.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [mutB_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_mutBpos)
transposed_data = standardized_data.T

# PCA (need at least 3 components now)
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_mutBpos.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5)
    plt.text(pca_result[i, 0]-0.1, pca_result[i, 2]+0.1, custom_labels[col], fontsize=7,
             weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_mutBpos.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_mutBpos.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices][:, [0, 2]]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2, linestyle='dashed')
    plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Centroids (PC1 vs PC3)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('MutBpos PC1 VS PC3.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

Data_mutBpos = MutBpos_filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_mutBpos.columns if "Control" in col]
mutB_cols = [col for col in Data_mutBpos.columns if "MutBpos" in col or "MutB-Baseline" in col]

# Custom labels for annotation
# Custom labels for annotation
custom_labels = {}
for col in Data_mutBpos.columns:
    # Extract the sample number after the last hyphen
    match = re.search(r'-(\d+)$', col)
    col_number = match.group(1) if match else ''
    if col in control_cols:
        custom_labels[col] = f'HC{col_number}'
    elif col in mutB_cols:
        custom_labels[col] = f'MutB_{col_number}'
    else:
        custom_labels[col] = col

column_colors = {
    col: 'DimGray' if col in control_cols 
         else ('#800080' if col in mutB_cols else 'black')
    for col in Data_mutBpos.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [mutB_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_mutBpos)
transposed_data = standardized_data.T

# PCA (need at least 3 components now)
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_mutBpos.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5)
    #plt.text(pca_result[i, 0]-0.1, pca_result[i, 2]+0.1, custom_labels[col], fontsize=7,
    #         weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_mutBpos.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_mutBpos.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices][:, [0, 2]]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    #ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
    #                  facecolor='none', linewidth=2, linestyle='dashed')
    #plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Centroids (PC1 vs PC3)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('MutBpos PC1 VS PC3 Centroids.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

Data_mutBpos = MutBpos_filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_mutBpos.columns if "Control" in col]
mutB_cols = [col for col in Data_mutBpos.columns if "MutBpos" in col or "MutB-Baseline" in col]

# Custom labels for annotation
# Custom labels for annotation
custom_labels = {}
for col in Data_mutBpos.columns:
    # Extract the sample number after the last hyphen
    match = re.search(r'-(\d+)$', col)
    col_number = match.group(1) if match else ''
    if col in control_cols:
        custom_labels[col] = f'HC{col_number}'
    elif col in mutB_cols:
        custom_labels[col] = f'MutB_{col_number}'
    else:
        custom_labels[col] = col

column_colors = {
    col: 'DimGray' if col in control_cols 
         else ('#800080' if col in mutB_cols else 'black')
    for col in Data_mutBpos.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [mutB_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_mutBpos)
transposed_data = standardized_data.T

# PCA (need at least 3 components now)
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_mutBpos.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 1], pca_result[i, 2], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5)
    plt.text(pca_result[i, 1]-0.1, pca_result[i, 2]+0.1, custom_labels[col], fontsize=7,
             weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_mutBpos.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 1], pca_result[i, 2], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_mutBpos.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices][:, [1, 2]]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2, linestyle='dashed')
    plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Centroids (PC2 vs PC3)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('MutBpos PC2 VS PC3.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

Data_mutBpos = MutBpos_filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_mutBpos.columns if "Control" in col]
mutB_cols = [col for col in Data_mutBpos.columns if "MutBpos" in col or "MutB-Baseline" in col]

# Custom labels for annotation
custom_labels = {}
for col in Data_mutBpos.columns:
    # Extract the sample number after the last hyphen
    match = re.search(r'-(\d+)$', col)
    col_number = match.group(1) if match else ''
    if col in control_cols:
        custom_labels[col] = f'HC{col_number}'
    elif col in mutB_cols:
        custom_labels[col] = f'MutB_{col_number}'
    else:
        custom_labels[col] = col

column_colors = {
    col: 'DimGray' if col in control_cols 
         else ('#800080' if col in mutB_cols else 'black')
    for col in Data_mutBpos.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [mutB_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_mutBpos)
transposed_data = standardized_data.T

# PCA (need at least 3 components now)
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_mutBpos.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 1], pca_result[i, 2], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5)
   # plt.text(pca_result[i, 1]-0.1, pca_result[i, 2]+0.1, custom_labels[col], fontsize=7,
      #       weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_mutBpos.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 1], pca_result[i, 2], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_mutBpos.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices][:, [1, 2]]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    #ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
    #                  facecolor='none', linewidth=2, linestyle='dashed')
    #plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Centroids (PC2 vs PC3)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('MutBpos PC2 VS PC3 Centroids.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from mpl_toolkits.mplot3d import Axes3D

Data_mutBpos = MutBpos_filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_mutBpos.columns if "Control" in col]
mutB_cols = [col for col in Data_mutBpos.columns if "MutBpos" in col or "MutB-Baseline" in col]

# Custom labels for annotation
custom_labels = {}
for col in Data_mutBpos.columns:
    # Extract the sample number after the last hyphen
    match = re.search(r'-(\d+)$', col)
    col_number = match.group(1) if match else ''
    if col in control_cols:
        custom_labels[col] = f'HC{col_number}'
    elif col in mutB_cols:
        custom_labels[col] = f'MutB_{col_number}'
    else:
        custom_labels[col] = col

column_colors = {
    col: 'DimGray' if col in control_cols 
         else ('#800080' if col in mutB_cols else 'black')
    for col in Data_mutBpos.columns
}

centroid_colors = {
    mutB_cols[0]: '#800080',
    control_cols[0]: 'DimGray'
}

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_mutBpos)
transposed_data = standardized_data.T

# PCA (3 components for 3D plotting)
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# 3D plot
fig = plt.figure(figsize=(11, 8))
ax = fig.add_subplot(111, projection='3d')

for i, col in enumerate(Data_mutBpos.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    else:
        marker = 'x'
    ax.scatter(pca_result[i, 0], pca_result[i, 1], pca_result[i, 2], color=color, marker=marker, s=100, edgecolor='k', linewidth=0.5)
    ax.text(pca_result[i, 0], pca_result[i, 1], pca_result[i, 2], custom_labels[col], fontsize=7, weight='bold')

# Centroids
selected_columns_list = [mutB_cols, control_cols]
for group_cols in selected_columns_list:
    if not group_cols:
        continue
    indices = [Data_mutBpos.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices, :3]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    color = centroid_colors[group_cols[0]]
    ax.scatter(*centroid, marker='X', s=250, color=color, edgecolor='k', linewidth=1)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
ax.set_zlabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
ax.set_title('3D PCA of Case CSF Samples with Centroids')
plt.tight_layout()
plt.savefig('MutBpos 3D PCA.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from mpl_toolkits.mplot3d import Axes3D

Data_mutBpos = MutBpos_filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_mutBpos.columns if "Control" in col]
mutB_cols = [col for col in Data_mutBpos.columns if "MutBpos" in col or "MutB-Baseline" in col]

# Custom labels for annotation
custom_labels = {}
for col in Data_mutBpos.columns:
    # Extract the sample number after the last hyphen
    match = re.search(r'-(\d+)$', col)
    col_number = match.group(1) if match else ''
    if col in control_cols:
        custom_labels[col] = f'HC{col_number}'
    elif col in mutB_cols:
        custom_labels[col] = f'MutB_{col_number}'
    else:
        custom_labels[col] = col

column_colors = {
    col: 'DimGray' if col in control_cols 
         else ('#800080' if col in mutB_cols else 'black')
    for col in Data_mutBpos.columns
}

centroid_colors = {
    mutB_cols[0]: '#800080',
    control_cols[0]: 'DimGray'
}

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_mutBpos)
transposed_data = standardized_data.T

# PCA (3 components for 3D plotting)
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# 3D plot
fig = plt.figure(figsize=(11, 8))
ax = fig.add_subplot(111, projection='3d')

for i, col in enumerate(Data_mutBpos.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    else:
        marker = 'x'
    ax.scatter(pca_result[i, 0], pca_result[i, 1], pca_result[i, 2], color=color, marker=marker, s=0, edgecolor='k', linewidth=0.5)
    #ax.text(pca_result[i, 0], pca_result[i, 1], pca_result[i, 2], custom_labels[col], fontsize=7, weight='bold')

# Centroids
selected_columns_list = [mutB_cols, control_cols]
for group_cols in selected_columns_list:
    if not group_cols:
        continue
    indices = [Data_mutBpos.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices, :3]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    color = centroid_colors[group_cols[0]]
    ax.scatter(*centroid, marker='X', s=250, color=color, edgecolor='k', linewidth=1)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
ax.set_zlabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
ax.set_title('3D PCA of Case CSF Samples with Centroids')
plt.tight_layout()
plt.savefig('MutBpos 3D PCA Centroids.png')
plt.show()


In [ ]:
# Extract PC loadings (components show how much each gene contributes to each PC)
mutBpos_pc_loadings = pca.components_

# Create DataFrame: genes as index, PCs as columns
mutBpos_pc_loadings_df = pd.DataFrame(mutBpos_pc_loadings.T, columns=['PC1', 'PC2', 'PC3'], index=Data_mutBpos.index)

# Save to CSV
mutBpos_pc_loadings_df.to_csv('mutBpos_pc_loadings.csv')

# Display the loadings
mutBpos_pc_loadings_df

In [ ]:
# Extract only PC1 loadings from the full MutBpos loadings DataFrame
mutBpos_pc1_sorted_df = pd.DataFrame(mutBpos_pc_loadings_df['PC1'])

# Sort genes by their PC1 loading in descending order
mutBpos_pc1_loading_df_sorted = mutBpos_pc1_sorted_df.sort_values(by='PC1', ascending=False)

# Save the sorted PC1 loadings to CSV
mutBpos_pc1_loading_df_sorted.to_csv('mutBpos_pc1_loading_df_sorted.csv')

# Display the sorted DataFrame
mutBpos_pc1_loading_df_sorted

In [ ]:
import matplotlib.colors as mcolors

# Define custom colormap
custom_cmap = mcolors.TwoSlopeNorm(vmin=mutBpos_pc1_loading_df_sorted['PC1'].min(),
                                   vcenter=0,
                                   vmax=mutBpos_pc1_loading_df_sorted['PC1'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC1 values
colors = plt.cm.RdBu_r(custom_cmap(mutBpos_pc1_loading_df_sorted['PC1']))

# Create the bar plot
bars = ax.bar(mutBpos_pc1_loading_df_sorted.index, mutBpos_pc1_loading_df_sorted['PC1'], color=colors)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC1 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC1 Loadings')
ax.set_title('PC1 Loadings for Proteins (MutBpos)')

# Rotate gene names for better readability
ax.set_xticklabels(mutBpos_pc1_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('MutBpos_All_PC1_loadings.png')
plt.show()

In [ ]:
# Sort the DataFrame by 'PC1' values in descending order
mutBpos_pc1_loading_df_sorted = mutBpos_pc1_loading_df_sorted.sort_values(by='PC1', ascending=False)

# Extract the top 20 and bottom 20 genes
mutBpos_top_20_genes_PC1 = mutBpos_pc1_loading_df_sorted.head(20)
mutBpos_bottom_20_genes_PC1 = mutBpos_pc1_loading_df_sorted.tail(20)

# Combine both into one DataFrame for convenience
mutBpos_top_bottom_20_genes_PC1 = pd.concat([mutBpos_top_20_genes_PC1, mutBpos_bottom_20_genes_PC1])

# Save the results to a CSV file
mutBpos_top_bottom_20_genes_PC1.to_csv('mutBpos_top_bottom_20_genes_PC1.csv')

# Display the combined DataFrame
mutBpos_top_bottom_20_genes_PC1

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Define custom colormap for PC1 with vcenter set to zero
custom_cmap = mcolors.TwoSlopeNorm(vmin=mutBpos_top_bottom_20_genes_PC1['PC1'].min(),
                                   vcenter=0,
                                   vmax=mutBpos_top_bottom_20_genes_PC1['PC1'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(10, 10))

# Define colors for the bars based on the PC1 values
colors = plt.cm.RdBu_r(custom_cmap(mutBpos_top_bottom_20_genes_PC1['PC1']))

# Create the horizontal bar plot using the index (gene names) as labels
bars = ax.barh(mutBpos_top_bottom_20_genes_PC1.index, mutBpos_top_bottom_20_genes_PC1['PC1'], color=colors)

# Invert the y-axis to have the top gene at the top
ax.invert_yaxis()

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, orientation='vertical')
cbar.set_label('PC1 Loadings')

# Set labels and title
ax.set_xlabel('PC1 Loadings')
ax.set_ylabel('Proteins')
ax.set_title('PC1 Loadings: Top and Bottom 20 (MutBpos)')

# Show the plot
plt.tight_layout()
plt.savefig('MutBpos_Top_Bottom_20_PC1_loadings.png')
plt.show()

In [ ]:
# Extract only PC2 loadings from the full loadings DataFrame
mutBpos_pc2_sorted_df = pd.DataFrame(mutBpos_pc_loadings_df['PC2'])

# Sort genes by their PC2 loading in descending order
mutBpos_pc2_loading_df_sorted = mutBpos_pc2_sorted_df.sort_values(by='PC2', ascending=False)

# Save the sorted PC2 loadings to CSV
mutBpos_pc2_loading_df_sorted.to_csv('mutBpos_pc2_loading_df_sorted.csv')

# Display the sorted DataFrame
mutBpos_pc2_loading_df_sorted

In [ ]:
import matplotlib.colors as mcolors

# Define custom colormap
custom_cmap = mcolors.TwoSlopeNorm(vmin=mutBpos_pc2_loading_df_sorted['PC2'].min(),
                                   vcenter=0,
                                   vmax=mutBpos_pc2_loading_df_sorted['PC2'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC2 values
colors = plt.cm.RdBu_r(custom_cmap(mutBpos_pc2_loading_df_sorted['PC2']))

# Create the bar plot
bars = ax.bar(mutBpos_pc2_loading_df_sorted.index, mutBpos_pc2_loading_df_sorted['PC2'], color=colors)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC2 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC2 Loadings')
ax.set_title('PC2 Loadings for Proteins (MutBpos)')

# Rotate gene names for better readability
ax.set_xticklabels(mutBpos_pc2_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('MutBpos_All_PC2_loadings.png')
plt.show()

In [ ]:
# Sort the DataFrame by 'PC2' values in descending order
mutBpos_pc2_loading_df_sorted = mutBpos_pc2_loading_df_sorted.sort_values(by='PC2', ascending=False)

# Extract the top 20 and bottom 20 genes
mutBpos_top_20_genes_PC2 = mutBpos_pc2_loading_df_sorted.head(20)
mutBpos_bottom_20_genes_PC2 = mutBpos_pc2_loading_df_sorted.tail(20)

# Combine both into one DataFrame for convenience
mutBpos_top_bottom_20_genes_PC2 = pd.concat([mutBpos_top_20_genes_PC2, mutBpos_bottom_20_genes_PC2])

# Save the results to a CSV file
mutBpos_top_bottom_20_genes_PC2.to_csv('mutBpos_top_bottom_20_genes_PC2.csv')

# Display the combined DataFrame
mutBpos_top_bottom_20_genes_PC2

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Define custom colormap for PC2 with vcenter set to zero
custom_cmap = mcolors.TwoSlopeNorm(vmin=mutBpos_top_bottom_20_genes_PC2['PC2'].min(),
                                   vcenter=0,
                                   vmax=mutBpos_top_bottom_20_genes_PC2['PC2'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(10, 10))

# Define colors for the bars based on the PC2 values
colors = plt.cm.RdBu_r(custom_cmap(mutBpos_top_bottom_20_genes_PC2['PC2']))

# Create the horizontal bar plot using the index (gene names) as labels
bars = ax.barh(mutBpos_top_bottom_20_genes_PC2.index, mutBpos_top_bottom_20_genes_PC2['PC2'], color=colors)

# Invert the y-axis to have the top gene at the top
ax.invert_yaxis()

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, orientation='vertical')
cbar.set_label('PC2 Loadings')

# Set labels and title
ax.set_xlabel('PC2 Loadings')
ax.set_ylabel('Proteins')
ax.set_title('PC2 Loadings: Top and Bottom 20 (MutBpos)')

# Show the plot
plt.tight_layout()
plt.savefig('MutBpos_Top_Bottom_20_PC2_loadings.png')
plt.show()

In [ ]:
# Extract only PC3 loadings from the full loadings DataFrame
mutBpos_pc3_sorted_df = pd.DataFrame(mutBpos_pc_loadings_df['PC3'])

# Sort genes by their PC3 loading in descending order
mutBpos_pc3_loading_df_sorted = mutBpos_pc3_sorted_df.sort_values(by='PC3', ascending=False)

# Save the sorted PC3 loadings to CSV
mutBpos_pc3_loading_df_sorted.to_csv('mutBpos_pc3_loading_df_sorted.csv')

# Display the sorted DataFrame
mutBpos_pc3_loading_df_sorted

In [ ]:
import matplotlib.colors as mcolors

# Define custom colormap
custom_cmap = mcolors.TwoSlopeNorm(vmin=mutBpos_pc3_loading_df_sorted['PC3'].min(),
                                   vcenter=0,
                                   vmax=mutBpos_pc3_loading_df_sorted['PC3'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC3 values
colors = plt.cm.RdBu_r(custom_cmap(mutBpos_pc3_loading_df_sorted['PC3']))

# Create the bar plot
bars = ax.bar(mutBpos_pc3_loading_df_sorted.index, mutBpos_pc3_loading_df_sorted['PC3'], color=colors)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC3 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC3 Loadings')
ax.set_title('PC3 Loadings for Proteins (MutBpos)')

# Rotate gene names for better readability
ax.set_xticklabels(mutBpos_pc3_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('MutBpos_All_PC3_loadings.png')
plt.show()

In [ ]:
# Sort the DataFrame by 'PC3' values in descending order
mutBpos_pc3_loading_df_sorted = mutBpos_pc3_loading_df_sorted.sort_values(by='PC3', ascending=False)

# Extract the top 20 and bottom 20 genes
mutBpos_top_20_genes_PC3 = mutBpos_pc3_loading_df_sorted.head(20)
mutBpos_bottom_20_genes_PC3 = mutBpos_pc3_loading_df_sorted.tail(20)

# Combine both into one DataFrame for convenience
mutBpos_top_bottom_20_genes_PC3 = pd.concat([mutBpos_top_20_genes_PC3, mutBpos_bottom_20_genes_PC3])

# Save the results to a CSV file
mutBpos_top_bottom_20_genes_PC3.to_csv('mutBpos_top_bottom_20_genes_PC3.csv')

# Display the combined DataFrame
mutBpos_top_bottom_20_genes_PC3

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Define custom colormap for PC3 with vcenter set to zero
custom_cmap = mcolors.TwoSlopeNorm(vmin=mutBpos_top_bottom_20_genes_PC3['PC3'].min(),
                                   vcenter=0,
                                   vmax=mutBpos_top_bottom_20_genes_PC3['PC3'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(10, 10))

# Define colors for the bars based on the PC3 values
colors = plt.cm.RdBu_r(custom_cmap(mutBpos_top_bottom_20_genes_PC3['PC3']))

# Create the horizontal bar plot using the index (gene names) as labels
bars = ax.barh(mutBpos_top_bottom_20_genes_PC3.index, mutBpos_top_bottom_20_genes_PC3['PC3'], color=colors)

# Invert the y-axis to have the top gene at the top
ax.invert_yaxis()

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, orientation='vertical')
cbar.set_label('PC3 Loadings')

# Set labels and title
ax.set_xlabel('PC3 Loadings')
ax.set_ylabel('Proteins')
ax.set_title('PC3 Loadings: Top and Bottom 20 (MutBpos)')

# Show the plot
plt.tight_layout()
plt.savefig('MutBpos_Top_Bottom_20_PC3_loadings.png')
plt.show()

In [ ]:
# Remove columns with 'DrugA-12months' or 'DrugB-4month' in their names
MutBpos_filtered_df_with_baseline = Data.loc[:, ~Data.columns.str.contains('DrugA|DrugB|MutC|NoMut')]
MutBpos_filtered_df_with_baseline

In [ ]:
list(MutBpos_filtered_df_with_baseline.columns)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Complete color assignment for all columns in filtered_df
samples = {
    'Control-1': 'DimGray',
    'Control-30': 'DimGray',
    'Control-11': 'DimGray',
    'MutBpos-2': '#800080',
    'Control-27': 'DimGray',
    'Control-10': 'DimGray',
    'Control-9': 'DimGray',
    'MutBpos-21': '#800080',
    'Control-22': 'DimGray',
    'Control-25': 'DimGray',
    'MutBpos-15': '#800080',
    'Control-13': 'DimGray',
    'MutBpos-16': '#800080',
    'Control-5': 'DimGray',
    'Control-19': 'DimGray',
    'MutBpos-4': '#800080',
    'Control-7': 'DimGray',
    'Control-26': 'DimGray',
    'MutBpos-13': '#800080',
    'Control-17': 'DimGray',
    'Control-16': 'DimGray',
    'Control-24': 'DimGray',
    'Control-20': 'DimGray',
    'MutBpos-12': '#800080',
    'Control-31': 'DimGray',
    'Control-4': 'DimGray',
    'Control-28': 'DimGray',
    'Control-34': 'DimGray',
    'MutBpos-17': '#800080',
    'MutBpos-10': '#800080',
    'MutBpos-18': '#800080',
    'Control-3': 'DimGray',
    'MutBpos-14': '#800080',
    'Control-6': 'DimGray',
    'MutBpos-11': '#800080',
    'Control-23': 'DimGray',
    'Control-33': 'DimGray',
    'MutBpos-3': '#800080',
    'Control-2': 'DimGray',
    'Control-32': 'DimGray',
    'MutBpos-1': '#800080',
    'MutBpos-6': '#800080',
    'Control-15': 'DimGray',
    'Control-29': 'DimGray',
    'Control-12': 'DimGray',
    'Control-35': 'DimGray',
    'MutBpos-5': '#800080',
    'MutBpos-8': '#800080',
    'Control-14': 'DimGray',
    'Control-21': 'DimGray',
    'Control-18': 'DimGray',
    'MutBpos-20': '#800080',
    'Control-37': 'DimGray',
    'Control-36': 'DimGray',
    'Control-38': 'DimGray',
    'MutBpos-19': '#800080',
    'MutBpos-9': '#800080',
    'MutBpos-7': '#800080',
    'Control-8': 'DimGray',
    'MutB-Baseline-1': '#800080',
    'MutB-Baseline-2': '#800080',
    'MutB-Baseline-3': '#800080',
    'MutB-Baseline-4': '#800080',
    'MutB-Baseline-5': '#800080',
    'MutB-Baseline-6': '#800080',
    'MutB-Baseline-7': '#800080'
}


# Assign colors to all columns of your dataframe using the updated mapping
column_colors = [samples[col] for col in MutBpos_filtered_df_with_baseline.columns]

plt.figure(figsize=(50, 50))

plot4 = sns.clustermap(
    MutBpos_filtered_df_with_baseline, z_score=0, cmap="vlag", figsize=(20, 40),
    vmin=-1, vmax=1, cbar_kws={"label": "z score"},
    col_colors=[column_colors],
    cbar_pos=(1, 0.5, 0.05, 0.15), xticklabels=True
)

# Set x tick labels color to white
plot4.ax_heatmap.tick_params(axis='x', colors='white')

# Increase font size of color bar tick labels and label
plot4.cax.set_yticklabels(plot4.cax.get_yticklabels(), fontsize=15)
plot4.cax.set_ylabel('z score', fontsize=15)

# Example: Define legend labels for specific columns of samples
legend_labels = {
    "Control-1": "Control",
    "MutBpos-2": "MutBpos"
}

# Create legend handles from defined legend_labels
handles1 = [plt.Rectangle((0, 0), 1.5, 1.5, color=samples[col]) for col in legend_labels.keys()]
legend1 = plot4.ax_col_dendrogram.legend(
    handles=handles1,
    labels=legend_labels.values(),
    loc='upper left',
    bbox_to_anchor=(1.05, -0.25),
    fontsize='xx-large',
    frameon=False
)
plot4.ax_col_dendrogram.add_artist(legend1)

plt.savefig('MutBpos_Control_with_baseline_all_prots_clustermap.png', bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(MutBpos_filtered_df_with_baseline)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Create an instance of PCA with more components
pca_all = PCA(n_components=len(MutBpos_filtered_df_with_baseline.columns))

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_all.fit(data_transposed)

# Calculate the explained variance ratio (as a percentage) for each principal component
explained_variance_ratio_percentage = pca_all.explained_variance_ratio_ * 100

# Create a figure with one subplot
fig, ax = plt.subplots(figsize=(50, 20))

# Plot the bar plot for explained variance of each principal component
ax.bar(np.arange(1, len(MutBpos_filtered_df_with_baseline.columns) + 1), explained_variance_ratio_percentage, color='skyblue', alpha=0.8)

# Customize the plot
ax.set_xlabel('Principal Component')
ax.set_ylabel('Percentage of Variance Explained')
ax.set_title('Percentage of Variance Explained by Principal Components')
ax.set_xticks(np.arange(1, len(MutBpos_filtered_df_with_baseline.columns) + 1))
#ax.grid(True)

plt.tight_layout()
plt.savefig('MutBpos_with_baseline_Eigen_values_Bar_graph.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2
import re

Data_mutBpos = MutBpos_filtered_df_with_baseline.copy()

# Define group membership based on naming
control_cols = [col for col in Data_mutBpos.columns if "Control" in col]
mutB_cols = [col for col in Data_mutBpos.columns if "MutBpos" in col or "MutB-Baseline" in col]

# Custom labels for annotation
custom_labels = {}
for col in Data_mutBpos.columns:
    if col.startswith("MutB-Baseline"):
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_B_{number}'
    elif col.startswith("MutBpos"):
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_{number}'
    elif col in control_cols:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'HC{number}'
    else:
        custom_labels[col] = col

column_colors = {
    col: 'DimGray' if col in control_cols 
         else ('#800080' if col in mutB_cols else 'black')
    for col in Data_mutBpos.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [mutB_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_mutBpos)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_mutBpos.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5)
    plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
             weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_mutBpos.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_mutBpos.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2, linestyle='dashed')
    plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Centroids and Confidence Ellipses')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('MutBpos with baseline PC1 VS PC2.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2
import re

Data_mutBpos = MutBpos_filtered_df_with_baseline.copy()

# Define group membership based on naming
control_cols = [col for col in Data_mutBpos.columns if "Control" in col]
mutB_cols = [col for col in Data_mutBpos.columns if "MutBpos" in col or "MutB-Baseline" in col]

# Custom labels for annotation
custom_labels = {}
for col in Data_mutBpos.columns:
    if col.startswith("MutB-Baseline"):
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_B_{number}'
    elif col.startswith("MutBpos"):
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_{number}'
    elif col in control_cols:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'HC{number}'
    else:
        custom_labels[col] = col

column_colors = {
    col: 'DimGray' if col in control_cols 
         else ('#800080' if col in mutB_cols else 'black')
    for col in Data_mutBpos.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [mutB_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_mutBpos)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_mutBpos.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5)
    #plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
        #     weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_mutBpos.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_mutBpos.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
   # ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
      #                facecolor='none', linewidth=2, linestyle='dashed')
    #plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Centroids and Confidence Ellipses')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('MutBpos_with_baseline_PC1_VS_PC2_centroids_only.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2
import re

Data_mutBpos = MutBpos_filtered_df_with_baseline.copy()

# Define group membership based on naming
control_cols = [col for col in Data_mutBpos.columns if "Control" in col]
mutB_cols = [col for col in Data_mutBpos.columns if "MutBpos" in col or "MutB-Baseline" in col]

# Custom labels for annotation
custom_labels = {}
for col in Data_mutBpos.columns:
    if col.startswith("MutB-Baseline"):
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_B_{number}'
    elif col.startswith("MutBpos"):
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_{number}'
    elif col in control_cols:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'HC{number}'
    else:
        custom_labels[col] = col

column_colors = {
    col: 'DimGray' if col in control_cols 
         else ('#800080' if col in mutB_cols else 'black')
    for col in Data_mutBpos.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [mutB_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_mutBpos)
transposed_data = standardized_data.T

# PCA - IMPORTANT: Use at least 3 components for PC1 vs PC3!
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_mutBpos.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5)
    plt.text(pca_result[i, 0]-0.1, pca_result[i, 2]+0.1, custom_labels[col], fontsize=7,
             weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_mutBpos.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_mutBpos.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices][:, [0, 2]]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2, linestyle='dashed')
    plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Centroids and Confidence Ellipses (PC1 vs PC3)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('MutBpos_with_baseline_PC1_vs_PC3.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2
import re

Data_mutBpos = MutBpos_filtered_df_with_baseline.copy()

# Define group membership based on naming
control_cols = [col for col in Data_mutBpos.columns if "Control" in col]
mutB_cols = [col for col in Data_mutBpos.columns if "MutBpos" in col or "MutB-Baseline" in col]

# Custom labels for annotation
custom_labels = {}
for col in Data_mutBpos.columns:
    if col.startswith("MutB-Baseline"):
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_B_{number}'
    elif col.startswith("MutBpos"):
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_{number}'
    elif col in control_cols:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'HC{number}'
    else:
        custom_labels[col] = col

column_colors = {
    col: 'DimGray' if col in control_cols 
         else ('#800080' if col in mutB_cols else 'black')
    for col in Data_mutBpos.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [mutB_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_mutBpos)
transposed_data = standardized_data.T

# PCA - IMPORTANT: Use at least 3 components for PC1 vs PC3!
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_mutBpos.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5)
    #plt.text(pca_result[i, 0]-0.1, pca_result[i, 2]+0.1, custom_labels[col], fontsize=7,
      #       weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_mutBpos.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_mutBpos.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices][:, [0, 2]]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
   # ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
      #                facecolor='none', linewidth=2, linestyle='dashed')
    #plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Centroids and Confidence Ellipses (PC1 vs PC3)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('MutBpos_with_baseline_PC1_vs_PC3_centroids_only.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2
import re

Data_mutBpos = MutBpos_filtered_df_with_baseline.copy()

# Define group membership based on naming
control_cols = [col for col in Data_mutBpos.columns if "Control" in col]
mutB_cols = [col for col in Data_mutBpos.columns if "MutBpos" in col or "MutB-Baseline" in col]

# Custom labels for annotation
custom_labels = {}
for col in Data_mutBpos.columns:
    if col.startswith("MutB-Baseline"):
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_B_{number}'
    elif col.startswith("MutBpos"):
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_{number}'
    elif col in control_cols:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'HC{number}'
    else:
        custom_labels[col] = col

column_colors = {
    col: 'DimGray' if col in control_cols 
         else ('#800080' if col in mutB_cols else 'black')
    for col in Data_mutBpos.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [mutB_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_mutBpos)
transposed_data = standardized_data.T

# PCA - IMPORTANT: Use at least 3 components for PC1 vs PC3!
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_mutBpos.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 1], pca_result[i, 2], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5)
    plt.text(pca_result[i, 1]-0.1, pca_result[i, 2]+0.1, custom_labels[col], fontsize=7,
             weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_mutBpos.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 1], pca_result[i, 2], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_mutBpos.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices][:, [1, 2]]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2, linestyle='dashed')
    plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Centroids and Confidence Ellipses (PC2 vs PC3)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('MutBpos_with_baseline_PC2_vs_PC3.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2
import re

Data_mutBpos = MutBpos_filtered_df_with_baseline.copy()

# Define group membership based on naming
control_cols = [col for col in Data_mutBpos.columns if "Control" in col]
mutB_cols = [col for col in Data_mutBpos.columns if "MutBpos" in col or "MutB-Baseline" in col]

# Custom labels for annotation
custom_labels = {}
for col in Data_mutBpos.columns:
    if col.startswith("MutB-Baseline"):
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_B_{number}'
    elif col.startswith("MutBpos"):
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_{number}'
    elif col in control_cols:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'HC{number}'
    else:
        custom_labels[col] = col

column_colors = {
    col: 'DimGray' if col in control_cols 
         else ('#800080' if col in mutB_cols else 'black')
    for col in Data_mutBpos.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [mutB_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_mutBpos)
transposed_data = standardized_data.T

# PCA - IMPORTANT: Use at least 3 components for PC1 vs PC3!
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_mutBpos.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 1], pca_result[i, 2], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5)
    #plt.text(pca_result[i, 1]-0.1, pca_result[i, 2]+0.1, custom_labels[col], fontsize=7,
        #     weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_mutBpos.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 1], pca_result[i, 2], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_mutBpos.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices][:, [1, 2]]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    #ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
    #                  facecolor='none', linewidth=2, linestyle='dashed')
    #plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Centroids and Confidence Ellipses (PC2 vs PC3)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('MutBpos_with_baseline_PC2_vs_PC3_centroids_only.png')
plt.show()

In [ ]:
# Extract PC loadings (components show how much each gene contributes to each PC)
mutBpos_with_baseline_pc_loadings = pca.components_

# Create DataFrame: genes as index, PCs as columns
mutBpos_with_baseline_pc_loadings_df = pd.DataFrame(mutBpos_with_baseline_pc_loadings.T, columns=['PC1', 'PC2', 'PC3'], index=Data_mutBpos.index)

# Save to CSV
mutBpos_with_baseline_pc_loadings_df.to_csv('mutBpos_with_baseline_pc_loadings.csv')

# Display the loadings
mutBpos_with_baseline_pc_loadings_df

In [ ]:
# Extract only PC1 loadings from the full MutBpos_with_baseline loadings DataFrame
mutBpos_with_baseline_pc1_sorted_df = pd.DataFrame(mutBpos_with_baseline_pc_loadings_df['PC1'])

# Sort genes by their PC1 loading in descending order
mutBpos_with_baseline_pc1_loading_df_sorted = mutBpos_with_baseline_pc1_sorted_df.sort_values(by='PC1', ascending=False)

# Save the sorted PC1 loadings to CSV
mutBpos_with_baseline_pc1_loading_df_sorted.to_csv('mutBpos_with_baseline_pc1_loading_df_sorted.csv')

# Display the sorted DataFrame
mutBpos_with_baseline_pc1_loading_df_sorted

In [ ]:
import matplotlib.colors as mcolors

# Define custom colormap
custom_cmap = mcolors.TwoSlopeNorm(
    vmin=mutBpos_with_baseline_pc1_loading_df_sorted['PC1'].min(),
    vcenter=0,
    vmax=mutBpos_with_baseline_pc1_loading_df_sorted['PC1'].max()
)

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC1 values
colors = plt.cm.RdBu_r(custom_cmap(mutBpos_with_baseline_pc1_loading_df_sorted['PC1']))

# Create the bar plot
bars = ax.bar(
    mutBpos_with_baseline_pc1_loading_df_sorted.index, 
    mutBpos_with_baseline_pc1_loading_df_sorted['PC1'], 
    color=colors
)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC1 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC1 Loadings')
ax.set_title('PC1 Loadings for Proteins (MutBpos_with_baseline)')

# Rotate gene names for better readability
ax.set_xticklabels(mutBpos_with_baseline_pc1_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('MutBpos_with_baseline_All_PC1_loadings.png')
plt.show()

In [ ]:
# Sort the DataFrame by 'PC1' values in descending order
mutBpos_with_baseline_pc1_loading_df_sorted = mutBpos_with_baseline_pc1_loading_df_sorted.sort_values(by='PC1', ascending=False)

# Extract the top 20 and bottom 20 genes
mutBpos_with_baseline_top_20_genes_PC1 = mutBpos_with_baseline_pc1_loading_df_sorted.head(20)
mutBpos_with_baseline_bottom_20_genes_PC1 = mutBpos_with_baseline_pc1_loading_df_sorted.tail(20)

# Combine both into one DataFrame for convenience
mutBpos_with_baseline_top_bottom_20_genes_PC1 = pd.concat([
    mutBpos_with_baseline_top_20_genes_PC1,
    mutBpos_with_baseline_bottom_20_genes_PC1
])

# Save the results to a CSV file
mutBpos_with_baseline_top_bottom_20_genes_PC1.to_csv('mutBpos_with_baseline_top_bottom_20_genes_PC1.csv')

# Display the combined DataFrame
mutBpos_with_baseline_top_bottom_20_genes_PC1

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Define custom colormap for PC1 with vcenter set to zero
custom_cmap = mcolors.TwoSlopeNorm(
    vmin=mutBpos_with_baseline_top_bottom_20_genes_PC1['PC1'].min(),
    vcenter=0,
    vmax=mutBpos_with_baseline_top_bottom_20_genes_PC1['PC1'].max()
)

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(10, 10))

# Define colors for the bars based on the PC1 values
colors = plt.cm.RdBu_r(custom_cmap(mutBpos_with_baseline_top_bottom_20_genes_PC1['PC1']))

# Create the horizontal bar plot using the index (gene names) as labels
bars = ax.barh(mutBpos_with_baseline_top_bottom_20_genes_PC1.index, 
               mutBpos_with_baseline_top_bottom_20_genes_PC1['PC1'], 
               color=colors)

# Invert the y-axis to have the top gene at the top
ax.invert_yaxis()

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, orientation='vertical')
cbar.set_label('PC1 Loadings')

# Set labels and title
ax.set_xlabel('PC1 Loadings')
ax.set_ylabel('Proteins')
ax.set_title('PC1 Loadings: Top and Bottom 20 (MutBpos_with_baseline)')

# Show the plot
plt.tight_layout()
plt.savefig('MutBpos_with_baseline_Top_Bottom_20_PC1_loadings.png')
plt.show()

In [ ]:
# Extract only PC2 loadings from the full MutBpos_with_baseline loadings DataFrame
mutBpos_with_baseline_pc2_sorted_df = pd.DataFrame(mutBpos_with_baseline_pc_loadings_df['PC2'])

# Sort genes by their PC2 loading in descending order
mutBpos_with_baseline_pc2_loading_df_sorted = mutBpos_with_baseline_pc2_sorted_df.sort_values(by='PC2', ascending=False)

# Save the sorted PC2 loadings to CSV
mutBpos_with_baseline_pc2_loading_df_sorted.to_csv('mutBpos_with_baseline_pc2_loading_df_sorted.csv')

# Display the sorted DataFrame
mutBpos_with_baseline_pc2_loading_df_sorted

In [ ]:
import matplotlib.colors as mcolors

# Define custom colormap
custom_cmap = mcolors.TwoSlopeNorm(
    vmin=mutBpos_with_baseline_pc2_loading_df_sorted['PC2'].min(),
    vcenter=0,
    vmax=mutBpos_with_baseline_pc2_loading_df_sorted['PC2'].max()
)

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC2 values
colors = plt.cm.RdBu_r(custom_cmap(mutBpos_with_baseline_pc2_loading_df_sorted['PC2']))

# Create the bar plot
bars = ax.bar(
    mutBpos_with_baseline_pc2_loading_df_sorted.index, 
    mutBpos_with_baseline_pc2_loading_df_sorted['PC2'], 
    color=colors
)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC2 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC2 Loadings')
ax.set_title('PC2 Loadings for Proteins (MutBpos_with_baseline)')

# Rotate gene names for better readability
ax.set_xticklabels(mutBpos_with_baseline_pc2_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('MutBpos_with_baseline_All_PC2_loadings.png')
plt.show()

In [ ]:
# Sort the DataFrame by 'PC2' values in descending order
mutBpos_with_baseline_pc2_loading_df_sorted = mutBpos_with_baseline_pc2_loading_df_sorted.sort_values(by='PC2', ascending=False)

# Extract the top 20 and bottom 20 genes
mutBpos_with_baseline_top_20_genes_PC2 = mutBpos_with_baseline_pc2_loading_df_sorted.head(20)
mutBpos_with_baseline_bottom_20_genes_PC2 = mutBpos_with_baseline_pc2_loading_df_sorted.tail(20)

# Combine both into one DataFrame for convenience
mutBpos_with_baseline_top_bottom_20_genes_PC2 = pd.concat([
    mutBpos_with_baseline_top_20_genes_PC2,
    mutBpos_with_baseline_bottom_20_genes_PC2
])

# Save the results to a CSV file
mutBpos_with_baseline_top_bottom_20_genes_PC2.to_csv('mutBpos_with_baseline_top_bottom_20_genes_PC2.csv')

# Display the combined DataFrame
mutBpos_with_baseline_top_bottom_20_genes_PC2

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Define custom colormap for PC2 with vcenter set to zero
custom_cmap = mcolors.TwoSlopeNorm(
    vmin=mutBpos_with_baseline_top_bottom_20_genes_PC2['PC2'].min(),
    vcenter=0,
    vmax=mutBpos_with_baseline_top_bottom_20_genes_PC2['PC2'].max()
)

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(10, 10))

# Define colors for the bars based on the PC2 values
colors = plt.cm.RdBu_r(custom_cmap(mutBpos_with_baseline_top_bottom_20_genes_PC2['PC2']))

# Create the horizontal bar plot using the index (gene names) as labels
bars = ax.barh(
    mutBpos_with_baseline_top_bottom_20_genes_PC2.index, 
    mutBpos_with_baseline_top_bottom_20_genes_PC2['PC2'], 
    color=colors
)

# Invert the y-axis to have the top gene at the top
ax.invert_yaxis()

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, orientation='vertical')
cbar.set_label('PC2 Loadings')

# Set labels and title
ax.set_xlabel('PC2 Loadings')
ax.set_ylabel('Proteins')
ax.set_title('PC2 Loadings: Top and Bottom 20 (MutBpos_with_baseline)')

# Show the plot
plt.tight_layout()
plt.savefig('MutBpos_with_baseline_Top_Bottom_20_PC2_loadings.png')
plt.show()

In [ ]:
# Extract only PC3 loadings from the full MutBpos_with_baseline loadings DataFrame
mutBpos_with_baseline_pc3_sorted_df = pd.DataFrame(mutBpos_with_baseline_pc_loadings_df['PC3'])

# Sort genes by their PC3 loading in descending order
mutBpos_with_baseline_pc3_loading_df_sorted = mutBpos_with_baseline_pc3_sorted_df.sort_values(by='PC3', ascending=False)

# Save the sorted PC3 loadings to CSV
mutBpos_with_baseline_pc3_loading_df_sorted.to_csv('mutBpos_with_baseline_pc3_loading_df_sorted.csv')

# Display the sorted DataFrame
mutBpos_with_baseline_pc3_loading_df_sorted

In [ ]:
import matplotlib.colors as mcolors

# Define custom colormap
custom_cmap = mcolors.TwoSlopeNorm(
    vmin=mutBpos_with_baseline_pc3_loading_df_sorted['PC3'].min(),
    vcenter=0,
    vmax=mutBpos_with_baseline_pc3_loading_df_sorted['PC3'].max()
)

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC3 values
colors = plt.cm.RdBu_r(custom_cmap(mutBpos_with_baseline_pc3_loading_df_sorted['PC3']))

# Create the bar plot
bars = ax.bar(
    mutBpos_with_baseline_pc3_loading_df_sorted.index, 
    mutBpos_with_baseline_pc3_loading_df_sorted['PC3'], 
    color=colors
)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC3 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC3 Loadings')
ax.set_title('PC3 Loadings for Proteins (MutBpos_with_baseline)')

# Rotate gene names for better readability
ax.set_xticklabels(mutBpos_with_baseline_pc3_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('MutBpos_with_baseline_All_PC3_loadings.png')
plt.show()


In [ ]:
# Sort the DataFrame by 'PC3' values in descending order
mutBpos_with_baseline_pc3_loading_df_sorted = mutBpos_with_baseline_pc3_loading_df_sorted.sort_values(by='PC3', ascending=False)

# Extract the top 20 and bottom 20 genes
mutBpos_with_baseline_top_20_genes_PC3 = mutBpos_with_baseline_pc3_loading_df_sorted.head(20)
mutBpos_with_baseline_bottom_20_genes_PC3 = mutBpos_with_baseline_pc3_loading_df_sorted.tail(20)

# Combine both into one DataFrame for convenience
mutBpos_with_baseline_top_bottom_20_genes_PC3 = pd.concat([
    mutBpos_with_baseline_top_20_genes_PC3,
    mutBpos_with_baseline_bottom_20_genes_PC3
])

# Save the results to a CSV file
mutBpos_with_baseline_top_bottom_20_genes_PC3.to_csv('mutBpos_with_baseline_top_bottom_20_genes_PC3.csv')

# Display the combined DataFrame
mutBpos_with_baseline_top_bottom_20_genes_PC3

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Define custom colormap for PC3 with vcenter set to zero
custom_cmap = mcolors.TwoSlopeNorm(
    vmin=mutBpos_with_baseline_top_bottom_20_genes_PC3['PC3'].min(),
    vcenter=0,
    vmax=mutBpos_with_baseline_top_bottom_20_genes_PC3['PC3'].max()
)

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(10, 10))

# Define colors for the bars based on the PC3 values
colors = plt.cm.RdBu_r(custom_cmap(mutBpos_with_baseline_top_bottom_20_genes_PC3['PC3']))

# Create the horizontal bar plot using the index (gene names) as labels
bars = ax.barh(
    mutBpos_with_baseline_top_bottom_20_genes_PC3.index, 
    mutBpos_with_baseline_top_bottom_20_genes_PC3['PC3'], 
    color=colors
)

# Invert the y-axis to have the top gene at the top
ax.invert_yaxis()

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, orientation='vertical')
cbar.set_label('PC3 Loadings')

# Set labels and title
ax.set_xlabel('PC3 Loadings')
ax.set_ylabel('Proteins')
ax.set_title('PC3 Loadings: Top and Bottom 20 (MutBpos_with_baseline)')

# Show the plot
plt.tight_layout()
plt.savefig('MutBpos_with_baseline_Top_Bottom_20_PC3_loadings.png')
plt.show()


In [ ]:
# Remove columns with 'DrugA-12months' or 'DrugB-4month' in their names
DrugB_filtered_df = Data.loc[:, ~Data.columns.str.contains('DrugA|MutBpos|MutC|NoMut|Control')]
DrugB_filtered_df

In [ ]:
list(DrugB_filtered_df.columns)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Define your color mapping
samples = {
    # MutB-Baseline group (base purple)
    'MutB-Baseline-1': '#800080',
    'MutB-Baseline-2': '#800080',
    'MutB-Baseline-3': '#800080',
    'MutB-Baseline-4': '#800080',
    'MutB-Baseline-5': '#800080',
    'MutB-Baseline-6': '#800080',
    'MutB-Baseline-7': '#800080',

    # MutB-DrugB-1month group (lighter purple)
    'MutB-DrugB-1month-1': '#a64ca6',
    'MutB-DrugB-1month-2': '#a64ca6',
    'MutB-DrugB-1month-3': '#a64ca6',
    'MutB-DrugB-1month-4': '#a64ca6',
    'MutB-DrugB-1month-5': '#a64ca6',
    'MutB-DrugB-1month-6': '#a64ca6',
    'MutB-DrugB-1month-7': '#a64ca6',

    # MutB-DrugB-2month group (lighter purple)
    'MutB-DrugB-2month-1': '#c080c0',
    'MutB-DrugB-2month-2': '#c080c0',
    'MutB-DrugB-2month-4': '#c080c0',
    'MutB-DrugB-2month-5': '#c080c0',
    'MutB-DrugB-2month-7': '#c080c0',

    # MutB-DrugB-3month group (even lighter purple)
    'MutB-DrugB-3month-1': '#d9b2d9',
    'MutB-DrugB-3month-4': '#d9b2d9',
    'MutB-DrugB-3month-5': '#d9b2d9',
    'MutB-DrugB-3month-7': '#d9b2d9',

    # MutB-DrugB-4month group (pale purple)
    'MutB-DrugB-4month-5': '#ecd9ec',
    'MutB-DrugB-4month-7': '#ecd9ec',

    # MutB-DrugB-5month (very pale purple)
    'MutB-DrugB-5month-5': '#f2e6f2'
}

# Assign colors to all columns for df1 using the mapping
column_colors_df1 = [samples[col] for col in DrugB_filtered_df.columns]

plt.figure(figsize=(50, 50))

plot_df1 = sns.clustermap(
    DrugB_filtered_df, z_score=0, cmap="vlag", figsize=(20, 40),
    vmin=-1, vmax=1, cbar_kws={"label": "z score"},
    col_colors=[column_colors_df1],
    cbar_pos=(1, 0.5, 0.05, 0.15), xticklabels=True
)

plot_df1.ax_heatmap.tick_params(axis='x', colors='white')
plot_df1.cax.set_yticklabels(plot_df1.cax.get_yticklabels(), fontsize=15)
plot_df1.cax.set_ylabel('z score', fontsize=15)

# Legend (adjusted to your df1 content)
legend_labels_df1 = {
    "MutB-Baseline-1": "Baseline",
    "MutB-DrugB-1month-1": "1 month",
    "MutB-DrugB-2month-1": "2 months",
    "MutB-DrugB-3month-1": "3 months",
    "MutB-DrugB-4month-5": "4 months",
    "MutB-DrugB-5month-5": "5 months",
}
handles_df1 = [plt.Rectangle((0, 0), 1.5, 1.5, color=samples[col]) for col in legend_labels_df1.keys()]
legend1_df1 = plot_df1.ax_col_dendrogram.legend(
    handles=handles_df1,
    labels=legend_labels_df1.values(),
    loc='upper left',
    bbox_to_anchor=(1.05, -0.25),
    fontsize='xx-large',
    frameon=False
)
plot_df1.ax_col_dendrogram.add_artist(legend1_df1)

#plt.savefig('MutBpos_Control_df1_clustermap.png', bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(DrugB_filtered_df)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Create an instance of PCA with more components
pca_all = PCA(n_components=len(DrugB_filtered_df.columns))

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_all.fit(data_transposed)

# Calculate the explained variance ratio (as a percentage) for each principal component
explained_variance_ratio_percentage = pca_all.explained_variance_ratio_ * 100

# Create a figure with one subplot
fig, ax = plt.subplots(figsize=(50, 20))

# Plot the bar plot for explained variance of each principal component
ax.bar(np.arange(1, len(DrugB_filtered_df.columns) + 1), explained_variance_ratio_percentage, color='skyblue', alpha=0.8)

# Customize the plot
ax.set_xlabel('Principal Component')
ax.set_ylabel('Percentage of Variance Explained')
ax.set_title('Percentage of Variance Explained by Principal Components')
ax.set_xticks(np.arange(1, len(DrugB_filtered_df.columns) + 1))
#ax.grid(True)

plt.tight_layout()
plt.savefig('DrugB_Eigen_values_Bar_graph.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2
import re

Data_mutBpos = DrugB_filtered_df.copy()

# --- UPDATE: Define group membership based on naming ---
mutB_baseline_samples = [
    'MutB-Baseline-1', 'MutB-Baseline-2', 'MutB-Baseline-3', 'MutB-Baseline-4',
    'MutB-Baseline-5', 'MutB-Baseline-6', 'MutB-Baseline-7'
]
drugB_1month_samples = [
    'MutB-DrugB-1month-1', 'MutB-DrugB-1month-2', 'MutB-DrugB-1month-3',
    'MutB-DrugB-1month-4', 'MutB-DrugB-1month-5', 'MutB-DrugB-1month-6', 'MutB-DrugB-1month-7'
]
drugB_2month_samples = [
    'MutB-DrugB-2month-1', 'MutB-DrugB-2month-2', 'MutB-DrugB-2month-4', 'MutB-DrugB-2month-5', 'MutB-DrugB-2month-7'
]
drugB_3month_samples = [
    'MutB-DrugB-3month-1', 'MutB-DrugB-3month-4', 'MutB-DrugB-3month-5', 'MutB-DrugB-3month-7'
]
drugB_4month_samples = [
    'MutB-DrugB-4month-5', 'MutB-DrugB-4month-7'
]
drugB_5month_samples = [
    'MutB-DrugB-5month-5'
]
all_samples = (
    mutB_baseline_samples + drugB_1month_samples + drugB_2month_samples +
    drugB_3month_samples + drugB_4month_samples + drugB_5month_samples
)

# Now build groupings (replace 'control_cols' and 'mutB_cols')
mutB_cols = mutB_baseline_samples
drugB_cols = (
    drugB_1month_samples + drugB_2month_samples + 
    drugB_3month_samples + drugB_4month_samples + drugB_5month_samples
)
all_cols = mutB_cols + drugB_cols

# --- Use color mapping as previously defined ---
sample_colors = {
    **{k: '#800080' for k in mutB_baseline_samples},
    **{k: '#a64ca6' for k in drugB_1month_samples},
    **{k: '#c080c0' for k in drugB_2month_samples},
    **{k: '#d9b2d9' for k in drugB_3month_samples},
    **{k: '#ecd9ec' for k in drugB_4month_samples},
    **{k: '#f2e6f2' for k in drugB_5month_samples}
}

# --- Custom labels ---
custom_labels = {}
for col in Data_mutBpos.columns:
    # retain original naming for any column not in the above groups
    if col in mutB_baseline_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_B_{number}'
    elif col in drugB_1month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A1_{number}'
    elif col in drugB_2month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A2_{number}'
    elif col in drugB_3month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A3_{number}'
    elif col in drugB_4month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A4_{number}'
    elif col in drugB_5month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A5_{number}'
    else:
        custom_labels[col] = col

# Color dict for PCA plotting
column_colors = {
    col: sample_colors.get(col, 'black') for col in Data_mutBpos.columns
}

# Legend selection
selected_legend_columns = {
    mutB_baseline_samples[0]: 'Baseline',
    drugB_1month_samples[0]: '1 month',
    drugB_2month_samples[0]: '2 months',
    drugB_3month_samples[0]: '3 months',
    drugB_4month_samples[0]: '4 months',
    drugB_5month_samples[0]: '5 months'
}

# Centroid/ellipse plotting: one per group
selected_columns_list = [
    mutB_baseline_samples,
    drugB_1month_samples,
    drugB_2month_samples,
    drugB_3month_samples,
    drugB_4month_samples,
    drugB_5month_samples
]
centroid_colors = {
    mutB_baseline_samples[0]: '#800080',
    drugB_1month_samples[0]: '#a64ca6',
    drugB_2month_samples[0]: '#c080c0',
    drugB_3month_samples[0]: '#d9b2d9',
    drugB_4month_samples[0]: '#ecd9ec',
    drugB_5month_samples[0]: '#f2e6f2'
}
outline_colors = [
    '#800080', '#a64ca6', '#c080c0', '#d9b2d9', '#ecd9ec', '#f2e6f2'
]

# --- PCA and plotting code (no changes needed here) ---
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_mutBpos)
transposed_data = standardized_data.T

pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_mutBpos.columns):
    color = column_colors[col]
    marker = 'D' if col in all_cols else 'x'
    plt.scatter(
        pca_result[i, 0], pca_result[i, 1],
        color=color, marker=marker, s=100, edgecolor='k', linewidth=0.5
    )
    plt.text(
        pca_result[i, 0]-0.1, pca_result[i, 1]+0.1,
        custom_labels[col], fontsize=7, weight='bold', color='black', ha='center', va='center'
    )

# Legend
for col in selected_legend_columns:
    i = Data_mutBpos.columns.get_loc(col)
    color = column_colors[col]
    marker = 'D'
    plt.scatter(
        pca_result[i, 0], pca_result[i, 1],
        color=color, marker=marker, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col]
    )

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_mutBpos.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(
        *centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
        edgecolor='k', linewidth=1
    )
    # Indent everything below this line INSIDE the if
    if group_data.shape[0] > 1:
        cov = np.cov(group_data.T)
        eigvals, eigvecs = np.linalg.eig(cov)
        angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
        ellipse = Ellipse(
            centroid, width, height, angle=angle,
            edgecolor=edge_color, facecolor='none', linewidth=2, linestyle='dashed'
        )
        plt.gca().add_patch(ellipse)


plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of MutB Samples with Baseline and Sequential DrugB Timepoints')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('DrugB_PCA_PC1_PC2.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2
import re

Data_mutBpos = DrugB_filtered_df.copy()

# --- UPDATE: Define group membership based on naming ---
mutB_baseline_samples = [
    'MutB-Baseline-1', 'MutB-Baseline-2', 'MutB-Baseline-3', 'MutB-Baseline-4',
    'MutB-Baseline-5', 'MutB-Baseline-6', 'MutB-Baseline-7'
]
drugB_1month_samples = [
    'MutB-DrugB-1month-1', 'MutB-DrugB-1month-2', 'MutB-DrugB-1month-3',
    'MutB-DrugB-1month-4', 'MutB-DrugB-1month-5', 'MutB-DrugB-1month-6', 'MutB-DrugB-1month-7'
]
drugB_2month_samples = [
    'MutB-DrugB-2month-1', 'MutB-DrugB-2month-2', 'MutB-DrugB-2month-4', 'MutB-DrugB-2month-5', 'MutB-DrugB-2month-7'
]
drugB_3month_samples = [
    'MutB-DrugB-3month-1', 'MutB-DrugB-3month-4', 'MutB-DrugB-3month-5', 'MutB-DrugB-3month-7'
]
drugB_4month_samples = [
    'MutB-DrugB-4month-5', 'MutB-DrugB-4month-7'
]
drugB_5month_samples = [
    'MutB-DrugB-5month-5'
]
all_samples = (
    mutB_baseline_samples + drugB_1month_samples + drugB_2month_samples +
    drugB_3month_samples + drugB_4month_samples + drugB_5month_samples
)

# Now build groupings (replace 'control_cols' and 'mutB_cols')
mutB_cols = mutB_baseline_samples
drugB_cols = (
    drugB_1month_samples + drugB_2month_samples + 
    drugB_3month_samples + drugB_4month_samples + drugB_5month_samples
)
all_cols = mutB_cols + drugB_cols

# --- Use color mapping as previously defined ---
sample_colors = {
    **{k: '#800080' for k in mutB_baseline_samples},
    **{k: '#a64ca6' for k in drugB_1month_samples},
    **{k: '#c080c0' for k in drugB_2month_samples},
    **{k: '#d9b2d9' for k in drugB_3month_samples},
    **{k: '#ecd9ec' for k in drugB_4month_samples},
    **{k: '#f2e6f2' for k in drugB_5month_samples}
}

# --- Custom labels ---
custom_labels = {}
for col in Data_mutBpos.columns:
    # retain original naming for any column not in the above groups
    if col in mutB_baseline_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_B_{number}'
    elif col in drugB_1month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A1_{number}'
    elif col in drugB_2month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A2_{number}'
    elif col in drugB_3month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A3_{number}'
    elif col in drugB_4month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A4_{number}'
    elif col in drugB_5month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A5_{number}'
    else:
        custom_labels[col] = col

# Color dict for PCA plotting
column_colors = {
    col: sample_colors.get(col, 'black') for col in Data_mutBpos.columns
}

# Legend selection
selected_legend_columns = {
    mutB_baseline_samples[0]: 'Baseline',
    drugB_1month_samples[0]: '1 month',
    drugB_2month_samples[0]: '2 months',
    drugB_3month_samples[0]: '3 months',
    drugB_4month_samples[0]: '4 months',
    drugB_5month_samples[0]: '5 months'
}

# Centroid/ellipse plotting: one per group
selected_columns_list = [
    mutB_baseline_samples,
    drugB_1month_samples,
    drugB_2month_samples,
    drugB_3month_samples,
    drugB_4month_samples,
    drugB_5month_samples
]
centroid_colors = {
    mutB_baseline_samples[0]: '#800080',
    drugB_1month_samples[0]: '#a64ca6',
    drugB_2month_samples[0]: '#c080c0',
    drugB_3month_samples[0]: '#d9b2d9',
    drugB_4month_samples[0]: '#ecd9ec',
    drugB_5month_samples[0]: '#f2e6f2'
}
outline_colors = [
    '#800080', '#a64ca6', '#c080c0', '#d9b2d9', '#ecd9ec', '#f2e6f2'
]

# --- PCA and plotting code (no changes needed here) ---
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_mutBpos)
transposed_data = standardized_data.T

pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_mutBpos.columns):
    color = column_colors[col]
    marker = 'D' if col in all_cols else 'x'
    plt.scatter(
        pca_result[i, 0], pca_result[i, 1],
        color=color, marker=marker, s=0, edgecolor='k', linewidth=0.5
    )
    #plt.text(
   #     pca_result[i, 0]-0.1, pca_result[i, 1]+0.1,
    #    custom_labels[col], fontsize=7, weight='bold', color='black', ha='center', va='center'
    #)

# Legend
for col in selected_legend_columns:
    i = Data_mutBpos.columns.get_loc(col)
    color = column_colors[col]
    marker = 'D'
    plt.scatter(
        pca_result[i, 0], pca_result[i, 1],
        color=color, marker=marker, s=0, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col]
    )

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_mutBpos.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(
        *centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
        edgecolor='k', linewidth=1
    )
    # Indent everything below this line INSIDE the if
    if group_data.shape[0] > 1:
        cov = np.cov(group_data.T)
        eigvals, eigvecs = np.linalg.eig(cov)
        angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
       # ellipse = Ellipse(
         #   centroid, width, height, angle=angle,
        #    edgecolor=edge_color, facecolor='none', linewidth=2, linestyle='dashed'
        #)
        #plt.gca().add_patch(ellipse)


plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of MutB Samples with Baseline and Sequential DrugB Timepoints')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('DrugB_PCA_PC1_PC2_centroids.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2
import re

Data_mutBpos = DrugB_filtered_df.copy()

# --- UPDATE: Define group membership based on naming ---
mutB_baseline_samples = [
    'MutB-Baseline-1', 'MutB-Baseline-2', 'MutB-Baseline-3', 'MutB-Baseline-4',
    'MutB-Baseline-5', 'MutB-Baseline-6', 'MutB-Baseline-7'
]
drugB_1month_samples = [
    'MutB-DrugB-1month-1', 'MutB-DrugB-1month-2', 'MutB-DrugB-1month-3',
    'MutB-DrugB-1month-4', 'MutB-DrugB-1month-5', 'MutB-DrugB-1month-6', 'MutB-DrugB-1month-7'
]
drugB_2month_samples = [
    'MutB-DrugB-2month-1', 'MutB-DrugB-2month-2', 'MutB-DrugB-2month-4', 'MutB-DrugB-2month-5', 'MutB-DrugB-2month-7'
]
drugB_3month_samples = [
    'MutB-DrugB-3month-1', 'MutB-DrugB-3month-4', 'MutB-DrugB-3month-5', 'MutB-DrugB-3month-7'
]
drugB_4month_samples = [
    'MutB-DrugB-4month-5', 'MutB-DrugB-4month-7'
]
drugB_5month_samples = [
    'MutB-DrugB-5month-5'
]
all_samples = (
    mutB_baseline_samples + drugB_1month_samples + drugB_2month_samples +
    drugB_3month_samples + drugB_4month_samples + drugB_5month_samples
)

mutB_cols = mutB_baseline_samples
drugB_cols = (
    drugB_1month_samples + drugB_2month_samples + 
    drugB_3month_samples + drugB_4month_samples + drugB_5month_samples
)
all_cols = mutB_cols + drugB_cols

sample_colors = {
    **{k: '#800080' for k in mutB_baseline_samples},
    **{k: '#a64ca6' for k in drugB_1month_samples},
    **{k: '#c080c0' for k in drugB_2month_samples},
    **{k: '#d9b2d9' for k in drugB_3month_samples},
    **{k: '#ecd9ec' for k in drugB_4month_samples},
    **{k: '#f2e6f2' for k in drugB_5month_samples}
}

custom_labels = {}
for col in Data_mutBpos.columns:
    if col in mutB_baseline_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_B_{number}'
    elif col in drugB_1month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A1_{number}'
    elif col in drugB_2month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A2_{number}'
    elif col in drugB_3month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A3_{number}'
    elif col in drugB_4month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A4_{number}'
    elif col in drugB_5month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A5_{number}'
    else:
        custom_labels[col] = col

column_colors = {
    col: sample_colors.get(col, 'black') for col in Data_mutBpos.columns
}

selected_legend_columns = {
    mutB_baseline_samples[0]: 'Baseline',
    drugB_1month_samples[0]: '1 month',
    drugB_2month_samples[0]: '2 months',
    drugB_3month_samples[0]: '3 months',
    drugB_4month_samples[0]: '4 months',
    drugB_5month_samples[0]: '5 months'
}

selected_columns_list = [
    mutB_baseline_samples,
    drugB_1month_samples,
    drugB_2month_samples,
    drugB_3month_samples,
    drugB_4month_samples,
    drugB_5month_samples
]
centroid_colors = {
    mutB_baseline_samples[0]: '#800080',
    drugB_1month_samples[0]: '#a64ca6',
    drugB_2month_samples[0]: '#c080c0',
    drugB_3month_samples[0]: '#d9b2d9',
    drugB_4month_samples[0]: '#ecd9ec',
    drugB_5month_samples[0]: '#f2e6f2'
}
outline_colors = [
    '#800080', '#a64ca6', '#c080c0', '#d9b2d9', '#ecd9ec', '#f2e6f2'
]

scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_mutBpos)
transposed_data = standardized_data.T

pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_mutBpos.columns):
    color = column_colors[col]
    marker = 'D' if col in all_cols else 'x'
    plt.scatter(
        pca_result[i, 0], pca_result[i, 2],   # PC1 vs PC3
        color=color, marker=marker, s=100, edgecolor='k', linewidth=0.5
    )
    plt.text(
        pca_result[i, 0]-0.1, pca_result[i, 2]+0.1,   # PC1 vs PC3
        custom_labels[col], fontsize=7, weight='bold', color='black', ha='center', va='center'
    )

for col in selected_legend_columns:
    i = Data_mutBpos.columns.get_loc(col)
    color = column_colors[col]
    marker = 'D'
    plt.scatter(
        pca_result[i, 0], pca_result[i, 2],   # PC1 vs PC3
        color=color, marker=marker, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col]
    )

for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_mutBpos.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices][:, [0, 2]]  # PC1 vs PC3
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(
        *centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
        edgecolor='k', linewidth=1
    )
    if group_data.shape[0] > 1:
        cov = np.cov(group_data.T)
        eigvals, eigvecs = np.linalg.eig(cov)
        angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
        ellipse = Ellipse(
            centroid, width, height, angle=angle,
            edgecolor=edge_color, facecolor='none', linewidth=2, linestyle='dashed'
        )
        plt.gca().add_patch(ellipse)

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of MutB Samples with Baseline and Sequential DrugB Timepoints (PC1 vs PC3)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('DrugB_PCA_PC1_PC3.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2
import re

Data_mutBpos = DrugB_filtered_df.copy()

# --- UPDATE: Define group membership based on naming ---
mutB_baseline_samples = [
    'MutB-Baseline-1', 'MutB-Baseline-2', 'MutB-Baseline-3', 'MutB-Baseline-4',
    'MutB-Baseline-5', 'MutB-Baseline-6', 'MutB-Baseline-7'
]
drugB_1month_samples = [
    'MutB-DrugB-1month-1', 'MutB-DrugB-1month-2', 'MutB-DrugB-1month-3',
    'MutB-DrugB-1month-4', 'MutB-DrugB-1month-5', 'MutB-DrugB-1month-6', 'MutB-DrugB-1month-7'
]
drugB_2month_samples = [
    'MutB-DrugB-2month-1', 'MutB-DrugB-2month-2', 'MutB-DrugB-2month-4', 'MutB-DrugB-2month-5', 'MutB-DrugB-2month-7'
]
drugB_3month_samples = [
    'MutB-DrugB-3month-1', 'MutB-DrugB-3month-4', 'MutB-DrugB-3month-5', 'MutB-DrugB-3month-7'
]
drugB_4month_samples = [
    'MutB-DrugB-4month-5', 'MutB-DrugB-4month-7'
]
drugB_5month_samples = [
    'MutB-DrugB-5month-5'
]
all_samples = (
    mutB_baseline_samples + drugB_1month_samples + drugB_2month_samples +
    drugB_3month_samples + drugB_4month_samples + drugB_5month_samples
)

mutB_cols = mutB_baseline_samples
drugB_cols = (
    drugB_1month_samples + drugB_2month_samples + 
    drugB_3month_samples + drugB_4month_samples + drugB_5month_samples
)
all_cols = mutB_cols + drugB_cols

sample_colors = {
    **{k: '#800080' for k in mutB_baseline_samples},
    **{k: '#a64ca6' for k in drugB_1month_samples},
    **{k: '#c080c0' for k in drugB_2month_samples},
    **{k: '#d9b2d9' for k in drugB_3month_samples},
    **{k: '#ecd9ec' for k in drugB_4month_samples},
    **{k: '#f2e6f2' for k in drugB_5month_samples}
}

custom_labels = {}
for col in Data_mutBpos.columns:
    if col in mutB_baseline_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_B_{number}'
    elif col in drugB_1month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A1_{number}'
    elif col in drugB_2month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A2_{number}'
    elif col in drugB_3month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A3_{number}'
    elif col in drugB_4month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A4_{number}'
    elif col in drugB_5month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A5_{number}'
    else:
        custom_labels[col] = col

column_colors = {
    col: sample_colors.get(col, 'black') for col in Data_mutBpos.columns
}

selected_legend_columns = {
    mutB_baseline_samples[0]: 'Baseline',
    drugB_1month_samples[0]: '1 month',
    drugB_2month_samples[0]: '2 months',
    drugB_3month_samples[0]: '3 months',
    drugB_4month_samples[0]: '4 months',
    drugB_5month_samples[0]: '5 months'
}

selected_columns_list = [
    mutB_baseline_samples,
    drugB_1month_samples,
    drugB_2month_samples,
    drugB_3month_samples,
    drugB_4month_samples,
    drugB_5month_samples
]
centroid_colors = {
    mutB_baseline_samples[0]: '#800080',
    drugB_1month_samples[0]: '#a64ca6',
    drugB_2month_samples[0]: '#c080c0',
    drugB_3month_samples[0]: '#d9b2d9',
    drugB_4month_samples[0]: '#ecd9ec',
    drugB_5month_samples[0]: '#f2e6f2'
}
outline_colors = [
    '#800080', '#a64ca6', '#c080c0', '#d9b2d9', '#ecd9ec', '#f2e6f2'
]

scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_mutBpos)
transposed_data = standardized_data.T

pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_mutBpos.columns):
    color = column_colors[col]
    marker = 'D' if col in all_cols else 'x'
    plt.scatter(
        pca_result[i, 0], pca_result[i, 2],   # PC1 vs PC3
        color=color, marker=marker, s=0, edgecolor='k', linewidth=0.5
    )
    #plt.text(
      #  pca_result[i, 0]-0.1, pca_result[i, 2]+0.1,   # PC1 vs PC3
       # custom_labels[col], fontsize=7, weight='bold', color='black', ha='center', va='center'
   # )

for col in selected_legend_columns:
    i = Data_mutBpos.columns.get_loc(col)
    color = column_colors[col]
    marker = 'D'
    plt.scatter(
        pca_result[i, 0], pca_result[i, 2],   # PC1 vs PC3
        color=color, marker=marker, s=0, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col]
    )

for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_mutBpos.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices][:, [0, 2]]  # PC1 vs PC3
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(
        *centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
        edgecolor='k', linewidth=1
    )
    if group_data.shape[0] > 1:
        cov = np.cov(group_data.T)
        eigvals, eigvecs = np.linalg.eig(cov)
        angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
        #ellipse = Ellipse(
          #  centroid, width, height, angle=angle,
          #  edgecolor=edge_color, facecolor='none', linewidth=2, linestyle='dashed'
        #)
        #plt.gca().add_patch(ellipse)

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of MutB Samples with Baseline and Sequential DrugB Timepoints (PC1 vs PC3)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('DrugB_PCA_PC1_PC3_centroids.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2
import re

Data_mutBpos = DrugB_filtered_df.copy()

# --- Define group membership ---
mutB_baseline_samples = [
    'MutB-Baseline-1', 'MutB-Baseline-2', 'MutB-Baseline-3', 'MutB-Baseline-4',
    'MutB-Baseline-5', 'MutB-Baseline-6', 'MutB-Baseline-7'
]
drugB_1month_samples = [
    'MutB-DrugB-1month-1', 'MutB-DrugB-1month-2', 'MutB-DrugB-1month-3',
    'MutB-DrugB-1month-4', 'MutB-DrugB-1month-5', 'MutB-DrugB-1month-6', 'MutB-DrugB-1month-7'
]
drugB_2month_samples = [
    'MutB-DrugB-2month-1', 'MutB-DrugB-2month-2', 'MutB-DrugB-2month-4', 'MutB-DrugB-2month-5', 'MutB-DrugB-2month-7'
]
drugB_3month_samples = [
    'MutB-DrugB-3month-1', 'MutB-DrugB-3month-4', 'MutB-DrugB-3month-5', 'MutB-DrugB-3month-7'
]
drugB_4month_samples = [
    'MutB-DrugB-4month-5', 'MutB-DrugB-4month-7'
]
drugB_5month_samples = [
    'MutB-DrugB-5month-5'
]
all_samples = (
    mutB_baseline_samples + drugB_1month_samples + drugB_2month_samples +
    drugB_3month_samples + drugB_4month_samples + drugB_5month_samples
)

mutB_cols = mutB_baseline_samples
drugB_cols = (
    drugB_1month_samples + drugB_2month_samples + 
    drugB_3month_samples + drugB_4month_samples + drugB_5month_samples
)
all_cols = mutB_cols + drugB_cols

sample_colors = {
    **{k: '#800080' for k in mutB_baseline_samples},
    **{k: '#a64ca6' for k in drugB_1month_samples},
    **{k: '#c080c0' for k in drugB_2month_samples},
    **{k: '#d9b2d9' for k in drugB_3month_samples},
    **{k: '#ecd9ec' for k in drugB_4month_samples},
    **{k: '#f2e6f2' for k in drugB_5month_samples}
}

custom_labels = {}
for col in Data_mutBpos.columns:
    if col in mutB_baseline_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_B_{number}'
    elif col in drugB_1month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A1_{number}'
    elif col in drugB_2month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A2_{number}'
    elif col in drugB_3month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A3_{number}'
    elif col in drugB_4month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A4_{number}'
    elif col in drugB_5month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A5_{number}'
    else:
        custom_labels[col] = col

column_colors = {
    col: sample_colors.get(col, 'black') for col in Data_mutBpos.columns
}

selected_legend_columns = {
    mutB_baseline_samples[0]: 'Baseline',
    drugB_1month_samples[0]: '1 month',
    drugB_2month_samples[0]: '2 months',
    drugB_3month_samples[0]: '3 months',
    drugB_4month_samples[0]: '4 months',
    drugB_5month_samples[0]: '5 months'
}

selected_columns_list = [
    mutB_baseline_samples,
    drugB_1month_samples,
    drugB_2month_samples,
    drugB_3month_samples,
    drugB_4month_samples,
    drugB_5month_samples
]
centroid_colors = {
    mutB_baseline_samples[0]: '#800080',
    drugB_1month_samples[0]: '#a64ca6',
    drugB_2month_samples[0]: '#c080c0',
    drugB_3month_samples[0]: '#d9b2d9',
    drugB_4month_samples[0]: '#ecd9ec',
    drugB_5month_samples[0]: '#f2e6f2'
}
outline_colors = [
    '#800080', '#a64ca6', '#c080c0', '#d9b2d9', '#ecd9ec', '#f2e6f2'
]

scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_mutBpos)
transposed_data = standardized_data.T

# Compute three principal components for PC2 vs PC3
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_mutBpos.columns):
    color = column_colors[col]
    marker = 'D' if col in all_cols else 'x'
    plt.scatter(
        pca_result[i, 1], pca_result[i, 2],   # PC2 vs PC3
        color=color, marker=marker, s=100, edgecolor='k', linewidth=0.5
    )
    plt.text(
        pca_result[i, 1]-0.1, pca_result[i, 2]+0.1,   # PC2 vs PC3
        custom_labels[col], fontsize=7, weight='bold', color='black', ha='center', va='center'
    )

for col in selected_legend_columns:
    i = Data_mutBpos.columns.get_loc(col)
    color = column_colors[col]
    marker = 'D'
    plt.scatter(
        pca_result[i, 1], pca_result[i, 2],   # PC2 vs PC3
        color=color, marker=marker, s=100, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col]
    )

for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_mutBpos.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices][:, [1, 2]]  # PC2 vs PC3
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(
        *centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
        edgecolor='k', linewidth=1
    )
    if group_data.shape[0] > 1:
        cov = np.cov(group_data.T)
        eigvals, eigvecs = np.linalg.eig(cov)
        angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
        ellipse = Ellipse(
            centroid, width, height, angle=angle,
            edgecolor=edge_color, facecolor='none', linewidth=2, linestyle='dashed'
        )
        plt.gca().add_patch(ellipse)

plt.xlabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of MutB Samples with Baseline and Sequential DrugB Timepoints (PC2 vs PC3)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('DrugB_PCA_PC2_PC3.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2
import re

Data_mutBpos = DrugB_filtered_df.copy()

# --- Define group membership ---
mutB_baseline_samples = [
    'MutB-Baseline-1', 'MutB-Baseline-2', 'MutB-Baseline-3', 'MutB-Baseline-4',
    'MutB-Baseline-5', 'MutB-Baseline-6', 'MutB-Baseline-7'
]
drugB_1month_samples = [
    'MutB-DrugB-1month-1', 'MutB-DrugB-1month-2', 'MutB-DrugB-1month-3',
    'MutB-DrugB-1month-4', 'MutB-DrugB-1month-5', 'MutB-DrugB-1month-6', 'MutB-DrugB-1month-7'
]
drugB_2month_samples = [
    'MutB-DrugB-2month-1', 'MutB-DrugB-2month-2', 'MutB-DrugB-2month-4', 'MutB-DrugB-2month-5', 'MutB-DrugB-2month-7'
]
drugB_3month_samples = [
    'MutB-DrugB-3month-1', 'MutB-DrugB-3month-4', 'MutB-DrugB-3month-5', 'MutB-DrugB-3month-7'
]
drugB_4month_samples = [
    'MutB-DrugB-4month-5', 'MutB-DrugB-4month-7'
]
drugB_5month_samples = [
    'MutB-DrugB-5month-5'
]
all_samples = (
    mutB_baseline_samples + drugB_1month_samples + drugB_2month_samples +
    drugB_3month_samples + drugB_4month_samples + drugB_5month_samples
)

mutB_cols = mutB_baseline_samples
drugB_cols = (
    drugB_1month_samples + drugB_2month_samples + 
    drugB_3month_samples + drugB_4month_samples + drugB_5month_samples
)
all_cols = mutB_cols + drugB_cols

sample_colors = {
    **{k: '#800080' for k in mutB_baseline_samples},
    **{k: '#a64ca6' for k in drugB_1month_samples},
    **{k: '#c080c0' for k in drugB_2month_samples},
    **{k: '#d9b2d9' for k in drugB_3month_samples},
    **{k: '#ecd9ec' for k in drugB_4month_samples},
    **{k: '#f2e6f2' for k in drugB_5month_samples}
}

custom_labels = {}
for col in Data_mutBpos.columns:
    if col in mutB_baseline_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_B_{number}'
    elif col in drugB_1month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A1_{number}'
    elif col in drugB_2month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A2_{number}'
    elif col in drugB_3month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A3_{number}'
    elif col in drugB_4month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A4_{number}'
    elif col in drugB_5month_samples:
        match = re.search(r'-(\d+)$', col)
        number = match.group(1) if match else ''
        custom_labels[col] = f'MutB_A5_{number}'
    else:
        custom_labels[col] = col

column_colors = {
    col: sample_colors.get(col, 'black') for col in Data_mutBpos.columns
}

selected_legend_columns = {
    mutB_baseline_samples[0]: 'Baseline',
    drugB_1month_samples[0]: '1 month',
    drugB_2month_samples[0]: '2 months',
    drugB_3month_samples[0]: '3 months',
    drugB_4month_samples[0]: '4 months',
    drugB_5month_samples[0]: '5 months'
}

selected_columns_list = [
    mutB_baseline_samples,
    drugB_1month_samples,
    drugB_2month_samples,
    drugB_3month_samples,
    drugB_4month_samples,
    drugB_5month_samples
]
centroid_colors = {
    mutB_baseline_samples[0]: '#800080',
    drugB_1month_samples[0]: '#a64ca6',
    drugB_2month_samples[0]: '#c080c0',
    drugB_3month_samples[0]: '#d9b2d9',
    drugB_4month_samples[0]: '#ecd9ec',
    drugB_5month_samples[0]: '#f2e6f2'
}
outline_colors = [
    '#800080', '#a64ca6', '#c080c0', '#d9b2d9', '#ecd9ec', '#f2e6f2'
]

scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_mutBpos)
transposed_data = standardized_data.T

# Compute three principal components for PC2 vs PC3
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_mutBpos.columns):
    color = column_colors[col]
    marker = 'D' if col in all_cols else 'x'
    plt.scatter(
        pca_result[i, 1], pca_result[i, 2],   # PC2 vs PC3
        color=color, marker=marker, s=0, edgecolor='k', linewidth=0.5
    )
   # plt.text(
      #  pca_result[i, 1]-0.1, pca_result[i, 2]+0.1,   # PC2 vs PC3
       # custom_labels[col], fontsize=7, weight='bold', color='black', ha='center', va='center'
    #)

for col in selected_legend_columns:
    i = Data_mutBpos.columns.get_loc(col)
    color = column_colors[col]
    marker = 'D'
    plt.scatter(
        pca_result[i, 1], pca_result[i, 2],   # PC2 vs PC3
        color=color, marker=marker, s=0, edgecolor='k', linewidth=0.5, label=selected_legend_columns[col]
    )

for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_mutBpos.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices][:, [1, 2]]  # PC2 vs PC3
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(
        *centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
        edgecolor='k', linewidth=1
    )
    if group_data.shape[0] > 1:
        cov = np.cov(group_data.T)
        eigvals, eigvecs = np.linalg.eig(cov)
        angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
        width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
        #ellipse = Ellipse(
          #  centroid, width, height, angle=angle,
           # edgecolor=edge_color, facecolor='none', linewidth=2, linestyle='dashed'
        #)
        #plt.gca().add_patch(ellipse)

plt.xlabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of MutB Samples with Baseline and Sequential DrugB Timepoints (PC2 vs PC3)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('DrugB_PCA_PC2_PC3_centroids.png')
plt.show()

In [ ]:
# Extract PC loadings (components show how much each gene contributes to each PC)
DrugB_pc_loadings = pca.components_

# Create DataFrame: genes as index, PCs as columns
DrugB_pc_loadings_df = pd.DataFrame(DrugB_pc_loadings.T, columns=['PC1', 'PC2', 'PC3'], index=Data_mutBpos.index)

# Save to CSV
#DrugB_pc_loadings_df.to_csv('DrugB_pc_loadings_df.csv')

# Display the loadings
DrugB_pc_loadings_df

In [ ]:
# Extract only PC1 loadings from the full MutBpos_with_baseline loadings DataFrame
DrugB_pc1_sorted_df = pd.DataFrame(DrugB_pc_loadings_df['PC1'])

# Sort genes by their PC1 loading in descending order
DrugB_pc1_loading_df_sorted = DrugB_pc1_sorted_df.sort_values(by='PC1', ascending=False)

# Save the sorted PC1 loadings to CSV
DrugB_pc1_loading_df_sorted.to_csv('DrugB_pc1_loading_df_sorted.csv')

# Display the sorted DataFrame
DrugB_pc1_loading_df_sorted

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Ensure DrugB_pc1_loading_df_sorted is available:
# It must be a DataFrame with 'PC1' as a column, and index as protein/gene names

# Define custom colormap normalization for the DrugB PC1 loadings
custom_cmap_drugB = mcolors.TwoSlopeNorm(
    vmin=DrugB_pc1_loading_df_sorted['PC1'].min(),
    vcenter=0,
    vmax=DrugB_pc1_loading_df_sorted['PC1'].max()
)

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC1 values using the RdBu_r colormap
colors = plt.cm.RdBu_r(custom_cmap_drugB(DrugB_pc1_loading_df_sorted['PC1']))

# Create the bar plot for DrugB PC1 loadings
bars = ax.bar(
    DrugB_pc1_loading_df_sorted.index,
    DrugB_pc1_loading_df_sorted['PC1'],
    color=colors
)

# Add colorbar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap_drugB)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC1 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC1 Loadings')
ax.set_title('PC1 Loadings for Proteins (DrugB)')

# Rotate gene names for better readability
ax.set_xticklabels(DrugB_pc1_loading_df_sorted.index, rotation=90)

# Adjust layout, save and show plot
plt.tight_layout()
plt.savefig('DrugB_All_PC1_loadings.png')
plt.show()


In [ ]:
# Extract only PC1 loadings from the full MutBpos_with_baseline loadings DataFrame
DrugB_pc2_sorted_df = pd.DataFrame(DrugB_pc_loadings_df['PC2'])

# Sort genes by their PC1 loading in descending order
DrugB_pc2_loading_df_sorted = DrugB_pc2_sorted_df.sort_values(by='PC2', ascending=False)

# Save the sorted PC1 loadings to CSV
DrugB_pc2_loading_df_sorted.to_csv('DrugB_pc2_loading_df_sorted.csv')

# Display the sorted DataFrame
DrugB_pc2_loading_df_sorted

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Extract only PC2 loadings (assuming DrugB_pc_loadings_df has a 'PC2' column)
DrugB_pc2_sorted_df = pd.DataFrame(DrugB_pc_loadings_df['PC2'])
DrugB_pc2_loading_df_sorted = DrugB_pc2_sorted_df.sort_values(by='PC2', ascending=False)

# Define custom colormap normalization for PC2
custom_cmap_drugB_pc2 = mcolors.TwoSlopeNorm(
    vmin=DrugB_pc2_loading_df_sorted['PC2'].min(),
    vcenter=0,
    vmax=DrugB_pc2_loading_df_sorted['PC2'].max()
)

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC2 values using the RdBu_r colormap
colors = plt.cm.RdBu_r(custom_cmap_drugB_pc2(DrugB_pc2_loading_df_sorted['PC2']))

# Create the bar plot
bars = ax.bar(
    DrugB_pc2_loading_df_sorted.index,
    DrugB_pc2_loading_df_sorted['PC2'],
    color=colors
)

# Add colorbar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap_drugB_pc2)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC2 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC2 Loadings')
ax.set_title('PC2 Loadings for Proteins (DrugB)')

# Properly set ticks to avoid warnings
ax.set_xticks(range(len(DrugB_pc2_loading_df_sorted.index)))
ax.set_xticklabels(DrugB_pc2_loading_df_sorted.index, rotation=90)

plt.tight_layout()
plt.savefig('DrugB_All_PC2_loadings.png')
plt.show()

In [ ]:
# Extract only PC1 loadings from the full MutBpos_with_baseline loadings DataFrame
DrugB_pc3_sorted_df = pd.DataFrame(DrugB_pc_loadings_df['PC3'])

# Sort genes by their PC1 loading in descending order
DrugB_pc3_loading_df_sorted = DrugB_pc3_sorted_df.sort_values(by='PC3', ascending=False)

# Save the sorted PC1 loadings to CSV
DrugB_pc3_loading_df_sorted.to_csv('DrugB_pc3_loading_df_sorted.csv')

# Display the sorted DataFrame
DrugB_pc3_loading_df_sorted

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Extract only PC3 loadings (assuming DrugB_pc_loadings_df has a 'PC3' column)
DrugB_pc3_sorted_df = pd.DataFrame(DrugB_pc_loadings_df['PC3'])
DrugB_pc3_loading_df_sorted = DrugB_pc3_sorted_df.sort_values(by='PC3', ascending=False)

# Define custom colormap normalization for PC3
custom_cmap_drugB_pc3 = mcolors.TwoSlopeNorm(
    vmin=DrugB_pc3_loading_df_sorted['PC3'].min(),
    vcenter=0,
    vmax=DrugB_pc3_loading_df_sorted['PC3'].max()
)

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC3 values using the RdBu_r colormap
colors = plt.cm.RdBu_r(custom_cmap_drugB_pc3(DrugB_pc3_loading_df_sorted['PC3']))

# Create the bar plot
bars = ax.bar(
    DrugB_pc3_loading_df_sorted.index,
    DrugB_pc3_loading_df_sorted['PC3'],
    color=colors
)

# Add colorbar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap_drugB_pc3)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC3 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC3 Loadings')
ax.set_title('PC3 Loadings for Proteins (DrugB)')

# Properly set ticks to avoid warnings
ax.set_xticks(range(len(DrugB_pc3_loading_df_sorted.index)))
ax.set_xticklabels(DrugB_pc3_loading_df_sorted.index, rotation=90)

plt.tight_layout()
plt.savefig('DrugB_All_PC3_loadings.png')
plt.show()


In [ ]:
df1 = pd.read_excel("NoMut_Correlations.xlsx", sheet_name=0, index_col=1)  # 0 refers to the first sheet
df1

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df1['Score-R Slope']
y = df1['NEFH']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='green')

# Index labels
for index, row in df1.iterrows():
    plt.text(row['Score-R Slope'], row['NEFH'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('NEFH')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.4f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('NEFH vs Score-R Slope.png', bbox_inches='tight')

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df1['Score-R Slope']
y = df1['NEFH']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='green')

# Index labels
for index, row in df1.iterrows():
    plt.text(row['Score-R Slope'], row['NEFH'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('NEFH')
plt.grid(False)

# ✅ Set y-axis range only
plt.ylim(15, 35)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.4f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('NEFH vs Score-R Slope y axis altered.png', bbox_inches='tight')

plt.show()

In [ ]:
# Remove patient number 65 from the DataFrame df1
df1_wo65 = df1[df1.index != 65]
df1_wo65

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df1_wo65['Score-R Slope']
y = df1_wo65['NEFH']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='green')

# Index labels
for index, row in df1_wo65.iterrows():
    plt.text(row['Score-R Slope'], row['NEFH'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('NEFH')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('NEFH vs Score-R Slope wo outlier.png', bbox_inches='tight')

plt.show()

In [ ]:
df2 = pd.read_excel("NoMut_Correlations.xlsx", sheet_name=1, index_col=1)  # 0 refers to the first sheet
df2

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df2['Score-R Slope']
y = df2['NEFL']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='green')

# Index labels
for index, row in df2.iterrows():
    plt.text(row['Score-R Slope'], row['NEFL'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('NEFL')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('NEFL vs Score-R Slope.png', bbox_inches='tight')

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df2['Score-R Slope']
y = df2['NEFL']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='green')

# Index labels
for index, row in df2.iterrows():
    plt.text(row['Score-R Slope'], row['NEFL'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('NEFL')
plt.grid(False)

# ✅ Set y-axis range only
plt.ylim(12, 30)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('NEFL vs Score-R Slope y axis altered.png', bbox_inches='tight')

plt.show()

In [ ]:
# Remove patient number 65 from the DataFrame df1
df2_wo65 = df2[df2.index != 65]
df2_wo65

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df2_wo65['Score-R Slope']
y = df2_wo65['NEFL']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='green')

# Index labels
for index, row in df2_wo65.iterrows():
    plt.text(row['Score-R Slope'], row['NEFL'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('NEFL')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('NEFL vs Score-R Slope wo outlier.png', bbox_inches='tight')

plt.show()

In [ ]:
df3 = pd.read_excel("NoMut_Correlations.xlsx", sheet_name=2, index_col=1)  # 0 refers to the first sheet
df3

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df3['Score-R Slope']
y = df3['CHIT1']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='green')

# Index labels
for index, row in df3.iterrows():
    plt.text(row['Score-R Slope'], row['CHIT1'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('CHIT1')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('CHIT1 vs Score-R Slope.png', bbox_inches='tight')

plt.show()

In [ ]:
df3_wo65 = df3[df3.index != 65]
df3_wo65

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df3_wo65['Score-R Slope']
y = df3_wo65['CHIT1']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='green')

# Index labels
for index, row in df3_wo65.iterrows():
    plt.text(row['Score-R Slope'], row['CHIT1'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('CHIT1')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('CHIT1 vs Score-R Slope wo outlier.png', bbox_inches='tight')

plt.show()

In [ ]:
df4 = pd.read_excel("NoMut_Correlations.xlsx", sheet_name=3, index_col=1)  # 0 refers to the first sheet
df4

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df4['Score-R Slope']
y = df4['CCL2']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='green')

# Index labels
for index, row in df4.iterrows():
    plt.text(row['Score-R Slope'], row['CCL2'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('CCL2')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('CCL2 vs Score-R Slope.png', bbox_inches='tight')

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df4['Score-R Slope']
y = df4['CCL2']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='green')

# Index labels
for index, row in df4.iterrows():
    plt.text(row['Score-R Slope'], row['CCL2'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('CCL2')
plt.grid(False)

# ✅ Set y-axis range only
plt.ylim(10, 17)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('CCL2 vs Score-R Slope y axis altered.png', bbox_inches='tight')

plt.show()

In [ ]:
df4_wo65 = df4[df4.index != 65]
df4_wo65

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df4_wo65['Score-R Slope']
y = df4_wo65['CCL2']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='green')

# Index labels
for index, row in df4_wo65.iterrows():
    plt.text(row['Score-R Slope'], row['CCL2'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('CCL2')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('CCL2 vs Score-R Slope wo outlier.png', bbox_inches='tight')

plt.show()

In [ ]:
df5 = pd.read_excel("NoMut_Correlations.xlsx", sheet_name=4, index_col=1)  # 0 refers to the first sheet
df5

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df5['Score-R Slope']
y = df5['CCL3']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='green')

# Index labels
for index, row in df5.iterrows():
    plt.text(row['Score-R Slope'], row['CCL3'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('CCL3')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('CCL3 vs Score-R Slope.png', bbox_inches='tight')

plt.show()

In [ ]:
df5_wo65 = df5[df5.index != 65]
df5_wo65

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df5_wo65['Score-R Slope']
y = df5_wo65['CCL3']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='green')

# Index labels
for index, row in df5_wo65.iterrows():
    plt.text(row['Score-R Slope'], row['CCL3'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('CCL3')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('CCL3 vs Score-R Slope wo outlier.png', bbox_inches='tight')

plt.show()

In [ ]:
df6 = pd.read_excel("NoMut_Correlations.xlsx", sheet_name=5, index_col=1)  # 0 refers to the first sheet
df6 = df6.fillna(0)
df6

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df6['Score-R Slope']
y = df6['CHI3L']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='green')

# Index labels
for index, row in df6.iterrows():
    plt.text(row['Score-R Slope'], row['CHI3L'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('CHI3L')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('CHI3L vs Score-R Slope.png', bbox_inches='tight')

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df6['Score-R Slope']
y = df6['CHI3L']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='green')

# Index labels
for index, row in df6.iterrows():
    plt.text(row['Score-R Slope'], row['CHI3L'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('CHI3L')
plt.grid(False)

# ✅ Set y-axis range only
plt.ylim(10, 22)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('CHI3L vs Score-R Slope y axis altered.png', bbox_inches='tight')

plt.show()

In [ ]:
df6 = pd.read_excel("NoMut_Correlations.xlsx", sheet_name=5, index_col=1)  # 0 refers to the first sheet
df6 = df6.fillna(0)
df6

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df6['Score-R Slope']
y = df6['CHI3L']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='green')

# Index labels
for index, row in df6.iterrows():
    plt.text(row['Score-R Slope'], row['CHI3L'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('CHI3L')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('CHI3L vs Score-R Slope.png', bbox_inches='tight')

plt.show()

In [ ]:
df6_wo65 = df6[df6.index != 65]
df6_wo65

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df6_wo65['Score-R Slope']
y = df6_wo65['CHI3L']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='green')

# Index labels
for index, row in df6_wo65.iterrows():
    plt.text(row['Score-R Slope'], row['CHI3L'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('CHI3L')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('CHI3L vs Score-R Slope wo outliers.png', bbox_inches='tight')

plt.show()

In [ ]:
df7 = pd.read_excel("NoMut_Correlations.xlsx", sheet_name=6, index_col=1)  # 0 refers to the first sheet
df7 = df7.fillna(0)
df7

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Replace NaN with 0 in df7
df7 = df7.fillna(0)

# Prepare data
x = df7['Score-R Slope']
y = df7['IL7']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='green')

# Index labels
for index, row in df7.iterrows():
    plt.text(row['Score-R Slope'], row['IL7'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('IL7')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('IL7 vs Score-R Slope.png', bbox_inches='tight')

plt.show()

In [ ]:
df8 = pd.read_excel("NoMut_Correlations.xlsx", sheet_name=7, index_col=1)  # 0 refers to the first sheet
df8 = df8.fillna(0)
df8

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Replace NaN with 0 in df8
df8 = df8.fillna(0)

# Prepare data
x = df8['Score-R Slope']
y = df8['GFAP']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='green')

# Index labels
for index, row in df8.iterrows():
    plt.text(row['Score-R Slope'], row['GFAP'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('GFAP')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('GFAP vs Score-R Slope.png', bbox_inches='tight')

plt.show()

In [ ]:
df9 = pd.read_excel("NoMut_Correlations.xlsx", sheet_name=8, index_col=1)  # 0 refers to the first sheet
df9 = df9.fillna(0)
df9

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Replace NaN with 0 in df9
df9 = df9.fillna(0)

# Prepare data
x = df9['Score-R Slope']
y = df9['TNF']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='green')

# Index labels
for index, row in df9.iterrows():
    plt.text(row['Score-R Slope'], row['TNF'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('TNF')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('TNF vs Score-R Slope.png', bbox_inches='tight')

plt.show()

In [ ]:
df10 = pd.read_excel("NoMut_Correlations.xlsx", sheet_name=9, index_col=1)  # 0 refers to the first sheet
df10 = df10.fillna(0)
df10

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Replace NaN with 0 in df10
df10 = df10.fillna(0)

# Prepare data
x = df10['Score-R Slope']
y = df10['UCHL1']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='green')

# Index labels
for index, row in df10.iterrows():
    plt.text(row['Score-R Slope'], row['UCHL1'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('UCHL1')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('UCHL1 vs Score-R Slope.png', bbox_inches='tight')

plt.show()

In [ ]:
df10 = pd.read_excel("NoMut_Correlations.xlsx", sheet_name=9, index_col=1)  # 0 refers to the first sheet
df10 = df10.fillna(0)
df10 = df10[df10.index != 65]
df10

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Replace NaN with 0 in df10
df10 = df10.fillna(0)

# Prepare data
x = df10['Score-R Slope']
y = df10['UCHL1']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='green')

# Index labels
for index, row in df10.iterrows():
    plt.text(row['Score-R Slope'], row['UCHL1'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('UCHL1')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('UCHL1 vs Score-R Slope wo outlier.png', bbox_inches='tight')

plt.show()

In [ ]:
df11 = pd.read_excel("MutBpos_Correlations.xlsx", sheet_name=0, index_col=1)  # 0 refers to the first sheet
df11 = df11.fillna(0)
df11 = df11[df11.index != 77]
df11

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df11['Score-R Slope']
y = df11['NEFH']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='D', s=100, color='#800080')

# Index labels
for index, row in df11.iterrows():
    plt.text(row['Score-R Slope'], row['NEFH'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('NEFH')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('MutBpos NEFH vs Score-R Slope.png', bbox_inches='tight')

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df11['Score-R Slope']
y = df11['NEFH']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

pearson_corr, pearson_p = pearsonr(x, y)

plt.figure(figsize=(10, 10))

# Scatter points
plt.scatter(x, y, marker='D', s=100, color='#800080')

# Create x grid from 0 to 6
x_grid = np.linspace(0, 6, 200)
X_grid = sm.add_constant(x_grid)

# Predicted values and CI from statsmodels
pred = model.get_prediction(X_grid)
pred_mean = pred.predicted_mean
pred_ci_low, pred_ci_high = pred.conf_int().T   # 95% CI

# Plot CI band
plt.fill_between(x_grid, pred_ci_low, pred_ci_high,
                 color='grey', alpha=0.4, linewidth=0)

# Plot regression line
plt.plot(x_grid, pred_mean, color='black', linestyle='--')

# X-limits
plt.xlim(0, 6)

# Index labels
for index, row in df11.iterrows():
    plt.text(row['Score-R Slope'], row['NEFH'], str(index),
             fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('NEFH')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()
plt.savefig('MutBpos NEFH vs Score-R Slope axis altered.png', bbox_inches='tight')
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df11['Score-R Slope']
y = df11['NEFH']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

pearson_corr, pearson_p = pearsonr(x, y)

plt.figure(figsize=(10, 10))

# Scatter points
plt.scatter(x, y, marker='D', s=100, color='#800080')

# Create x grid from 0 to 6
x_grid = np.linspace(0, 6, 200)
X_grid = sm.add_constant(x_grid)

# Predicted values and CI from statsmodels
pred = model.get_prediction(X_grid)
pred_mean = pred.predicted_mean
pred_ci_low, pred_ci_high = pred.conf_int().T   # 95% CI

# Plot CI band
plt.fill_between(x_grid, pred_ci_low, pred_ci_high,
                 color='grey', alpha=0.4, linewidth=0)

# Plot regression line
plt.plot(x_grid, pred_mean, color='black', linestyle='--')

# X-limits
plt.xlim(0, 6)

# Index labels
for index, row in df11.iterrows():
    plt.text(row['Score-R Slope'], row['NEFH'], str(index),
             fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('NEFH')
plt.grid(False)

# ✅ Set y-axis range only
plt.ylim(15, 35)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()
plt.savefig('MutBpos NEFH vs Score-R Slope x and y axis altered.png', bbox_inches='tight')
plt.show()


In [ ]:
df12 = pd.read_excel("MutBpos_Correlations.xlsx", sheet_name=1, index_col=1)  # 0 refers to the first sheet
df12 = df12.fillna(0)
df12 = df12[df12.index != 77]
df12

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df12['Score-R Slope']
y = df12['NEFL']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='D', s=100, color='#800080')

# Index labels
for index, row in df12.iterrows():
    plt.text(row['Score-R Slope'], row['NEFL'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('NEFL')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('MutBpos NEFL vs Score-R Slope.png', bbox_inches='tight')

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df12['Score-R Slope']
y = df12['NEFL']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

pearson_corr, pearson_p = pearsonr(x, y)

plt.figure(figsize=(10, 10))

# Scatter points
plt.scatter(x, y, marker='D', s=100, color='#800080')

# Create x grid from 0 to 6
x_grid = np.linspace(0, 6, 200)
X_grid = sm.add_constant(x_grid)

# Predicted values and CI from statsmodels
pred = model.get_prediction(X_grid)
pred_mean = pred.predicted_mean
pred_ci_low, pred_ci_high = pred.conf_int().T   # 95% CI

# Plot CI band
plt.fill_between(x_grid, pred_ci_low, pred_ci_high,
                 color='grey', alpha=0.4, linewidth=0)

# Plot regression line
plt.plot(x_grid, pred_mean, color='black', linestyle='--')

# X-limits
plt.xlim(0, 6)

# Index labels
for index, row in df12.iterrows():
    plt.text(row['Score-R Slope'], row['NEFL'], str(index),
             fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('NEFL')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()
plt.savefig('MutBpos NEFL vs Score-R Slope axis altered.png', bbox_inches='tight')
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df12['Score-R Slope']
y = df12['NEFL']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

pearson_corr, pearson_p = pearsonr(x, y)

plt.figure(figsize=(10, 10))

# Scatter points
plt.scatter(x, y, marker='D', s=100, color='#800080')

# Create x grid from 0 to 6
x_grid = np.linspace(0, 6, 200)
X_grid = sm.add_constant(x_grid)

# Predicted values and CI from statsmodels
pred = model.get_prediction(X_grid)
pred_mean = pred.predicted_mean
pred_ci_low, pred_ci_high = pred.conf_int().T   # 95% CI

# Plot CI band
plt.fill_between(x_grid, pred_ci_low, pred_ci_high,
                 color='grey', alpha=0.4, linewidth=0)

# Plot regression line
plt.plot(x_grid, pred_mean, color='black', linestyle='--')

# X-limits
plt.xlim(0, 6)

# Index labels
for index, row in df12.iterrows():
    plt.text(row['Score-R Slope'], row['NEFL'], str(index),
             fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('NEFL')
plt.grid(False)

# ✅ Set y-axis range only
plt.ylim(12, 30)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()
plt.savefig('MutBpos NEFL vs Score-R Slope x and y axis altered.png', bbox_inches='tight')
plt.show()

In [ ]:
df13 = pd.read_excel("MutBpos_Correlations.xlsx", sheet_name=2, index_col=1)  # 0 refers to the first sheet
df13 = df13.fillna(0)
df13 = df13[df13.index != 77]
df13

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df13['Score-R Slope']
y = df13['CHIT1']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='D', s=100, color='#800080')

# Index labels
for index, row in df13.iterrows():
    plt.text(row['Score-R Slope'], row['CHIT1'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('CHIT1')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('MutBpos CHIT1 vs Score-R Slope.png', bbox_inches='tight')

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df13['Score-R Slope']
y = df13['CHIT1']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

pearson_corr, pearson_p = pearsonr(x, y)

plt.figure(figsize=(10, 10))

# Scatter points
plt.scatter(x, y, marker='D', s=100, color='#800080')

# Create x grid from 0 to 6
x_grid = np.linspace(0, 6, 200)
X_grid = sm.add_constant(x_grid)

# Predicted values and CI from statsmodels
pred = model.get_prediction(X_grid)
pred_mean = pred.predicted_mean
pred_ci_low, pred_ci_high = pred.conf_int().T   # 95% CI

# Plot CI band
plt.fill_between(x_grid, pred_ci_low, pred_ci_high,
                 color='grey', alpha=0.4, linewidth=0)

# Plot regression line
plt.plot(x_grid, pred_mean, color='black', linestyle='--')

# X-limits
plt.xlim(0, 6)

# Index labels
for index, row in df13.iterrows():
    plt.text(row['Score-R Slope'], row['CHIT1'], str(index),
             fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('CHIT1')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()
plt.savefig('MutBpos CHIT1 vs Score-R Slope axis altered.png', bbox_inches='tight')
plt.show()


In [ ]:
df14 = pd.read_excel("MutBpos_Correlations.xlsx", sheet_name=3, index_col=1)  # 0 refers to the first sheet
df14 = df14.fillna(0)
df14 = df14[df14.index != 77]
df14

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df14['Score-R Slope']
y = df14['CHI3L1']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='D', s=100, color='#800080')

# Index labels
for index, row in df14.iterrows():
    plt.text(row['Score-R Slope'], row['CHI3L1'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('CHI3L1')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('MutBpos CHI3L1 vs Score-R Slope.png', bbox_inches='tight')

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df14['Score-R Slope']
y = df14['CHI3L1']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

pearson_corr, pearson_p = pearsonr(x, y)

plt.figure(figsize=(10, 10))

# Scatter points
plt.scatter(x, y, marker='D', s=100, color='#800080')

# Create x grid from 0 to 6
x_grid = np.linspace(0, 6, 200)
X_grid = sm.add_constant(x_grid)

# Predicted values and CI from statsmodels
pred = model.get_prediction(X_grid)
pred_mean = pred.predicted_mean
pred_ci_low, pred_ci_high = pred.conf_int().T   # 95% CI

# Plot CI band
plt.fill_between(x_grid, pred_ci_low, pred_ci_high,
                 color='grey', alpha=0.4, linewidth=0)

# Plot regression line
plt.plot(x_grid, pred_mean, color='black', linestyle='--')

# X-limits
plt.xlim(0, 6)

# Index labels
for index, row in df14.iterrows():
    plt.text(row['Score-R Slope'], row['CHI3L1'], str(index),
             fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('CHI3L1')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()
plt.savefig('MutBpos CHI3L1 vs Score-R Slope axis altered.png', bbox_inches='tight')
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df14['Score-R Slope']
y = df14['CHI3L1']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

pearson_corr, pearson_p = pearsonr(x, y)

plt.figure(figsize=(10, 10))

# Scatter points
plt.scatter(x, y, marker='D', s=100, color='#800080')

# Create x grid from 0 to 6
x_grid = np.linspace(0, 6, 200)
X_grid = sm.add_constant(x_grid)

# Predicted values and CI from statsmodels
pred = model.get_prediction(X_grid)
pred_mean = pred.predicted_mean
pred_ci_low, pred_ci_high = pred.conf_int().T   # 95% CI

# Plot CI band
plt.fill_between(x_grid, pred_ci_low, pred_ci_high,
                 color='grey', alpha=0.4, linewidth=0)

# Plot regression line
plt.plot(x_grid, pred_mean, color='black', linestyle='--')

# X-limits
plt.xlim(0, 6)

# Index labels
for index, row in df14.iterrows():
    plt.text(row['Score-R Slope'], row['CHI3L1'], str(index),
             fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('CHI3L1')
plt.grid(False)

# ✅ Set y-axis range only
plt.ylim(10, 22)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()
plt.savefig('MutBpos CHI3L1 vs Score-R Slope x and y axis altered.png', bbox_inches='tight')
plt.show()

In [ ]:
df15 = pd.read_excel("MutBpos_Correlations.xlsx", sheet_name=4, index_col=1)  # 0 refers to the first sheet
df15 = df15.fillna(0)
df15 = df15[df15.index != 77]
df15

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df15['Score-R Slope']
y = df15['CCL2']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='D', s=100, color='#800080')

# Index labels
for index, row in df15.iterrows():
    plt.text(row['Score-R Slope'], row['CCL2'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('CCL2')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('MutBpos CCL2 vs Score-R Slope.png', bbox_inches='tight')

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df15['Score-R Slope']
y = df15['CCL2']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

pearson_corr, pearson_p = pearsonr(x, y)

plt.figure(figsize=(10, 10))

# Scatter points
plt.scatter(x, y, marker='D', s=100, color='#800080')

# Create x grid from 0 to 6
x_grid = np.linspace(0, 6, 200)
X_grid = sm.add_constant(x_grid)

# Predicted values and CI from statsmodels
pred = model.get_prediction(X_grid)
pred_mean = pred.predicted_mean
pred_ci_low, pred_ci_high = pred.conf_int().T   # 95% CI

# Plot CI band
plt.fill_between(x_grid, pred_ci_low, pred_ci_high,
                 color='grey', alpha=0.4, linewidth=0)

# Plot regression line
plt.plot(x_grid, pred_mean, color='black', linestyle='--')

# X-limits
plt.xlim(0, 6)

# Index labels
for index, row in df15.iterrows():
    plt.text(row['Score-R Slope'], row['CCL2'], str(index),
             fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('CCL2')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()
plt.savefig('MutBpos CCL2 vs Score-R Slope axis altered.png', bbox_inches='tight')
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df15['Score-R Slope']
y = df15['CCL2']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

pearson_corr, pearson_p = pearsonr(x, y)

plt.figure(figsize=(10, 10))

# Scatter points
plt.scatter(x, y, marker='D', s=100, color='#800080')

# Create x grid from 0 to 6
x_grid = np.linspace(0, 6, 200)
X_grid = sm.add_constant(x_grid)

# Predicted values and CI from statsmodels
pred = model.get_prediction(X_grid)
pred_mean = pred.predicted_mean
pred_ci_low, pred_ci_high = pred.conf_int().T   # 95% CI

# Plot CI band
plt.fill_between(x_grid, pred_ci_low, pred_ci_high,
                 color='grey', alpha=0.4, linewidth=0)

# Plot regression line
plt.plot(x_grid, pred_mean, color='black', linestyle='--')

# X-limits
plt.xlim(0, 6)

# Index labels
for index, row in df15.iterrows():
    plt.text(row['Score-R Slope'], row['CCL2'], str(index),
             fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('CCL2')
plt.grid(False)

# ✅ Set y-axis range only
plt.ylim(10, 17)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()
plt.savefig('MutBpos CCL2 vs Score-R Slope x and y axis altered.png', bbox_inches='tight')
plt.show()

In [ ]:
df16 = pd.read_excel("MutBpos_Correlations.xlsx", sheet_name=5, index_col=1)  # 0 refers to the first sheet
df16 = df16.fillna(0)
df16 = df16[df16.index != 77]
df16

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df16['Score-R Slope']
y = df16['CCL3']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='D', s=100, color='#800080')

# Index labels
for index, row in df16.iterrows():
    plt.text(row['Score-R Slope'], row['CCL3'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('CCL3')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('MutBpos CCL3 vs Score-R Slope.png', bbox_inches='tight')

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df16['Score-R Slope']
y = df16['CCL3']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

pearson_corr, pearson_p = pearsonr(x, y)

plt.figure(figsize=(10, 10))

# Scatter points
plt.scatter(x, y, marker='D', s=100, color='#800080')

# Create x grid from 0 to 6
x_grid = np.linspace(0, 6, 200)
X_grid = sm.add_constant(x_grid)

# Predicted values and CI from statsmodels
pred = model.get_prediction(X_grid)
pred_mean = pred.predicted_mean
pred_ci_low, pred_ci_high = pred.conf_int().T   # 95% CI

# Plot CI band
plt.fill_between(x_grid, pred_ci_low, pred_ci_high,
                 color='grey', alpha=0.4, linewidth=0)

# Plot regression line
plt.plot(x_grid, pred_mean, color='black', linestyle='--')

# X-limits
plt.xlim(0, 6)

# Index labels
for index, row in df16.iterrows():
    plt.text(row['Score-R Slope'], row['CCL3'], str(index),
             fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('CCL3')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()
plt.savefig('MutBpos CCL3 vs Score-R Slope axis altered.png', bbox_inches='tight')
plt.show()


In [ ]:
df17 = pd.read_excel("MutBpos_Correlations.xlsx", sheet_name=6, index_col=1)  # 0 refers to the first sheet
df17 = df17.fillna(0)
df17 = df17[df17.index != 77]
df17

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df17['Score-R Slope']
y = df17['TREM1']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='D', s=100, color='#800080')

# Index labels
for index, row in df17.iterrows():
    plt.text(row['Score-R Slope'], row['TREM1'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('TREM1')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('MutBpos TREM1 vs Score-R Slope.png', bbox_inches='tight')

plt.show()

In [ ]:
df18 = pd.read_excel("MutBpos_Correlations.xlsx", sheet_name=7, index_col=1)  # 0 refers to the first sheet
df18 = df18.fillna(0)
df18 = df18[df18.index != 77]
df18

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df18['Score-R Slope']
y = df18['ENO2']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='D', s=100, color='#800080')

# Index labels
for index, row in df18.iterrows():
    plt.text(row['Score-R Slope'], row['ENO2'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('ENO2')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('MutBpos ENO2 vs Score-R Slope.png', bbox_inches='tight')

plt.show()

In [ ]:
df19 = pd.read_excel("MutBpos_Correlations.xlsx", sheet_name=8, index_col=1)  # 0 refers to the first sheet
df19 = df19.fillna(0)
df19 = df19[df19.index != 77]
df19

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df19['Score-R Slope']
y = df19['UCHL1']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='D', s=100, color='#800080')

# Index labels
for index, row in df19.iterrows():
    plt.text(row['Score-R Slope'], row['UCHL1'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('UCHL1')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('MutBpos UCHL1 vs Score-R Slope.png', bbox_inches='tight')

plt.show()

In [ ]:
df20 = pd.read_excel("MutBpos_Correlations.xlsx", sheet_name=9, index_col=1)  # 0 refers to the first sheet
df20 = df20.fillna(0)
df20 = df20[df20.index != 77]
df20

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df20['Score-R Slope']
y = df20['IL12p70']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='D', s=100, color='#800080')

# Index labels
for index, row in df20.iterrows():
    plt.text(row['Score-R Slope'], row['IL12p70'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('IL12p70')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('MutBpos IL12p70 vs Score-R Slope.png', bbox_inches='tight')

plt.show()

In [ ]:
df21 = pd.read_excel("MutBpos_Correlations.xlsx", sheet_name=10, index_col=1)  # 0 refers to the first sheet
df21 = df21.fillna(0)
df21 = df21[df21.index != 77]
df21

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df21['Score-R Slope']
y = df21['TIMP3']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='D', s=100, color='#800080')

# Index labels
for index, row in df21.iterrows():
    plt.text(row['Score-R Slope'], row['TIMP3'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('TIMP3')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('MutBpos TIMP3 vs Score-R Slope.png', bbox_inches='tight')

plt.show()

In [ ]:
df22 = pd.read_excel("MutBpos_Correlations.xlsx", sheet_name=11, index_col=1)  # 0 refers to the first sheet
df22 = df22.fillna(0)
df22 = df22[df22.index != 77]
df22

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df22['Score-R Slope']
y = df22['IL9']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='D', s=100, color='#800080')

# Index labels
for index, row in df22.iterrows():
    plt.text(row['Score-R Slope'], row['IL9'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('IL9')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('MutBpos IL9 vs Score-R Slope.png', bbox_inches='tight')

plt.show()

In [ ]:
df23 = pd.read_excel("MutBpos_Correlations.xlsx", sheet_name=12, index_col=1)  # 0 refers to the first sheet
df23 = df23.fillna(0)
df23 = df23[df23.index != 77]
df23

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df23['Score-R Slope']
y = df23['IFNG']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='D', s=100, color='#800080')

# Index labels
for index, row in df23.iterrows():
    plt.text(row['Score-R Slope'], row['IFNG'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('IFNG')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('MutBpos IFNG vs Score-R Slope.png', bbox_inches='tight')

plt.show()

In [ ]:
df24 = pd.read_excel("GeoMean_Corr.xlsx", sheet_name=0, index_col=0)  # 0 refers to the first sheet
df24

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df24['Score-R NoMut']
y = df24['Geo_mean']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R NoMut']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R NoMut']
intercept = model.params['const']
slope_se = model.bse['Score-R NoMut']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='green')

# Index labels
for index, row in df24.iterrows():
    plt.text(row['Score-R NoMut'], row['Geo_mean'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('Geo_mean')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.4f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('Geo_mean NoMut vs Score-R Slope.png', bbox_inches='tight')

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df24['Score-R NoMut']
y = df24['Geo_mean']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R NoMut']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R NoMut']
intercept = model.params['const']
slope_se = model.bse['Score-R NoMut']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='green')

# Index labels
for index, row in df24.iterrows():
    plt.text(row['Score-R NoMut'], row['Geo_mean'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('Geo_mean')
plt.grid(False)

# ✅ Set y-axis range only
plt.ylim(0.5, 2)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.4f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('Geo_mean NoMut vs Score-R Slope axis altered.png', bbox_inches='tight')

plt.show()

In [ ]:
df24_wo65 = df24[df24.index != 29]
df24_wo65

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df24_wo65['Score-R NoMut']
y = df24_wo65['Geo_mean']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R NoMut']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R NoMut']
intercept = model.params['const']
slope_se = model.bse['Score-R NoMut']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='green')

# Index labels
for index, row in df24_wo65.iterrows():
    plt.text(row['Score-R NoMut'], row['Geo_mean'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('Geo_mean')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.4f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('Geo_mean NoMut vs Score-R Slope wo outlier.png', bbox_inches='tight')

plt.show()

In [ ]:
df25 = pd.read_excel("GeoMean_Corr.xlsx", sheet_name=1, index_col=0)  # 0 refers to the first sheet
df25

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df25['Score-R MutB']
y = df25['Geo_mean']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R MutB']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R MutB']
intercept = model.params['const']
slope_se = model.bse['Score-R MutB']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='D', s=100, color='#800080')

# Index labels
for index, row in df25.iterrows():
    plt.text(row['Score-R MutB'], row['Geo_mean'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('GeoMean MutB')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('GeoMean MutBpos vs Score-R Slope.png', bbox_inches='tight')

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df25['Score-R MutB']
y = df25['Geo_mean']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R MutB']
intercept_conf_int = model.conf_int().loc['const']

r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R MutB']
intercept = model.params['const']
slope_se = model.bse['Score-R MutB']
intercept_se = model.bse['const']

pearson_corr, pearson_p = pearsonr(x, y)

plt.figure(figsize=(10, 10))

# Scatter points
plt.scatter(x, y, marker='D', s=100, color='#800080')

# Create x grid from 0 to 6
x_grid = np.linspace(0, 6, 200)
X_grid = sm.add_constant(x_grid)

# Predicted values and CI from statsmodels
pred = model.get_prediction(X_grid)
pred_mean = pred.predicted_mean
pred_ci_low, pred_ci_high = pred.conf_int().T   # 95% CI

# Plot CI band
plt.fill_between(x_grid, pred_ci_low, pred_ci_high,
                 color='grey', alpha=0.4, linewidth=0)

# Plot regression line
plt.plot(x_grid, pred_mean, color='black', linestyle='--')

# X-limits
plt.xlim(0, 6)

# Index labels
for index, row in df25.iterrows():
    plt.text(row['Score-R MutB'], row['Geo_mean'], str(index),
             fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('Geo_mean')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["MutB", f"{slope:.2f} ± {slope_se:.2f}"],
    ["MutB 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()
plt.savefig('MutBpos Geo_mean vs Score-R Slope MutBpos axis altered.png', bbox_inches='tight')
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df25['Score-R MutB']
y = df25['Geo_mean']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R MutB']
intercept_conf_int = model.conf_int().loc['const']

r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R MutB']
intercept = model.params['const']
slope_se = model.bse['Score-R MutB']
intercept_se = model.bse['const']

pearson_corr, pearson_p = pearsonr(x, y)

plt.figure(figsize=(10, 10))

# Scatter points
plt.scatter(x, y, marker='D', s=100, color='#800080')

# Create x grid from 0 to 6
x_grid = np.linspace(0, 6, 200)
X_grid = sm.add_constant(x_grid)

# Predicted values and CI from statsmodels
pred = model.get_prediction(X_grid)
pred_mean = pred.predicted_mean
pred_ci_low, pred_ci_high = pred.conf_int().T   # 95% CI

# Plot CI band
plt.fill_between(x_grid, pred_ci_low, pred_ci_high,
                 color='grey', alpha=0.4, linewidth=0)

# Plot regression line
plt.plot(x_grid, pred_mean, color='black', linestyle='--')

# X-limits
plt.xlim(0, 6)

# Index labels
for index, row in df25.iterrows():
    plt.text(row['Score-R MutB'], row['Geo_mean'], str(index),
             fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('Geo_mean')
plt.grid(False)

plt.ylim(0.5, 2)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["MutB", f"{slope:.2f} ± {slope_se:.2f}"],
    ["MutB 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()
plt.savefig('MutBpos Geo_mean vs Score-R Slope MutBpos x and y axis altered.png', bbox_inches='tight')
plt.show()

In [ ]:
df26 = pd.read_excel("GeoMean_Corr.xlsx", sheet_name=2, index_col=0)  # 0 refers to the first sheet
df26

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df26['Score-R NoMut']
y = df26['Geo_mean']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R NoMut']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R NoMut']
intercept = model.params['const']
slope_se = model.bse['Score-R NoMut']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='green')

# Index labels
for index, row in df26.iterrows():
    plt.text(row['Score-R NoMut'], row['Geo_mean'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('Geo_mean_wo_NFs')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.4f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('Geo_mean_wo_NFs NoMut vs Score-R Slope.png', bbox_inches='tight')

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df26['Score-R NoMut']
y = df26['Geo_mean']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R NoMut']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R NoMut']
intercept = model.params['const']
slope_se = model.bse['Score-R NoMut']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='green')

# Index labels
for index, row in df26.iterrows():
    plt.text(row['Score-R NoMut'], row['Geo_mean'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('Geo_mean_wo_NFs')
plt.grid(False)

plt.ylim(0.5, 2)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.4f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('Geo_mean_wo_NFs NoMut vs Score-R Slope axis altered.png', bbox_inches='tight')

plt.show()

In [ ]:
df26_wo65 = df26[df26.index != 29]
df26_wo65

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df26_wo65['Score-R NoMut']
y = df26_wo65['Geo_mean']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R NoMut']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R NoMut']
intercept = model.params['const']
slope_se = model.bse['Score-R NoMut']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='green')

# Index labels
for index, row in df26_wo65.iterrows():
    plt.text(row['Score-R NoMut'], row['Geo_mean'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('Geo_mean_wo_NFs')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.4f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('Geo_mean_wo_NFs NoMut vs Score-R Slope wo outliers.png', bbox_inches='tight')

plt.show()

In [ ]:
df27 = pd.read_excel("GeoMean_Corr.xlsx", sheet_name=3, index_col=0)  # 0 refers to the first sheet
df27

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df27['Score-R MutB']
y = df27['Geo_mean']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R MutB']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R MutB']
intercept = model.params['const']
slope_se = model.bse['Score-R MutB']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='D', s=100, color='#800080')

# Index labels
for index, row in df27.iterrows():
    plt.text(row['Score-R MutB'], row['Geo_mean'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('GeoMean_wo_NFs MutB')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('GeoMean_wo_NFs MutBpos vs Score-R Slope.png', bbox_inches='tight')

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df27['Score-R MutB']
y = df27['Geo_mean']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R MutB']
intercept_conf_int = model.conf_int().loc['const']

r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R MutB']
intercept = model.params['const']
slope_se = model.bse['Score-R MutB']
intercept_se = model.bse['const']

pearson_corr, pearson_p = pearsonr(x, y)

plt.figure(figsize=(10, 10))

# Scatter points
plt.scatter(x, y, marker='D', s=100, color='#800080')

# Create x grid from 0 to 6
x_grid = np.linspace(0, 6, 200)
X_grid = sm.add_constant(x_grid)

# Predicted values and CI from statsmodels
pred = model.get_prediction(X_grid)
pred_mean = pred.predicted_mean
pred_ci_low, pred_ci_high = pred.conf_int().T   # 95% CI

# Plot CI band
plt.fill_between(x_grid, pred_ci_low, pred_ci_high,
                 color='grey', alpha=0.4, linewidth=0)

# Plot regression line
plt.plot(x_grid, pred_mean, color='black', linestyle='--')

# X-limits
plt.xlim(0, 6)

# Index labels
for index, row in df27.iterrows():
    plt.text(row['Score-R MutB'], row['Geo_mean'], str(index),
             fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('GeoMean_wo_NFs')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["MutB", f"{slope:.2f} ± {slope_se:.2f}"],
    ["MutB 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()
plt.savefig('MutBpos GeoMean_wo_NFs vs Score-R Slope MutBpos axis altered.png', bbox_inches='tight')
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df27['Score-R MutB']
y = df27['Geo_mean']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R MutB']
intercept_conf_int = model.conf_int().loc['const']

r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R MutB']
intercept = model.params['const']
slope_se = model.bse['Score-R MutB']
intercept_se = model.bse['const']

pearson_corr, pearson_p = pearsonr(x, y)

plt.figure(figsize=(10, 10))

# Scatter points
plt.scatter(x, y, marker='D', s=100, color='#800080')

# Create x grid from 0 to 6
x_grid = np.linspace(0, 6, 200)
X_grid = sm.add_constant(x_grid)

# Predicted values and CI from statsmodels
pred = model.get_prediction(X_grid)
pred_mean = pred.predicted_mean
pred_ci_low, pred_ci_high = pred.conf_int().T   # 95% CI

# Plot CI band
plt.fill_between(x_grid, pred_ci_low, pred_ci_high,
                 color='grey', alpha=0.4, linewidth=0)

# Plot regression line
plt.plot(x_grid, pred_mean, color='black', linestyle='--')

# X-limits
plt.xlim(0, 6)

# Index labels
for index, row in df27.iterrows():
    plt.text(row['Score-R MutB'], row['Geo_mean'], str(index),
             fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('GeoMean_wo_NFs')
plt.grid(False)

plt.ylim(0.5,2)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["MutB", f"{slope:.2f} ± {slope_se:.2f}"],
    ["MutB 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()
plt.savefig('MutBpos GeoMean_wo_NFs vs Score-R Slope MutBpos x and y axis altered.png', bbox_inches='tight')
plt.show()


In [ ]:
df28 = pd.read_excel("Geomean_corr_Case_patients.xlsx", sheet_name=0, index_col=0)  # 0 refers to the first sheet
df28

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# df28: index = patient IDs (e.g. 'S1', 'S2', 'C1', 'C2'),
#       columns: 'Geo_mean', 'Score-R'

# Prepare data
x = df28['Score-R']
y = df28['Geo_mean']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R']
intercept_conf_int = model.conf_int().loc['const']

# Model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R']
intercept = model.params['const']
slope_se = model.bse['Score-R']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# --- custom scatter points by patient type ---

# Boolean masks based on index prefix: all S* and all C*
mask_S = df28.index.str.startswith('S')
mask_C = df28.index.str.startswith('C')
mask_other = ~(mask_S | mask_C)

# S* patients: green triangles
plt.scatter(
    df28.loc[mask_S, 'Score-R'],
    df28.loc[mask_S, 'Geo_mean'],
    marker='^',           # triangle
    s=100,
    color='green',
    label='S patients'
)

# C* patients: purple diamonds
plt.scatter(
    df28.loc[mask_C, 'Score-R'],
    df28.loc[mask_C, 'Geo_mean'],
    marker='D',           # diamond
    s=100,
    color='#800080',
    label='C patients'
)

# Optional: plot all remaining patients
plt.scatter(
    df28.loc[mask_other, 'Score-R'],
    df28.loc[mask_other, 'Geo_mean'],
    marker='D',
    s=80,
    color='gray',
    alpha=0.6,
    label='Other patients'
)

# Index labels for all patients
for idx, row in df28.iterrows():
    plt.text(
        row['Score-R'],
        row['Geo_mean'],
        str(idx),
        fontsize=9,
        weight='bold'
    )

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('GeoMean')
plt.grid(False)
plt.legend(loc='upper left')

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('GeoMean_vs_ScoreR_Slope_highlighted.png', bbox_inches='tight')
plt.show()

In [ ]:
df29 = pd.read_excel("Geomean_corr_Case_patients.xlsx", sheet_name=1, index_col=0)  # 0 refers to the first sheet
df29

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# df28: index = patient IDs (e.g. 'S1', 'S2', 'C1', 'C2'),
#       columns: 'Geo_mean', 'Score-R'

# Prepare data
x = df29['Score-R']
y = df29['Geo_mean']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R']
intercept_conf_int = model.conf_int().loc['const']

# Model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R']
intercept = model.params['const']
slope_se = model.bse['Score-R']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# --- custom scatter points by patient type ---

# Boolean masks based on index prefix: all S* and all C*
mask_S = df29.index.str.startswith('S')
mask_C = df29.index.str.startswith('C')
mask_other = ~(mask_S | mask_C)

# S* patients: green triangles
plt.scatter(
    df29.loc[mask_S, 'Score-R'],
    df29.loc[mask_S, 'Geo_mean'],
    marker='^',           # triangle
    s=100,
    color='green',
    label='S patients'
)

# C* patients: purple diamonds
plt.scatter(
    df29.loc[mask_C, 'Score-R'],
    df29.loc[mask_C, 'Geo_mean'],
    marker='D',           # diamond
    s=100,
    color='#800080',
    label='C patients'
)

# Optional: plot all remaining patients
plt.scatter(
    df29.loc[mask_other, 'Score-R'],
    df29.loc[mask_other, 'Geo_mean'],
    marker='D',
    s=80,
    color='gray',
    alpha=0.6,
    label='Other patients'
)

# Index labels for all patients
for idx, row in df28.iterrows():
    plt.text(
        row['Score-R'],
        row['Geo_mean'],
        str(idx),
        fontsize=9,
        weight='bold'
    )

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('GeoMean')
plt.grid(False)
plt.legend(loc='upper left')

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('GeoMean_wo_NFs_vs_ScoreR_Slope_highlighted.png', bbox_inches='tight')
plt.show()

In [ ]:
df30 = pd.read_excel("Case_Corr.xlsx", sheet_name=0, index_col=0)  # 0 refers to the first sheet
df30

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# df30: index = sample names (contain 'NoMut' or not),
#       columns: 'Geo_mean', 'Score-R'

# Prepare data
x = df30['Score-R Slope']
y = df30['NEFH']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# --- custom scatter points by sample type ---

# Boolean masks based on sample name containing 'NoMut'
mask_noMut = df30.index.str.contains('NoMut', case=False, na=False)
mask_other = ~mask_noMut

# NoMut samples: green triangles
plt.scatter(
    df30.loc[mask_noMut, 'Score-R Slope'],
    df30.loc[mask_noMut, 'NEFH'],
    marker='^',           # triangle
    s=100,
    color='green',
    label='NoMut Case'
)

# All other samples: purple diamonds
plt.scatter(
    df30.loc[mask_other, 'Score-R Slope'],
    df30.loc[mask_other, 'NEFH'],
    marker='D',           # diamond
    s=100,
    color='#800080',
    label='Other'
)


# Axis titles
plt.title('Scatter Plot df30', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('NEFH')
plt.grid(False)
plt.legend(loc='upper left')

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('NEFH_Case_all_vs_ScoreR_Slope.png', bbox_inches='tight')
plt.show()

In [ ]:
df31= pd.read_excel("Case_Corr.xlsx", sheet_name=1, index_col=0)  # 0 refers to the first sheet
df31

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

#Prepare data
x = df31['Score-R Slope']
y = df31['NEFL']
x_with_const = sm.add_constant(x)

#Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

#Model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

#Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

plt.figure(figsize=(10, 10))

#Regression line only
sns.regplot(
x=x,
y=y,
scatter=False,
line_kws={'color': 'black', 'linestyle': '--'},
ci=95
)

#--- custom scatter points by sample type ---
#Boolean masks based on sample name containing 'NoMut'
mask_noMut = df31.index.str.contains('NoMut', case=False, na=False)
mask_other = ~mask_noMut

#NoMut samples: green triangles
plt.scatter(
df31.loc[mask_noMut, 'Score-R Slope'],
df31.loc[mask_noMut, 'NEFL'],
marker='^', # triangle
s=100,
color='green',
label='NoMut Case'
)

#All other samples: purple diamonds
plt.scatter(
df31.loc[mask_other, 'Score-R Slope'],
df31.loc[mask_other, 'NEFL'],
marker='D', # diamond
s=100,
color='#800080',
label='Other'
)

#Axis titles
plt.title('Scatter Plot df31', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('NEFL')
plt.grid(False)
plt.legend(loc='upper left')

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]
#Draw table
plt.table(
cellText=table_data,
colLabels=["Statistic", "Value"],
cellLoc='center',
loc='bottom',
bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('NEFL_Case_all_vs_ScoreR_Slope.png', bbox_inches='tight')
plt.show()

In [ ]:
df32= pd.read_excel("Case_Corr.xlsx", sheet_name=2, index_col=0)  # 0 refers to the first sheet
df32

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df32['Score-R Slope']
y = df32['CHIT1']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# --- custom scatter points by sample type ---
# Boolean masks based on sample name containing 'NoMut'
mask_noMut = df32.index.str.contains('NoMut', case=False, na=False)
mask_other = ~mask_noMut

# NoMut samples: green triangles
plt.scatter(
    df32.loc[mask_noMut, 'Score-R Slope'],
    df32.loc[mask_noMut, 'CHIT1'],
    marker='^',           # triangle
    s=100,
    color='green',
    label='NoMut Case'
)

# All other samples: purple diamonds
plt.scatter(
    df32.loc[mask_other, 'Score-R Slope'],
    df32.loc[mask_other, 'CHIT1'],
    marker='D',           # diamond
    s=100,
    color='#800080',
    label='Other'
)

# Axis titles
plt.title('Scatter Plot df32', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('CHIT1')
plt.grid(False)
plt.legend(loc='upper left')

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('CHIT1_Case_all_vs_ScoreR_Slope.png', bbox_inches='tight')
plt.show()

In [ ]:
df33= pd.read_excel("Case_Corr.xlsx", sheet_name=3, index_col=0)  # 0 refers to the first sheet
df33

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df33['Score-R Slope']
y = df33['CHI3L1']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# --- custom scatter points by sample type ---
# Boolean masks based on sample name containing 'NoMut'
mask_noMut = df33.index.str.contains('NoMut', case=False, na=False)
mask_other = ~mask_noMut

# NoMut samples: green triangles
plt.scatter(
    df33.loc[mask_noMut, 'Score-R Slope'],
    df33.loc[mask_noMut, 'CHI3L1'],
    marker='^',           # triangle
    s=100,
    color='green',
    label='NoMut Case'
)

# All other samples: purple diamonds
plt.scatter(
    df33.loc[mask_other, 'Score-R Slope'],
    df33.loc[mask_other, 'CHI3L1'],
    marker='D',           # diamond
    s=100,
    color='#800080',
    label='Other'
)

# Axis titles
plt.title('Scatter Plot df33', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('CHI3L1')
plt.grid(False)
plt.legend(loc='upper left')

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('CHI3L1_Case_all_vs_ScoreR_Slope.png', bbox_inches='tight')
plt.show()


In [ ]:
df34= pd.read_excel("Case_Corr.xlsx", sheet_name=4, index_col=0)  # 0 refers to the first sheet
df34

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df34['Score-R Slope']
y = df34['CCL2']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# --- custom scatter points by sample type ---
# Boolean masks based on sample name containing 'NoMut'
mask_noMut = df34.index.str.contains('NoMut', case=False, na=False)
mask_other = ~mask_noMut

# NoMut samples: green triangles
plt.scatter(
    df34.loc[mask_noMut, 'Score-R Slope'],
    df34.loc[mask_noMut, 'CCL2'],
    marker='^',           # triangle
    s=100,
    color='green',
    label='NoMut Case'
)

# All other samples: purple diamonds
plt.scatter(
    df34.loc[mask_other, 'Score-R Slope'],
    df34.loc[mask_other, 'CCL2'],
    marker='D',           # diamond
    s=100,
    color='#800080',
    label='Other'
)

# Axis titles
plt.title('Scatter Plot df34', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('CCL2')
plt.grid(False)
plt.legend(loc='upper left')

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('CCL2_Case_all_vs_ScoreR_Slope.png', bbox_inches='tight')
plt.show()


In [ ]:
df35= pd.read_excel("Case_Corr.xlsx", sheet_name=5, index_col=0)  # 0 refers to the first sheet
df35

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df35['Score-R Slope']
y = df35['CCL3']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Score-R Slope']
intercept_conf_int = model.conf_int().loc['const']

# Model values
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Score-R Slope']
intercept = model.params['const']
slope_se = model.bse['Score-R Slope']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# --- custom scatter points by sample type ---
# Boolean masks based on sample name containing 'NoMut'
mask_noMut = df35.index.str.contains('NoMut', case=False, na=False)
mask_other = ~mask_noMut

# NoMut samples: green triangles
plt.scatter(
    df35.loc[mask_noMut, 'Score-R Slope'],
    df35.loc[mask_noMut, 'CCL3'],
    marker='^',           # triangle
    s=100,
    color='green',
    label='NoMut Case'
)

# All other samples: purple diamonds
plt.scatter(
    df35.loc[mask_other, 'Score-R Slope'],
    df35.loc[mask_other, 'CCL3'],
    marker='D',           # diamond
    s=100,
    color='#800080',
    label='Other'
)

# Axis titles
plt.title('Scatter Plot df35', fontsize=14)
plt.xlabel('Score-R Slope')
plt.ylabel('CCL3')
plt.grid(False)
plt.legend(loc='upper left')

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["Adjusted R²", f"{adj_r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('CCL3_Case_all_vs_ScoreR_Slope.png', bbox_inches='tight')
plt.show()